# START HERE - 500m

In [ ]:
# THIS MAKES A GRID OF POINTS
import os
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pyproj import CRS

# =========================
# USER SETTINGS
# =========================
IN_SHP = r"C:\Users\BNDRY.shp"
OUT_DIR = r"C:\Users"
SPACING_M = 500.0  # desired spacing in METERS (always)

# If your boundary is actually EPSG:2876 but the shapefile is missing/incorrectly labeled,
# set this to True to force-assign EPSG:2876 without reprojecting.
FORCE_ASSIGN_EPSG2876_IF_MISSING = True

# Outputs
OUT_POINTS_SHP_SRC = os.path.join(OUT_DIR, "RCVFD_grid_points.shp")
OUT_POINTS_SHP_4326 = os.path.join(OUT_DIR, "RCVFD_grid_points_WGS84.shp")
OUT_POINTS_CSV = os.path.join(OUT_DIR, "RCVFD_grid_points.csv")

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================
def _crs_units_and_to_crs_units_per_meter(crs: CRS):
    """
    Returns:
      units_name (str): e.g. "meters", "US survey foot", "foot"
      crs_units_per_meter (float): multiplier such that:
         spacing_in_crs_units = spacing_meters * crs_units_per_meter
    """
    # Default assumption: meters
    units_name = "metre"
    crs_units_per_meter = 1.0

    try:
        # For projected CRS, axis_info[0].unit_name often exists
        axis = crs.axis_info[0]
        units_name = (axis.unit_name or "").lower()

        # Use explicit conversions for common cases
        if "metre" in units_name or "meter" in units_name:
            crs_units_per_meter = 1.0
            units_name = axis.unit_name
        elif "us survey foot" in units_name or "foot_us" in units_name or "u.s. survey foot" in units_name:
            # 1 US survey foot = 1200/3937 meters => feet per meter = 3937/1200
            crs_units_per_meter = 3937.0 / 1200.0
            units_name = axis.unit_name
        elif units_name.strip() == "foot" or "international foot" in units_name:
            # 1 international foot = 0.3048 meters => feet per meter = 1/0.3048
            crs_units_per_meter = 1.0 / 0.3048
            units_name = axis.unit_name
        else:
            # Fallback: try to infer from unit conversion factor if available
            # pyproj doesn't always expose a direct factor cleanly; keep meters if unknown
            units_name = axis.unit_name or "unknown"
            crs_units_per_meter = 1.0
    except Exception:
        units_name = "unknown"
        crs_units_per_meter = 1.0

    return units_name, crs_units_per_meter

# =========================
# READ + PREP BOUNDARY
# =========================
bndry = gpd.read_file(IN_SHP)
if bndry.empty:
    raise RuntimeError("Boundary shapefile read as empty.")

# Fix/assign CRS if needed
if bndry.crs is None:
    if FORCE_ASSIGN_EPSG2876_IF_MISSING:
        bndry = bndry.set_crs(epsg=2876, allow_override=True)
    else:
        raise RuntimeError("Input shapefile has no CRS. Set it in GIS or set FORCE_ASSIGN_EPSG2876_IF_MISSING=True.")

src_crs = CRS.from_user_input(bndry.crs)
units_name, crs_units_per_meter = _crs_units_and_to_crs_units_per_meter(src_crs)

spacing_crs_units = SPACING_M * crs_units_per_meter

print("Boundary CRS:", src_crs.to_string())
print("Detected linear units:", units_name)
print(f"Requested spacing: {SPACING_M} m")
print(f"Using spacing in CRS units: {spacing_crs_units:.6f} ({units_name})")

# Dissolve to a single geometry for filtering
bndry_union = bndry.geometry.unary_union

# =========================
# BUILD REGULAR GRID IN SOURCE CRS UNITS
# =========================
minx, miny, maxx, maxy = bndry_union.bounds

xs = np.arange(minx, maxx + spacing_crs_units, spacing_crs_units)
ys = np.arange(miny, maxy + spacing_crs_units, spacing_crs_units)

pts = [Point(x, y) for y in ys for x in xs]
gpts = gpd.GeoDataFrame({"geometry": pts}, crs=bndry.crs)

# Filter to within boundary (strict). Use intersects if you want boundary-inclusive.
mask_within = gpts.within(bndry_union)
gpts_in = gpts.loc[mask_within].copy()

# Add stable ID and coordinates in source CRS
gpts_in.reset_index(drop=True, inplace=True)
gpts_in["id"] = [f"pt_{i:06d}" for i in range(len(gpts_in))]
gpts_in["x_src"] = gpts_in.geometry.x
gpts_in["y_src"] = gpts_in.geometry.y

# =========================
# WRITE OUTPUTS
# =========================
# 1) Shapefile in source CRS
gpts_in.to_file(OUT_POINTS_SHP_SRC)

# 2) Reproject to WGS84, add lon/lat, save shapefile
gpts_wgs84 = gpts_in.to_crs(epsg=4326)
gpts_wgs84["lon"] = gpts_wgs84.geometry.x
gpts_wgs84["lat"] = gpts_wgs84.geometry.y
gpts_wgs84.to_file(OUT_POINTS_SHP_4326)

# 3) CSV for NAIP batch (id, lon, lat)
gpts_wgs84[["id", "lon", "lat"]].to_csv(OUT_POINTS_CSV, index=False)

print("Done.")
print(f"Points created (within boundary): {len(gpts_in):,}")
print("Wrote:")
print(" -", OUT_POINTS_SHP_SRC)
print(" -", OUT_POINTS_SHP_4326)
print(" -", OUT_POINTS_CSV)

In [ ]:
# THIS IS THE MODEL RUN - 500m 
# FINAL NAIP POND / DRAFTING LOCATION WORKFLOW
# =============================================================================
# Purpose:
#   1) Run NAIP image chips through a conservative multimodal AI pond-screening prompt.
#   2) Prevent silent skipped failures: cases are skipped only when they are truly complete.
#   3) Retry failed, invalid, or ok=False cases.
#   4) Build a clean RESULTS folder that can be handed to another AI or reviewer to write Results.
#
# Main outputs:
#   OUT_ROOT/
#       CASES/                         per-chip outputs
#       index.csv                      one final row per input point
#       index_attempts.csv             every attempt, including retries
#       batch_log.txt                  errors and traceback log
#       RESULTS/
#           README_RESULTS.md
#           RESULTS_CONTEXT_FOR_AI.md
#           tables/
#           gis/
#           figures/
#           review_pack/
#           qa/
#           documents/
# =============================================================================

import os
import csv
import json
import time
import base64
import hashlib
import traceback
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from io import BytesIO

import pandas as pd
import numpy as np
from PIL import Image

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import rasterio
from rasterio.mask import mask
from rasterio.transform import xy as rio_xy

import geopandas as gpd
from shapely.geometry import shape, Point, box, mapping

import planetary_computer
from openai import OpenAI
import matplotlib.pyplot as plt

try:
    from importlib.metadata import version as pkg_version
except Exception:
    pkg_version = None


# =============================================================================
# CONFIG
# =============================================================================

OPENAI_MODEL = "gpt-5.1"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Set your OPENAI_API_KEY in the environment"
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_PRICING_PER_1M = {
    "gpt-5.1": {"input": 1.25, "output": 10.00},

}

RUN_COST_TRACKER = {
    "calls": 0,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
}

# Thread locks used when MAX_PARALLEL_WORKERS > 1.
# These prevent shared CSV/log/cost/matplotlib outputs from colliding across workers.
RUN_COST_LOCK = threading.Lock()
ATTEMPTS_CSV_LOCK = threading.Lock()
LOG_FILE_LOCK = threading.Lock()
MATPLOTLIB_LOCK = threading.Lock()
PRINT_LOCK = threading.Lock()

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SEARCH_URL = STAC_URL.rstrip("/") + "/search"

ASSUMED_GSD_M = 0.30

# zoom_m is half the scene width.
# 250 = 500 m x 500 m chips.
# 500 = 1000 m x 1000 m chips.
DEFAULT_ZOOM_M = 250
DEFAULT_START = "2023-01-01"
DEFAULT_END = "2024-12-31"

# Change this to a new folder for your final production run.
DEFAULT_OUT_ROOT = r"C:\Users"

# Input CSV must have lon/lat fields, or longitude/latitude, or x/y.
DEFAULT_POINTS_CSV = r"C:\UsersRCVFD_grid_points.csv"

CASE_DIR_NAME = "CASES"
RESULTS_DIR_NAME = "RESULTS"

# Final-run safeguards.
FORCE_RERUN_ALL = False
RERUN_FAILED_OR_INVALID = True
MAX_ATTEMPTS_PER_CASE = 3
SLEEP_BETWEEN_ATTEMPTS_SEC = 3

# Parallel processing.
# 1 = sequential. 3 is a good safe default for OpenAI + NAIP + local file writing.
MAX_PARALLEL_WORKERS = 1

# Outputs required for a case to be treated as complete.
REQUIRE_CROP_TIF_FOR_COMPLETE = True
REQUIRE_CASE_SHAPEFILE_FOR_COMPLETE = False

# Results packaging.
MAX_REVIEW_REPORTS_TO_COPY = 999999
MAX_EXAMPLE_REPORTS_PER_GROUP = 999999


# =============================================================================
# COST HELPERS
# =============================================================================

def get_model_pricing(model_name):
    return MODEL_PRICING_PER_1M.get(model_name, {"input": 0.0, "output": 0.0})


def estimate_cost_usd(model_name, prompt_tokens, completion_tokens):
    pricing = get_model_pricing(model_name)
    in_cost = (prompt_tokens or 0) / 1_000_000.0 * pricing["input"]
    out_cost = (completion_tokens or 0) / 1_000_000.0 * pricing["output"]
    return in_cost + out_cost


def update_run_cost_tracker(prompt_tokens, completion_tokens, total_tokens, cost_usd):
    with RUN_COST_LOCK:
        RUN_COST_TRACKER["calls"] += 1
        RUN_COST_TRACKER["prompt_tokens"] += int(prompt_tokens or 0)
        RUN_COST_TRACKER["completion_tokens"] += int(completion_tokens or 0)
        RUN_COST_TRACKER["total_tokens"] += int(total_tokens or 0)
        RUN_COST_TRACKER["estimated_cost_usd"] += float(cost_usd or 0.0)


def print_cost_summary(prefix="RUN"):
    with RUN_COST_LOCK:
        snap = RUN_COST_TRACKER.copy()
    print(
        f"[{prefix}] "
        f"calls={snap['calls']} | "
        f"prompt_tokens={snap['prompt_tokens']} | "
        f"completion_tokens={snap['completion_tokens']} | "
        f"total_tokens={snap['total_tokens']} | "
        f"estimated_cost_usd=${snap['estimated_cost_usd']:.6f}"
    )


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def write_json(path, obj):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def write_text(path, text):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")


def safe_lower(v):
    return "" if pd.isna(v) else str(v).strip().lower()


def parse_boolish(v):
    s = safe_lower(v)
    if s in {"true", "1", "yes", "y", "ok"}:
        return True
    if s in {"false", "0", "no", "n", ""}:
        return False
    return False


def clean_filename(s, max_len=120):
    s = "" if s is None else str(s)
    keep = []
    for ch in s:
        if ch.isalnum() or ch in "._-":
            keep.append(ch)
        else:
            keep.append("_")
    out = "".join(keep).strip("_")
    while "__" in out:
        out = out.replace("__", "_")
    return out[:max_len] if len(out) > max_len else out


def safe_case_id(row_id, lon, lat):
    rid = "point" if row_id in (None, "") else str(row_id)
    base = f"{rid}__{lat:.6f}__{lon:.6f}"
    h = hashlib.sha1(base.encode("utf-8")).hexdigest()[:8]
    safe = clean_filename(base.replace("-", "m").replace(".", "p"), max_len=140)
    return f"{safe}__{h}"


def get_pkg_versions():
    keys = [
        "numpy", "pandas", "rasterio", "geopandas", "shapely",
        "planetary-computer", "requests", "Pillow", "matplotlib", "openai"
    ]
    out = {}
    if pkg_version is None:
        return out
    for k in keys:
        try:
            out[k] = pkg_version(k)
        except Exception:
            pass
    return out


# =============================================================================
# NETWORK
# =============================================================================

def make_retry_session(
    total=10,
    backoff_factor=0.8,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET", "POST"),
    timeout=(10, 90),
):
    s = requests.Session()
    retry = Retry(
        total=total,
        connect=total,
        read=total,
        status=total,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=set(allowed_methods),
        raise_on_status=False,
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update(
        {
            "User-Agent": "naip-pond-final-results-workflow/1.0 (+python requests)",
            "Accept": "application/geo+json, application/json",
            "Content-Type": "application/json",
        }
    )
    return s, timeout


SESSION, REQ_TIMEOUT = make_retry_session()


# =============================================================================
# STAC SEARCH
# =============================================================================

def stac_search_naip(intersects_geojson, datetime_range, limit=200):
    payload = {
        "collections": ["naip"],
        "intersects": intersects_geojson,
        "datetime": datetime_range,
        "limit": limit,
    }
    last_err = None
    for attempt in range(1, 6):
        try:
            r = SESSION.post(SEARCH_URL, data=json.dumps(payload), timeout=REQ_TIMEOUT)
            r.raise_for_status()
            fc = r.json()
            return fc.get("features", []) or []
        except Exception as e:
            last_err = e
            time.sleep(min(2**attempt, 20))
    raise RuntimeError(f"STAC search failed after retries: {last_err}")


def find_best_naip_item(lon, lat, start=DEFAULT_START, end=DEFAULT_END, pad_deg=0.001):
    aoi = {
        "type": "Polygon",
        "coordinates": [[
            [lon - pad_deg, lat - pad_deg],
            [lon + pad_deg, lat - pad_deg],
            [lon + pad_deg, lat + pad_deg],
            [lon - pad_deg, lat + pad_deg],
            [lon - pad_deg, lat - pad_deg],
        ]],
    }

    feats = stac_search_naip(intersects_geojson=aoi, datetime_range=f"{start}/{end}", limit=200)
    if not feats:
        return None, aoi

    aoi_shape = shape(aoi)
    feats_sorted = sorted(
        feats,
        key=lambda f: shape(f["geometry"]).intersection(aoi_shape).area,
        reverse=True,
    )
    return feats_sorted[0], aoi


def extract_asset_href(feat):
    if not isinstance(feat, dict):
        return None

    assets = feat.get("assets")
    if not isinstance(assets, dict):
        assets = feat.get("properties", {}).get("assets", None)

    if not isinstance(assets, dict) or not assets:
        return None

    if "image" in assets and isinstance(assets["image"], dict) and "href" in assets["image"]:
        return assets["image"]["href"]

    for a in assets.values():
        if isinstance(a, dict) and str(a.get("href", "")).lower().endswith((".tif", ".tiff")):
            return a["href"]

    return None


# =============================================================================
# IMAGE HELPERS
# =============================================================================

def pct_scale_to_u8(rgb_arr):
    if rgb_arr.dtype == np.uint8:
        return rgb_arr
    out = []
    for b in range(rgb_arr.shape[0]):
        band = rgb_arr[b].astype(np.float32)
        if not np.isfinite(band).any():
            out.append(np.zeros_like(band, dtype=np.uint8))
            continue
        lo, hi = np.nanpercentile(band, [0.0, 99.5])
        if not np.isfinite(lo):
            lo = 0.0
        if not np.isfinite(hi) or hi <= lo:
            hi = lo + 1.0
        band = (np.clip((band - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
        out.append(band)
    return np.stack(out, axis=0)


def try_parse_json(text):
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.strip("`").strip()
        if t.lower().startswith("json"):
            t = t[4:].strip()
    if "{" in t and "}" in t:
        t2 = t[t.find("{"): t.rfind("}") + 1]
    else:
        t2 = t
    try:
        return json.loads(t2), t2
    except Exception:
        return None, t2


def export_crop_tif(href_signed, crop_geom, out_tif_path):
    with rasterio.open(href_signed) as src:
        data, out_transform = mask(src, crop_geom, crop=True)
        out_meta = src.meta.copy()
        out_meta.update(
            {
                "driver": "GTiff",
                "height": data.shape[1],
                "width": data.shape[2],
                "transform": out_transform,
                "count": data.shape[0],
                "compress": "deflate",
                "tiled": True,
                "blockxsize": 256,
                "blockysize": 256,
            }
        )
        out_tif_path = Path(out_tif_path)
        ensure_dir(out_tif_path.parent)
        with rasterio.open(out_tif_path, "w", **out_meta) as dst:
            dst.write(data)
    return str(out_tif_path)


def sanitize_point_list(points, img_w, img_h):
    clean = []
    if not isinstance(points, list):
        return clean

    for i, p in enumerate(points):
        if not isinstance(p, dict):
            continue
        try:
            x = float(p.get("x", np.nan))
            y = float(p.get("y", np.nan))
        except Exception:
            continue
        if not all(np.isfinite([x, y])):
            continue
        x = max(0.0, min(1.0, x))
        y = max(0.0, min(1.0, y))
        px = int(round(x * (img_w - 1)))
        py = int(round(y * (img_h - 1)))
        conf = p.get("confidence", None)
        try:
            conf = None if conf is None else max(0.0, min(1.0, float(conf)))
        except Exception:
            conf = None
        clean.append(
            {
                "label": str(p.get("label", f"pond_{i+1}")),
                "confidence": conf,
                "x": x,
                "y": y,
                "pixel_x": px,
                "pixel_y": py,
            }
        )
    return clean


def normalized_points_to_geodataframe(point_list, crop_transform, crop_crs, point_id, case_id, lon, lat, naip_id):
    rows = []
    if not point_list:
        return gpd.GeoDataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id",
            "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y"
        ], geometry=[], crs=crop_crs)

    img_h = int(crop_transform["img_h"])
    img_w = int(crop_transform["img_w"])
    aff = crop_transform["transform"]

    for p in point_list:
        px = max(0, min(img_w - 1, int(p["pixel_x"])))
        py = max(0, min(img_h - 1, int(p["pixel_y"])))
        map_x, map_y = rio_xy(aff, py, px, offset="center")
        rows.append(
            {
                "case_id": case_id,
                "point_id": point_id,
                "src_lon": lon,
                "src_lat": lat,
                "naip_id": naip_id,
                "pond_label": p.get("label", ""),
                "confidence": p.get("confidence", None),
                "norm_x": p.get("x", None),
                "norm_y": p.get("y", None),
                "pixel_x": px,
                "pixel_y": py,
                "geometry": Point(map_x, map_y),
            }
        )
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crop_crs)


def empty_pond_points_gdf(crs="EPSG:4326"):
    return gpd.GeoDataFrame(
        columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id",
            "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y"
        ],
        geometry=[],
        crs=crs,
    )


def write_vector_safe(gdf, path, driver=None):
    path = Path(path)
    ensure_dir(path.parent)
    out = gdf.copy() if gdf is not None else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    if out.crs is None:
        out = out.set_crs(4326)

    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)

    if path.suffix.lower() == ".shp":
        rename_map = {
            "pond_label": "pond_lbl",
            "confidence": "conf",
            "pixel_x": "pix_x",
            "pixel_y": "pix_y",
            "likely_draftable": "likely_dr",
            "road_or_access_nearby": "road_acc",
            "is_there_surface_water": "surf_wtr",
            "is_there_a_pond": "pond",
        }
        out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
        out.to_file(path)
    else:
        if driver:
            out.to_file(path, driver=driver)
        elif path.suffix.lower() == ".geojson":
            out.to_file(path, driver="GeoJSON")
        elif path.suffix.lower() == ".gpkg":
            layer_name = clean_filename(path.stem, max_len=60) or "layer"
            out.to_file(path, driver="GPKG", layer=layer_name)
        else:
            out.to_file(path)
    return str(path)


def _write_report_png_unlocked(pil_image, title_text, json_text, out_path, dpi=200):
    obj, norm_text = try_parse_json(json_text)
    display_text = json.dumps(obj, indent=2) if obj is not None else norm_text

    img_w, img_h = pil_image.size
    pond_points = []
    if isinstance(obj, dict):
        pond_points = sanitize_point_list(obj.get("pond_points", []), img_w=img_w, img_h=img_h)

    fig = plt.figure(figsize=(13, 6), dpi=dpi)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.0])
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])

    ax0.imshow(pil_image)
    ax0.axis("off")
    ax0.set_title(title_text, fontsize=10)

    for idx, p in enumerate(pond_points, start=1):
        x = p["pixel_x"]
        y = p["pixel_y"]
        ax0.scatter([x], [y], s=60, c="red", marker="o", edgecolors="white", linewidths=0.8)
        label = f"{idx}"
        conf = p.get("confidence", None)
        if conf is not None:
            label = f"{idx} ({float(conf):.2f})"
        ax0.text(
            x + 4,
            max(0, y - 4),
            label,
            color="white",
            fontsize=8,
            fontweight="bold",
            bbox=dict(facecolor="red", edgecolor="red", boxstyle="round,pad=0.2"),
        )

    ax1.axis("off")
    ax1.text(
        0.0,
        1.0,
        display_text,
        va="top",
        ha="left",
        family="monospace",
        fontsize=8.5,
        wrap=True,
    )

    plt.tight_layout()
    out_path = Path(out_path)
    ensure_dir(out_path.parent)
    plt.savefig(str(out_path), bbox_inches="tight")
    plt.close(fig)


def write_report_png(pil_image, title_text, json_text, out_path, dpi=200):
    # Matplotlib uses global state, so protect figure creation/saving when running threads.
    with MATPLOTLIB_LOCK:
        return _write_report_png_unlocked(pil_image, title_text, json_text, out_path, dpi=dpi)


# =============================================================================
# OPENAI VISION
# =============================================================================

def ask_image_question(pil_image, question, system_preamble=None, model=OPENAI_MODEL, temperature=0):
    buf = BytesIO()
    pil_image.save(buf, format="PNG")
    img_data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

    sys_msg = system_preamble or "You are a careful remote sensing analyst. Answer concisely and only from the image."

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": sys_msg},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": img_data_url}},
                ],
            },
        ],
        temperature=temperature,
    )

    answer_text = resp.choices[0].message.content.strip()

    usage = getattr(resp, "usage", None)
    prompt_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
    completion_tokens = getattr(usage, "completion_tokens", 0) if usage else 0
    total_tokens = getattr(usage, "total_tokens", 0) if usage else 0

    est_cost_usd = estimate_cost_usd(
        model_name=model,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
    )

    update_run_cost_tracker(
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        cost_usd=est_cost_usd,
    )

    print(
        f"[API CALL] model={model} | "
        f"prompt_tokens={prompt_tokens} | "
        f"completion_tokens={completion_tokens} | "
        f"total_tokens={total_tokens} | "
        f"estimated_cost_usd=${est_cost_usd:.6f}"
    )

    usage_info = {
        "model": model,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": est_cost_usd,
    }
    return answer_text, usage_info


def try_parse_json_strict_or_retry(pil, question, system_preamble, temperature=0, model=OPENAI_MODEL):
    ans1, usage1 = ask_image_question(
        pil,
        question,
        system_preamble=system_preamble,
        temperature=temperature,
        model=model,
    )
    obj1, _ = try_parse_json(ans1)
    if obj1 is not None:
        combined_usage = {
            "model": model,
            "prompt_tokens": usage1["prompt_tokens"],
            "completion_tokens": usage1["completion_tokens"],
            "total_tokens": usage1["total_tokens"],
            "estimated_cost_usd": usage1["estimated_cost_usd"],
            "num_api_calls_for_case": 1,
        }
        return ans1, True, obj1, combined_usage

    retry_q = (
        "Your previous response was not valid JSON.\n\n"
        "Return ONLY valid JSON matching the exact schema requested. "
        "Do not include markdown, prose, or comments.\n\n"
        + question
    )

    ans2, usage2 = ask_image_question(
        pil,
        retry_q,
        system_preamble=system_preamble,
        temperature=temperature,
        model=model,
    )
    obj2, _ = try_parse_json(ans2)

    combined_usage = {
        "model": model,
        "prompt_tokens": usage1["prompt_tokens"] + usage2["prompt_tokens"],
        "completion_tokens": usage1["completion_tokens"] + usage2["completion_tokens"],
        "total_tokens": usage1["total_tokens"] + usage2["total_tokens"],
        "estimated_cost_usd": usage1["estimated_cost_usd"] + usage2["estimated_cost_usd"],
        "num_api_calls_for_case": 2,
    }
    return ans2, (obj2 is not None), obj2, combined_usage


# =============================================================================
# PROMPT
# =============================================================================

def build_pond_prompt(scene_width_m=None):
    scene_width_m = scene_width_m or int(DEFAULT_ZOOM_M * 2)
    return rf"""
Analyze this NAIP RGB aerial image (~{ASSUMED_GSD_M:.2f} m / 30 cm ground sample distance).
Use ONLY what is visible in the image. Do not assume anything not directly observable.

This task must be conservative. False positives are worse than false negatives.
The primary goal is to identify visible open-water ponds that could plausibly be wildfire drafting sources.

Return ONLY valid JSON.

Context:
This image is a fixed-area crop (~{scene_width_m} m × {scene_width_m} m). The goal is to identify true visible open surface water
and specifically ponds or standing water bodies that might serve as potential drafting sources.

Whole-scene context rule:
- Do not judge a candidate feature in isolation.
- Compare the candidate to the rest of the image before deciding it is water.
- Use the whole scene to determine whether the dark area is more consistent with shadow, terrain shading, tree shadow, rock shadow, or other non-water dark features that appear elsewhere in the image.
- If similar dark shapes, tones, or textures occur throughout the scene in obvious shadows or shaded terrain, prefer "no" for pond.

Definitions (image-only):

Surface water:
Visible open water such as pond, lake, reservoir, stream reach with visible water, canal holding water,
or wetland/open water patch.

Water surface appearance rule:
- Open water usually appears smoother and more internally uniform than mud, grass, sediment, or disturbed ground.
- If the interior shows mottled texture, vegetation patches, hoof-disturbed mud, exposed sediment, or land-like texture across most of the basin, do NOT classify it as open water.

Pond:
A distinct, bounded, mostly standing open-water body with a visible open-water surface and visible edges or shoreline.

Pond basin / impoundment footprint:
A depression, stock tank, basin, or pond-shaped feature that may be dry, muddy, vegetated, only faintly damp,
or may contain too little visible water to matter operationally. A pond basin is NOT the same as visible open water.

Draftable pond:
A visible open-water pond that appears, from imagery alone, to be plausibly usable as a wildfire drafting source.
Positive evidence includes visible open water, a distinct shoreline, nearby road or vehicle access, and a feature that is not obviously too small.
This is only an image-based screening judgment and does not confirm depth, volume, bank stability, legal access, seasonal persistence, or actual field operability.

Manmade rectangular feature exclusion rules:
- Do NOT consider anything perfectly square or perfectly rectangular to be a pond unless unmistakable open water is clearly visible.
- Perfect geometric shapes are strong negative evidence for ponds in this task and should usually be treated as manmade features, not open-water ponds.
- Small dark rectangular or square features near houses, sheds, driveways, pads, or developed areas should NOT be classified as ponds unless clear open water is unmistakably visible.
- Roofs, sheds, covered tanks, liners, tarps, equipment pads, shadowed structures, and other manmade site features can appear dark and water-like from overhead.
- A neat rectangular feature is more likely to be a manmade object than a natural or draftable pond unless there is strong visible evidence of true open water.
- If a feature is adjacent to buildings or a maintained homesite, be especially cautious and prefer "no" unless water is obvious.

Shadow comparison rule:
- Before classifying a feature as pond, compare it to other dark areas in the image.
- If the candidate has similar darkness, texture, edge quality, or orientation as nearby shadows from trees, rocks, cliffs, buildings, or terrain, do NOT classify it as a pond.
- If the dark feature blends gradually into surrounding shadow or appears connected to shadowed terrain, choose "no" for pond.

Scene consistency rule:
- A true pond should remain visually distinct from the broader shadow pattern of the scene.
- If the feature can be explained by the same lighting and shadow behavior seen elsewhere in the image, it is not strong evidence for water.
- Use surrounding illumination, terrain, and nearby shadows to interpret ambiguous dark features conservatively.

Contextual shadow rule for trees and rock:
- In wooded, rocky, or mountainous scenes, many dark features are caused by tree shadow, terrain shadow, or rock relief.
- If a candidate dark feature resembles the shadow behavior of nearby trees, ridges, boulders, or slopes, do not classify it as a pond unless open water is unmistakable.

Important interpretation rules:
1. Darkness alone is NOT evidence of water.
2. Do NOT confuse shadow, vegetation, wet ground, mud, burn scar, terrain shading, roofs, tanks, troughs, containers,
   disturbed pads, equipment, or structures with ponds.
3. A true pond with open water requires BOTH:
   - a visible bounded shoreline or edge, AND
   - an interior that visibly looks like open water
4. A coherent basin shape alone is NOT enough.
5. If a feature appears mostly dry, muddy, vegetated, shallow, or only faintly damp, do NOT classify it as a pond with open water.
6. If a pond-shaped basin is visible but open water is not clearly visible, choose "no" for pond.
7. If a basin or stock tank footprint is visible without clear open water, do NOT classify it as a pond.
8. If uncertain between water and shadow, vegetation, wet ground, mud, structure, or tank, choose "no" for pond.
9. Small dark rectangular features in developed or disturbed areas are often NOT ponds.
10. Tiny dark features may be houses, sheds, tanks, pads, containers, shadows, or other manmade features. Do not call them ponds unless open water is unmistakable.
11. For wildfire drafting, size matters. If a feature looks too small, too narrow, too shallow-looking, or operationally insignificant, do not mark it draftable.
12. If the feature may contain water but appears too small to realistically support drafting operations, set "likely_draftable" to "no" or "uncertain".
13. If uncertain, choose "no" for pond and explain why.
14. If a feature is uncertain, do NOT include a point.
15. If there are no visible ponds, return an empty list for pond_points.
16. In steep, rocky, alpine, or heavily textured terrain, dark enclosed features are often shadows, rock hollows, or terrain depressions rather than open water.
17. In rocky terrain, do NOT classify a feature as a pond unless the interior appears smooth and water-like, with a clearly bounded shoreline that is distinct from surrounding rock texture and shadow.
18. If a dark feature contains visible internal texture, irregular rock pattern, or tonal variation similar to surrounding rock/shadow, do NOT classify it as open water.
19. Small dark pockets embedded in broken rock or cliff-like terrain should usually be treated as shadow or rock unless open water is unmistakable.
20. If the feature could be explained by terrain shadow or rock geometry, choose "no" for pond.

Output JSON with exactly these keys:

{{
  "is_there_surface_water": "<yes|no>",
  "is_there_a_pond": "<yes|no>",
  "road_or_access_nearby": "yes|no|uncertain",
  "likely_draftable": "<yes|no|uncertain>",
  "pond_count": <integer>,
  "evidence": "<=90 words",
  "notes": "<concise operational note>",
  "pond_points": [
    {{
      "label": "<pond>",
      "confidence": <float 0-1>,
      "x": <float 0-1>,
      "y": <float 0-1>
    }}
  ]
}}

Rules for pond_points:
- Coordinates must be normalized to the displayed crop:
  x = column / img_width
  y = row / img_height
- All coordinates must be between 0 and 1.
- Use one center point per clearly visible pond/open-water feature.
- The point should be at the approximate CENTER of the visible pond/open-water body.
- If a feature is uncertain, do NOT include a point.
- If there are no visible ponds, return [].

Decision rules:
- If there is no clearly visible open water, then "is_there_a_pond" must be "no".
- If shoreline is not visible, "is_there_a_pond" should usually be "no".
- If the interior does not look like open water, "is_there_a_pond" must be "no".
- If the feature could reasonably be shadow, vegetation, mud, wet ground, structure, container, equipment, roof, or tank, "is_there_a_pond" must be "no".
- If a pond-shaped basin or impoundment is visible but clear open water is not, "is_there_a_pond" must be "no".
- If "pond_count" is 0, "pond_points" must be [].
- If a pond appears too small or operationally insignificant for drafting, "likely_draftable" should be "no" or "uncertain".
- Do not set "likely_draftable" to "yes" unless the feature appears plausibly large enough and operationally usable from the image.
- "road_or_access_nearby" refers only to what is visible in the image, such as a road, driveway, turnout, pull-off, or clear vehicle approach immediately adjacent to the pond.
- If a clearly visible road, driveway, turnout, or open vehicle-access area reaches or lies immediately adjacent to a clearly visible pond, set "road_or_access_nearby" to "yes".
- If access is not clearly visible, set "road_or_access_nearby" to "no" or "uncertain".
- A "yes" for road_or_access_nearby does not guarantee draftability by itself, but it is strong positive evidence.
- If there is a clearly visible pond with open water AND road/access immediately adjacent to it AND the pond is not obviously tiny, then prefer "likely_draftable": "yes".
- If there is a clearly visible pond with open water AND road/access immediately adjacent, but the pond may be too small or operationally marginal, set "likely_draftable": "uncertain".
- If there is no visible access near the pond, "likely_draftable" should usually be "no" or "uncertain".
- In rocky or mountainous terrain, darkness plus enclosure is not enough; require a smooth open-water appearance and a distinct shoreline.
- If a feature is small, dark, irregular, and embedded in rocky textured terrain, prefer "is_there_a_pond": "no" unless open water is unmistakable.
- If confidence is not high that the feature is true open water, set "pond_count" to 0 and return no point.
- Be conservative, but do not ignore obvious direct road access when judging likely draftability.

THE NUMBER ONE RULE IS TO NOT HALLUCINATE
""".strip()


def build_system_preamble():
    return (
        "You are a remote sensing analyst specializing in aerial imagery interpretation for wildfire water-source screening. "
        "Use only what is directly visible in the image. "
        "Assume NAIP is about 30 cm GSD. "
        "Be conservative and prefer false negatives over false positives. "
        "Carefully distinguish true open water from shadow, vegetation, wet ground, mud, dark soil, tanks, roofs, and structures. "
        "Evaluate road or vehicle access separately from pond presence. "
        "If a visible road, driveway, turnout, or vehicle-access area lies immediately adjacent to a clearly visible pond, mark road_or_access_nearby as yes. "
        "If a clearly visible pond has direct visible access and is not obviously tiny, likely_draftable may be yes. "
        "Only return pond center points for clearly visible pond/open-water features. "
        "Each point must represent the approximate center of a clearly visible pond. "
        "Coordinates must be normalized from 0 to 1. "
        "If there is uncertainty, do not return a point. "
        "Return only valid JSON with the exact keys requested."
    )


# =============================================================================
# AI OUTPUT VALIDATION
# =============================================================================

REQUIRED_AI_KEYS = [
    "is_there_surface_water",
    "is_there_a_pond",
    "road_or_access_nearby",
    "likely_draftable",
    "pond_count",
    "evidence",
    "notes",
    "pond_points",
]


def normalize_ai_obj(ai_obj, img_w=None, img_h=None):
    if not isinstance(ai_obj, dict):
        return None

    out = dict(ai_obj)
    for k in REQUIRED_AI_KEYS:
        if k not in out:
            if k == "road_or_access_nearby":
                out[k] = "uncertain"
            elif k == "pond_points":
                out[k] = []
            elif k == "pond_count":
                out[k] = 0
            else:
                out[k] = ""

    out["is_there_surface_water"] = safe_lower(out.get("is_there_surface_water"))
    out["is_there_a_pond"] = safe_lower(out.get("is_there_a_pond"))
    out["road_or_access_nearby"] = safe_lower(out.get("road_or_access_nearby"))
    out["likely_draftable"] = safe_lower(out.get("likely_draftable"))

    if out["is_there_surface_water"] not in {"yes", "no"}:
        out["is_there_surface_water"] = "no"
    if out["is_there_a_pond"] not in {"yes", "no"}:
        out["is_there_a_pond"] = "no"
    if out["road_or_access_nearby"] not in {"yes", "no", "uncertain"}:
        out["road_or_access_nearby"] = "uncertain"
    if out["likely_draftable"] not in {"yes", "no", "uncertain"}:
        out["likely_draftable"] = "uncertain"

    try:
        out["pond_count"] = int(out.get("pond_count", 0))
    except Exception:
        out["pond_count"] = 0

    if img_w is not None and img_h is not None:
        out["pond_points"] = sanitize_point_list(out.get("pond_points", []), img_w=img_w, img_h=img_h)
    elif not isinstance(out.get("pond_points", []), list):
        out["pond_points"] = []

    if out["pond_count"] <= 0:
        out["pond_count"] = 0
        out["pond_points"] = []
    elif len(out["pond_points"]) == 0:
        # Conservative: no usable point means no mapped pond center.
        out["pond_count"] = 0
        out["is_there_a_pond"] = "no"
        out["likely_draftable"] = "no"

    out["evidence"] = "" if out.get("evidence") is None else str(out.get("evidence"))[:600]
    out["notes"] = "" if out.get("notes") is None else str(out.get("notes"))[:600]

    return out


def validate_ai_json_file(ai_json_path):
    ai_json_path = Path(ai_json_path)
    issues = []
    if not ai_json_path.exists():
        return False, ["missing_ai_json"]
    try:
        obj = json.loads(ai_json_path.read_text(encoding="utf-8"))
    except Exception as e:
        return False, [f"ai_json_read_error:{type(e).__name__}:{e}"]
    if not isinstance(obj, dict):
        return False, ["ai_json_not_dict"]
    for k in REQUIRED_AI_KEYS:
        if k not in obj:
            issues.append(f"missing_key:{k}")
    if safe_lower(obj.get("is_there_surface_water")) not in {"yes", "no"}:
        issues.append("bad_is_there_surface_water")
    if safe_lower(obj.get("is_there_a_pond")) not in {"yes", "no"}:
        issues.append("bad_is_there_a_pond")
    if safe_lower(obj.get("road_or_access_nearby")) not in {"yes", "no", "uncertain"}:
        issues.append("bad_road_or_access_nearby")
    if safe_lower(obj.get("likely_draftable")) not in {"yes", "no", "uncertain"}:
        issues.append("bad_likely_draftable")
    try:
        pc = int(obj.get("pond_count"))
        if pc < 0:
            issues.append("negative_pond_count")
    except Exception:
        issues.append("bad_pond_count")
    if not isinstance(obj.get("pond_points"), list):
        issues.append("bad_pond_points")
    return len(issues) == 0, issues


def ai_fields_for_index(ai_obj):
    if not isinstance(ai_obj, dict):
        return {
            "ai_is_there_surface_water": "",
            "ai_is_there_a_pond": "",
            "ai_road_or_access_nearby": "",
            "ai_likely_draftable": "",
            "ai_pond_count": "",
        }
    return {
        "ai_is_there_surface_water": "" if ai_obj.get("is_there_surface_water") is None else str(ai_obj.get("is_there_surface_water")),
        "ai_is_there_a_pond": "" if ai_obj.get("is_there_a_pond") is None else str(ai_obj.get("is_there_a_pond")),
        "ai_road_or_access_nearby": "" if ai_obj.get("road_or_access_nearby") is None else str(ai_obj.get("road_or_access_nearby")),
        "ai_likely_draftable": "" if ai_obj.get("likely_draftable") is None else str(ai_obj.get("likely_draftable")),
        "ai_pond_count": "" if ai_obj.get("pond_count") is None else str(ai_obj.get("pond_count")),
    }


def make_error_result(point_id, lon, lat, zoom_m, error, case_id="", case_dir="", attempt_num=""):
    return {
        "ok": False,
        "final_complete": False,
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": case_dir,
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": "",
        "parse_ok": "",
        "validation_ok": False,
        "validation_issues": "",
        "ai_is_there_surface_water": "",
        "ai_is_there_a_pond": "",
        "ai_road_or_access_nearby": "",
        "ai_likely_draftable": "",
        "ai_pond_count": "",
        "api_calls_for_case": "",
        "prompt_tokens": "",
        "completion_tokens": "",
        "total_tokens": "",
        "estimated_cost_usd": "",
        "report_png": "",
        "image_png": "",
        "ai_json": "",
        "ai_raw": "",
        "crop_tif": "",
        "pond_points_shp": "",
        "pond_points_geojson": "",
        "pond_points_csv": "",
        "error": str(error),
    }


INDEX_FIELDNAMES = [
    "ok",
    "final_complete",
    "attempt_num",
    "case_id",
    "case_dir",
    "point_id",
    "lon",
    "lat",
    "zoom_m",
    "naip_id",
    "parse_ok",
    "validation_ok",
    "validation_issues",
    "ai_is_there_surface_water",
    "ai_is_there_a_pond",
    "ai_road_or_access_nearby",
    "ai_likely_draftable",
    "ai_pond_count",
    "api_calls_for_case",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "estimated_cost_usd",
    "report_png",
    "image_png",
    "ai_json",
    "ai_raw",
    "crop_tif",
    "pond_points_shp",
    "pond_points_geojson",
    "pond_points_csv",
    "error",
]


def coerce_result_fields(res):
    out = {k: res.get(k, "") for k in INDEX_FIELDNAMES}
    return out


# =============================================================================
# CASE COMPLETENESS CHECKING
# =============================================================================

def get_expected_case_paths(out_root, case_id):
    case_dir = Path(out_root) / CASE_DIR_NAME / case_id
    return {
        "case_dir": case_dir,
        "crop_png": case_dir / "crop.png",
        "report_png": case_dir / "report.png",
        "ai_json": case_dir / "ai.json",
        "ai_raw": case_dir / "ai_raw.txt",
        "meta_json": case_dir / "meta.json",
        "crop_tif": case_dir / "crop.tif",
        "pond_points_shp": case_dir / "ai_pond_points.shp",
        "pond_points_geojson": case_dir / "ai_pond_points.geojson",
        "pond_points_csv": case_dir / "ai_pond_points.csv",
    }


def case_is_complete(out_root, case_id):
    paths = get_expected_case_paths(out_root, case_id)
    issues = []

    if not paths["case_dir"].exists():
        issues.append("missing_case_dir")
    if not paths["crop_png"].exists():
        issues.append("missing_crop_png")
    if not paths["report_png"].exists():
        issues.append("missing_report_png")
    if not paths["ai_json"].exists():
        issues.append("missing_ai_json")
    if not paths["meta_json"].exists():
        issues.append("missing_meta_json")
    if REQUIRE_CROP_TIF_FOR_COMPLETE and not paths["crop_tif"].exists():
        issues.append("missing_crop_tif")
    if REQUIRE_CASE_SHAPEFILE_FOR_COMPLETE and not paths["pond_points_shp"].exists():
        issues.append("missing_pond_points_shp")
    if not paths["pond_points_geojson"].exists():
        issues.append("missing_pond_points_geojson")
    if not paths["pond_points_csv"].exists():
        issues.append("missing_pond_points_csv")

    ai_ok, ai_issues = validate_ai_json_file(paths["ai_json"])
    if not ai_ok:
        issues.extend(ai_issues)

    return len(issues) == 0, issues


def reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0):
    paths = get_expected_case_paths(out_root, case_id)
    validation_ok, validation_issues = case_is_complete(out_root, case_id)
    ai_obj = None
    try:
        ai_obj = json.loads(paths["ai_json"].read_text(encoding="utf-8"))
    except Exception:
        ai_obj = None
    ai_fields = ai_fields_for_index(ai_obj)

    naip_id = ""
    parse_ok = validation_ok
    try:
        meta = json.loads(paths["meta_json"].read_text(encoding="utf-8"))
        naip_id = meta.get("naip_id", "")
        parse_ok = bool(meta.get("parse_ok", validation_ok))
    except Exception:
        pass

    return coerce_result_fields({
        "ok": bool(validation_ok),
        "final_complete": bool(validation_ok),
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": str(paths["case_dir"]),
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": naip_id,
        "parse_ok": bool(parse_ok),
        "validation_ok": bool(validation_ok),
        "validation_issues": ";".join(validation_issues),
        **ai_fields,
        "api_calls_for_case": 0,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "estimated_cost_usd": 0,
        "report_png": str(paths["report_png"]) if paths["report_png"].exists() else "",
        "image_png": str(paths["crop_png"]) if paths["crop_png"].exists() else "",
        "ai_json": str(paths["ai_json"]) if paths["ai_json"].exists() else "",
        "ai_raw": str(paths["ai_raw"]) if paths["ai_raw"].exists() else "",
        "crop_tif": str(paths["crop_tif"]) if paths["crop_tif"].exists() else "",
        "pond_points_shp": str(paths["pond_points_shp"]) if paths["pond_points_shp"].exists() else "",
        "pond_points_geojson": str(paths["pond_points_geojson"]) if paths["pond_points_geojson"].exists() else "",
        "pond_points_csv": str(paths["pond_points_csv"]) if paths["pond_points_csv"].exists() else "",
        "error": "" if validation_ok else "Incomplete existing case: " + ";".join(validation_issues),
    })


def validate_case_result(res):
    if not parse_boolish(res.get("ok")):
        return False, ["result_ok_false"]
    if not parse_boolish(res.get("parse_ok")):
        return False, ["parse_ok_false"]
    case_id = res.get("case_id", "")
    case_dir = res.get("case_dir", "")
    issues = []
    if not case_id:
        issues.append("missing_case_id")
    if not case_dir or not Path(case_dir).exists():
        issues.append("missing_case_dir")
    for key in ["report_png", "image_png", "ai_json", "pond_points_geojson", "pond_points_csv"]:
        p = res.get(key, "")
        if not p or not Path(p).exists():
            issues.append(f"missing_{key}")
    if REQUIRE_CROP_TIF_FOR_COMPLETE:
        p = res.get("crop_tif", "")
        if not p or not Path(p).exists():
            issues.append("missing_crop_tif")
    ai_ok, ai_issues = validate_ai_json_file(res.get("ai_json", ""))
    if not ai_ok:
        issues.extend(ai_issues)
    return len(issues) == 0, issues


# =============================================================================
# STUDENT / REVIEW TEMPLATE
# =============================================================================

def student_review_template(case_meta):
    return {
        "case_id": case_meta.get("case_id"),
        "point_id": case_meta.get("point_id"),
        "lon": case_meta.get("lon"),
        "lat": case_meta.get("lat"),
        "zoom_m": case_meta.get("zoom_m"),
        "reviewer_name": "",
        "review_date_local": "",
        "labels": {
            "is_there_surface_water": "",
            "is_there_a_pond": "",
            "road_or_access_nearby": "",
            "pond_count_estimate": None,
            "likely_draftable": "",
            "notes": ""
        },
        "agreement_with_ai": {
            "agree_overall": "",
            "disagreement_fields": [],
            "why": ""
        },
        "qc_flags": {
            "image_quality_issue": False,
            "shadow_confusion": False,
            "seasonal_dryness": False,
            "other": ""
        }
    }


# =============================================================================
# CORE CASE RUNNER
# =============================================================================

def naip_qa_case(
    point_id,
    lon,
    lat,
    out_root,
    question,
    zoom_m=DEFAULT_ZOOM_M,
    system_preamble=None,
    temperature=0,
    export_tif=True,
    export_case_shapefile=True,
    start=DEFAULT_START,
    end=DEFAULT_END,
    attempt_num=1,
):
    out_root = Path(out_root)
    cases_root = ensure_dir(out_root / CASE_DIR_NAME)

    case_id = safe_case_id(point_id, lon, lat)
    case_dir = ensure_dir(cases_root / case_id)

    feat, _aoi = find_best_naip_item(lon, lat, start=start, end=end)
    if feat is None:
        return make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(case_dir),
            attempt_num=attempt_num,
            error="No NAIP scene found in window.",
        )

    item_id = feat.get("id")
    href = extract_asset_href(feat)
    if href is None:
        res = make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(case_dir),
            attempt_num=attempt_num,
            error="NAIP item missing usable image href.",
        )
        res["naip_id"] = item_id or ""
        return res

    href_signed = planetary_computer.sign(href)

    with rasterio.open(href_signed) as src:
        pt = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs=4326).to_crs(src.crs)
        cx, cy = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
        crop_geom_srccrs = [mapping(box(cx - zoom_m, cy - zoom_m, cx + zoom_m, cy + zoom_m))]

        data, crop_transform_affine = mask(src, crop_geom_srccrs, crop=True)
        if data.shape[0] >= 3:
            rgb = data[:3, :, :]
        else:
            rgb = np.repeat(data[0:1, :, :], 3, axis=0)

        rgb_u8 = pct_scale_to_u8(rgb)
        pil = Image.fromarray(np.transpose(rgb_u8, (1, 2, 0)))

        src_crs = src.crs
        src_crs_wkt = src.crs.to_wkt() if src.crs else None
        src_res = getattr(src, "res", None)
        src_bounds = tuple(src.bounds) if getattr(src, "bounds", None) else None

    crop_png = case_dir / "crop.png"
    pil.save(str(crop_png), format="PNG")

    answer_text, parse_ok, ai_obj, usage_info = try_parse_json_strict_or_retry(
        pil,
        question,
        system_preamble=system_preamble,
        temperature=temperature,
        model=OPENAI_MODEL,
    )

    img_w, img_h = pil.size
    ai_obj = normalize_ai_obj(ai_obj, img_w=img_w, img_h=img_h) if isinstance(ai_obj, dict) else None
    ai_fields = ai_fields_for_index(ai_obj if parse_ok else None)

    ai_json_path = case_dir / "ai.json"
    ai_raw_path = case_dir / "ai_raw.txt"

    if isinstance(ai_obj, dict):
        write_json(ai_json_path, ai_obj)
        if ai_raw_path.exists():
            ai_raw_path.unlink()
        answer_text_for_report = json.dumps(ai_obj)
    else:
        obj, norm_text = try_parse_json(answer_text)
        obj = normalize_ai_obj(obj, img_w=img_w, img_h=img_h) if isinstance(obj, dict) else None
        if isinstance(obj, dict):
            write_json(ai_json_path, obj)
            if ai_raw_path.exists():
                ai_raw_path.unlink()
            answer_text_for_report = json.dumps(obj)
            ai_obj = obj
            parse_ok = True
            ai_fields = ai_fields_for_index(ai_obj)
        else:
            write_text(ai_raw_path, norm_text)
            if ai_json_path.exists():
                ai_json_path.unlink()
            answer_text_for_report = norm_text

    report_png = case_dir / "report.png"
    title_text = f"NAIP (~{ASSUMED_GSD_M:.2f} m) @ {lat:.6f}, {lon:.6f}\n{item_id} | scene ~{int(zoom_m * 2)} m x {int(zoom_m * 2)} m"
    write_report_png(pil, title_text, answer_text_for_report, report_png, dpi=200)

    crop_tif_path = ""
    if export_tif:
        crop_tif = case_dir / "crop.tif"
        crop_tif_path = export_crop_tif(href_signed=href_signed, crop_geom=crop_geom_srccrs, out_tif_path=crop_tif)

    pond_points_shp = ""
    pond_points_geojson = ""
    pond_points_csv = ""

    if isinstance(ai_obj, dict):
        pond_points = sanitize_point_list(ai_obj.get("pond_points", []), img_w=img_w, img_h=img_h)
        crop_transform = {
            "transform": crop_transform_affine,
            "img_w": img_w,
            "img_h": img_h,
        }

        case_points_gdf = normalized_points_to_geodataframe(
            point_list=pond_points,
            crop_transform=crop_transform,
            crop_crs=src_crs,
            point_id=point_id,
            case_id=case_id,
            lon=lon,
            lat=lat,
            naip_id=item_id,
        )

        if case_points_gdf is not None and len(case_points_gdf) > 0:
            case_points_gdf_wgs84 = case_points_gdf.to_crs(4326)
        else:
            case_points_gdf_wgs84 = empty_pond_points_gdf(crs="EPSG:4326")

        if export_case_shapefile:
            pond_points_shp = str(case_dir / "ai_pond_points.shp")
            write_vector_safe(case_points_gdf_wgs84, pond_points_shp)

        pond_points_geojson = str(case_dir / "ai_pond_points.geojson")
        write_vector_safe(case_points_gdf_wgs84, pond_points_geojson, driver="GeoJSON")

        pond_points_csv = str(case_dir / "ai_pond_points.csv")
        if len(case_points_gdf_wgs84) > 0:
            tmp = case_points_gdf_wgs84.copy()
            tmp["lon"] = tmp.geometry.x
            tmp["lat"] = tmp.geometry.y
            tmp.drop(columns=["geometry"]).to_csv(pond_points_csv, index=False)
        else:
            pd.DataFrame(columns=[
                "case_id", "point_id", "src_lon", "src_lat", "naip_id",
                "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat"
            ]).to_csv(pond_points_csv, index=False)

    props = feat.get("properties", {}) if isinstance(feat, dict) else {}
    case_meta = {
        "case_id": case_id,
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "scene_width_m": zoom_m * 2,
        "datetime_local": datetime.now().isoformat(timespec="seconds"),
        "assumed_gsd_m_prompt": ASSUMED_GSD_M,
        "naip_id": item_id,
        "naip_href_signed_runtime": href_signed,
        "stac_properties_subset": {
            "datetime": props.get("datetime"),
            "naip:year": props.get("naip:year"),
            "gsd": props.get("gsd"),
            "proj:epsg": props.get("proj:epsg"),
        },
        "src_crs_wkt": src_crs_wkt,
        "src_res": src_res,
        "src_bounds": src_bounds,
        "packages": get_pkg_versions(),
        "parse_ok": bool(parse_ok),
        "openai_model": OPENAI_MODEL,
        "temperature": temperature,
        "start": start,
        "end": end,
        "ai_index_fields": ai_fields,
        "openai_usage": usage_info,
        "run_cost_tracker_snapshot": RUN_COST_TRACKER.copy(),
        "exports": {
            "case_point_shapefile": pond_points_shp,
            "case_point_geojson": pond_points_geojson,
            "case_point_csv": pond_points_csv,
        }
    }
    write_json(case_dir / "meta.json", case_meta)
    write_json(case_dir / "student_review.json", student_review_template(case_meta))
    write_text(
        case_dir / "README.txt",
        "Open report.png first.\n"
        "Red points indicate AI-detected pond/open-water centers.\n"
        "Open ai_pond_points.shp or ai_pond_points.geojson for GIS use.\n"
        "Then fill student_review.json if this case is selected for review.\n"
        "Do NOT treat this as a confirmed drafting source without local/field verification.\n",
    )

    res = {
        "ok": True,
        "final_complete": False,
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": str(case_dir),
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": item_id or "",
        "parse_ok": bool(parse_ok),
        "validation_ok": False,
        "validation_issues": "",
        **ai_fields,
        "api_calls_for_case": usage_info["num_api_calls_for_case"],
        "prompt_tokens": usage_info["prompt_tokens"],
        "completion_tokens": usage_info["completion_tokens"],
        "total_tokens": usage_info["total_tokens"],
        "estimated_cost_usd": round(usage_info["estimated_cost_usd"], 6),
        "report_png": str(report_png),
        "image_png": str(crop_png),
        "ai_json": str(ai_json_path) if ai_json_path.exists() else "",
        "ai_raw": str(ai_raw_path) if ai_raw_path.exists() else "",
        "crop_tif": crop_tif_path or "",
        "pond_points_shp": pond_points_shp,
        "pond_points_geojson": pond_points_geojson,
        "pond_points_csv": pond_points_csv,
        "error": "",
    }

    validation_ok, validation_issues = validate_case_result(res)
    res["validation_ok"] = bool(validation_ok)
    res["validation_issues"] = ";".join(validation_issues)
    res["final_complete"] = bool(validation_ok)
    if not validation_ok:
        res["ok"] = False
        res["error"] = "Case failed validation: " + ";".join(validation_issues)

    return coerce_result_fields(res)


# =============================================================================
# INPUT CSV READING
# =============================================================================

def read_points_csv(points_csv, zoom_m_default=DEFAULT_ZOOM_M, start=DEFAULT_START, end=DEFAULT_END):
    points_csv = Path(points_csv)
    if not points_csv.exists():
        raise FileNotFoundError(f"Input points CSV does not exist: {points_csv}")

    with points_csv.open("r", encoding="utf-8-sig", newline="") as f_in:
        sample = f_in.read(8192)
        f_in.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";", "|"])
        except Exception:
            dialect = csv.get_dialect("excel")
            dialect.delimiter = "\t" if "\t" in sample and "," not in sample else ","
        reader = csv.DictReader(f_in, dialect=dialect)
        if not reader.fieldnames:
            raise RuntimeError("Could not read header row from points file.")

        raw_fields = list(reader.fieldnames)
        norm_fields = [((h or "").strip().lower()) for h in raw_fields]
        header_map = {raw: norm for raw, norm in zip(raw_fields, norm_fields)}

        def get_value(row, *keys):
            keys = set(keys)
            for k in keys:
                if k in row and row[k] not in (None, ""):
                    return row[k]
            for raw_k, norm_k in header_map.items():
                if norm_k in keys:
                    v = row.get(raw_k, "")
                    if v not in (None, ""):
                        return v
            return ""

        rows = []
        for i, row in enumerate(reader, start=1):
            point_id = get_value(row, "id", "point_id", "name", "fid") or str(i)
            lon_s = get_value(row, "lon", "longitude", "x")
            lat_s = get_value(row, "lat", "latitude", "y")
            if lon_s == "" or lat_s == "":
                rows.append({
                    "input_order": i,
                    "input_valid": False,
                    "point_id": point_id,
                    "lon": np.nan,
                    "lat": np.nan,
                    "zoom_m": zoom_m_default,
                    "start": start,
                    "end": end,
                    "input_error": f"Missing lon/lat in row {i}: {row}",
                })
                continue
            try:
                lon = float(str(lon_s).strip())
                lat = float(str(lat_s).strip())
            except Exception as e:
                rows.append({
                    "input_order": i,
                    "input_valid": False,
                    "point_id": point_id,
                    "lon": np.nan,
                    "lat": np.nan,
                    "zoom_m": zoom_m_default,
                    "start": start,
                    "end": end,
                    "input_error": f"Bad lon/lat in row {i}: {type(e).__name__}: {e}",
                })
                continue

            zoom_s = get_value(row, "zoom_m", "zoom", "buffer_m")
            try:
                zoom_m = float(str(zoom_s).strip()) if zoom_s not in ("", None) else float(zoom_m_default)
            except Exception:
                zoom_m = float(zoom_m_default)

            row_start = (get_value(row, "start") or start).strip()
            row_end = (get_value(row, "end") or end).strip()

            rows.append({
                "input_order": i,
                "input_valid": True,
                "point_id": point_id,
                "lon": lon,
                "lat": lat,
                "zoom_m": zoom_m,
                "start": row_start,
                "end": row_end,
                "input_error": "",
            })

    df = pd.DataFrame(rows)
    df["case_id"] = df.apply(
        lambda r: safe_case_id(r["point_id"], r["lon"], r["lat"]) if bool(r["input_valid"]) else "",
        axis=1,
    )
    return df


# =============================================================================
# BATCH SHAPEFILE / MERGED OUTPUT HELPERS
# =============================================================================

def build_batch_pond_points_layers(index_csv_path, out_root):
    out_root = Path(out_root)
    index_csv_path = Path(index_csv_path)
    merged = []

    if not index_csv_path.exists():
        return ""

    idx = pd.read_csv(index_csv_path, dtype=str).fillna("")
    for _, row in idx.iterrows():
        csv_path = str(row.get("pond_points_csv", "")).strip()
        if csv_path and Path(csv_path).exists():
            try:
                pts = pd.read_csv(csv_path)
                if len(pts) > 0 and {"lon", "lat"}.issubset(pts.columns):
                    for f in [
                        "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
                        "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json"
                    ]:
                        pts[f] = row.get(f, "")
                    merged.append(pts)
            except Exception as e:
                print(f"Warning: could not read pond point CSV {csv_path}: {e}")

    batch_shp = out_root / "batch_ai_pond_points.shp"
    batch_geojson = out_root / "batch_ai_pond_points.geojson"
    batch_gpkg = out_root / "batch_ai_pond_points.gpkg"
    batch_csv = out_root / "batch_ai_pond_points.csv"

    if len(merged) == 0:
        out_csv = pd.DataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id", "pond_label", "confidence",
            "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat",
            "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
            "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json"
        ])
        out_csv.to_csv(batch_csv, index=False)
        empty = empty_pond_points_gdf(crs="EPSG:4326")
        write_vector_safe(empty, batch_shp)
        write_vector_safe(empty, batch_geojson, driver="GeoJSON")
        write_vector_safe(empty, batch_gpkg)
        return str(batch_shp)

    all_pts = pd.concat(merged, ignore_index=True)
    all_pts.to_csv(batch_csv, index=False)
    gdf = gpd.GeoDataFrame(
        all_pts,
        geometry=gpd.points_from_xy(all_pts["lon"].astype(float), all_pts["lat"].astype(float)),
        crs="EPSG:4326",
    )
    write_vector_safe(gdf, batch_shp)
    write_vector_safe(gdf, batch_geojson, driver="GeoJSON")
    write_vector_safe(gdf, batch_gpkg)
    return str(batch_shp)


def write_index_csv(index_path, rows):
    index_path = Path(index_path)
    ensure_dir(index_path.parent)
    with index_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=INDEX_FIELDNAMES)
        writer.writeheader()
        for row in rows:
            writer.writerow(coerce_result_fields(row))
    return str(index_path)


def append_attempt_csv(attempts_path, row):
    attempts_path = Path(attempts_path)
    ensure_dir(attempts_path.parent)
    with ATTEMPTS_CSV_LOCK:
        write_header = not attempts_path.exists()
        with attempts_path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=INDEX_FIELDNAMES)
            if write_header:
                writer.writeheader()
            writer.writerow(coerce_result_fields(row))


# =============================================================================
# RESULTS PACKAGING
# =============================================================================

def safe_read_index(index_csv):
    if not Path(index_csv).exists():
        return pd.DataFrame(columns=INDEX_FIELDNAMES)
    return pd.read_csv(index_csv, dtype=str).fillna("")


def make_all_grid_gdf(idx):
    if len(idx) == 0:
        return gpd.GeoDataFrame(columns=list(idx.columns), geometry=[], crs="EPSG:4326")
    df = idx.copy()
    df["lon_num"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat_num"] = pd.to_numeric(df["lat"], errors="coerce")
    df = df.dropna(subset=["lon_num", "lat_num"])
    if len(df) == 0:
        return gpd.GeoDataFrame(columns=list(idx.columns), geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(
        df.drop(columns=["lon_num", "lat_num"]),
        geometry=gpd.points_from_xy(df["lon_num"], df["lat_num"]),
        crs="EPSG:4326",
    )


def read_candidate_points_from_index(idx):
    merged = []
    for _, row in idx.iterrows():
        p = str(row.get("pond_points_csv", "")).strip()
        if not p or not Path(p).exists():
            continue
        try:
            pts = pd.read_csv(p)
            if len(pts) == 0:
                continue
            for f in [
                "ok", "final_complete", "validation_ok", "validation_issues",
                "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
                "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json", "case_dir"
            ]:
                pts[f] = row.get(f, "")
            merged.append(pts)
        except Exception as e:
            print(f"Warning: failed reading candidate points {p}: {e}")

    if len(merged) == 0:
        return pd.DataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id", "pond_label", "confidence",
            "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat",
            "ai_likely_draftable", "ai_road_or_access_nearby", "report_png", "image_png", "ai_json"
        ])
    return pd.concat(merged, ignore_index=True)


def build_results_summary(idx, cand_df, attempts_df=None):
    total_input = len(idx)
    ok = idx["ok"].map(parse_boolish).sum() if "ok" in idx else 0
    complete = idx["final_complete"].map(parse_boolish).sum() if "final_complete" in idx else 0
    failures = total_input - complete

    def count_value(field, val):
        if field not in idx.columns:
            return 0
        return int((idx[field].map(safe_lower) == val).sum())

    summary = {
        "created_local": datetime.now().isoformat(timespec="seconds"),
        "model": OPENAI_MODEL,
        "assumed_gsd_m": ASSUMED_GSD_M,
        "total_input_points": int(total_input),
        "successful_complete_interpretations": int(complete),
        "ok_rows": int(ok),
        "failed_or_incomplete_rows": int(failures),
        "surface_water_yes": count_value("ai_is_there_surface_water", "yes"),
        "pond_yes": count_value("ai_is_there_a_pond", "yes"),
        "road_or_access_yes": count_value("ai_road_or_access_nearby", "yes"),
        "road_or_access_uncertain": count_value("ai_road_or_access_nearby", "uncertain"),
        "likely_draftable_yes": count_value("ai_likely_draftable", "yes"),
        "likely_draftable_uncertain": count_value("ai_likely_draftable", "uncertain"),
        "candidate_pond_center_points": int(len(cand_df)),
        "run_cost_tracker": RUN_COST_TRACKER.copy(),
    }
    if attempts_df is not None and len(attempts_df) > 0:
        summary["total_attempt_rows"] = int(len(attempts_df))
        summary["attempt_rows_ok"] = int(attempts_df["ok"].map(parse_boolish).sum()) if "ok" in attempts_df else 0
    return summary


def write_simple_bar(labels, values, title, ylabel, out_png):
    out_png = Path(out_png)
    ensure_dir(out_png.parent)
    fig, ax = plt.subplots(figsize=(8, 5), dpi=200)
    ax.bar(labels, values)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=25)
    for i, v in enumerate(values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def write_map_figure(all_gdf, cand_gdf, out_png):
    out_png = Path(out_png)
    ensure_dir(out_png.parent)
    fig, ax = plt.subplots(figsize=(8, 8), dpi=220)
    if all_gdf is not None and len(all_gdf) > 0:
        all_gdf.plot(ax=ax, markersize=5, color="lightgray", edgecolor="none", label="Interpreted image chips")
    if cand_gdf is not None and len(cand_gdf) > 0:
        cand_gdf.plot(ax=ax, markersize=18, color="red", edgecolor="black", linewidth=0.3, label="AI candidate pond centers")
    ax.set_title("AI-screened candidate pond centers")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def copy_review_reports(idx, cand_df, review_dir):
    review_dir = ensure_dir(review_dir)
    copied = []

    # Copy candidate reports first.
    if len(cand_df) > 0:
        tmp = cand_df.copy()
        if "confidence" in tmp.columns:
            tmp["confidence_num"] = pd.to_numeric(tmp["confidence"], errors="coerce").fillna(-1)
            tmp = tmp.sort_values("confidence_num", ascending=False)
        seen = set()
        for _, row in tmp.iterrows():
            case_id = str(row.get("case_id", ""))
            if not case_id or case_id in seen:
                continue
            seen.add(case_id)
            src = str(row.get("report_png", ""))
            if src and Path(src).exists():
                dst = review_dir / "candidate_reports" / f"candidate_{len(copied)+1:04d}_{clean_filename(case_id, 80)}.png"
                ensure_dir(dst.parent)
                shutil.copy2(src, dst)
                copied.append(str(dst))
                if len(copied) >= MAX_REVIEW_REPORTS_TO_COPY:
                    break

    # Copy a small set of negative reports for comparison.
    neg_dir = review_dir / "negative_examples"
    neg_count = 0
    if len(idx) > 0:
        neg = idx[idx.get("ai_is_there_a_pond", pd.Series([""] * len(idx))).map(safe_lower) == "no"].head(MAX_EXAMPLE_REPORTS_PER_GROUP)
        for _, row in neg.iterrows():
            src = str(row.get("report_png", ""))
            case_id = str(row.get("case_id", ""))
            if src and Path(src).exists():
                dst = neg_dir / f"negative_{neg_count+1:04d}_{clean_filename(case_id, 80)}.png"
                ensure_dir(dst.parent)
                shutil.copy2(src, dst)
                neg_count += 1

    return copied


def build_results_folder(out_root, points_csv, prompt_text, system_preamble):
    out_root = Path(out_root)
    results_root = ensure_dir(out_root / RESULTS_DIR_NAME)
    tables_dir = ensure_dir(results_root / "tables")
    gis_dir = ensure_dir(results_root / "gis")
    figs_dir = ensure_dir(results_root / "figures")
    review_dir = ensure_dir(results_root / "review_pack")
    qa_dir = ensure_dir(results_root / "qa")
    docs_dir = ensure_dir(results_root / "documents")

    index_csv = out_root / "index.csv"
    attempts_csv = out_root / "index_attempts.csv"
    idx = safe_read_index(index_csv)
    attempts_df = safe_read_index(attempts_csv) if attempts_csv.exists() else pd.DataFrame()
    cand_df = read_candidate_points_from_index(idx)

    # Tables.
    idx.to_csv(tables_dir / "final_index_one_row_per_input_point.csv", index=False)
    cand_df.to_csv(tables_dir / "candidate_pond_centers.csv", index=False)
    if len(attempts_df) > 0:
        attempts_df.to_csv(qa_dir / "all_attempts_including_retries.csv", index=False)

    failures = idx[~idx["final_complete"].map(parse_boolish)].copy() if len(idx) > 0 else idx.copy()
    failures.to_csv(qa_dir / "FAILURES_NEED_REVIEW.csv", index=False)
    failures.to_csv(tables_dir / "failures_need_review.csv", index=False)

    # GIS outputs.
    all_gdf = make_all_grid_gdf(idx)
    if len(all_gdf) > 0:
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.geojson", driver="GeoJSON")
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.gpkg")
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.shp")

    if len(cand_df) > 0:
        cand_gdf = gpd.GeoDataFrame(
            cand_df.copy(),
            geometry=gpd.points_from_xy(cand_df["lon"].astype(float), cand_df["lat"].astype(float)),
            crs="EPSG:4326",
        )
    else:
        cand_gdf = empty_pond_points_gdf(crs="EPSG:4326")

    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.geojson", driver="GeoJSON")
    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.gpkg")
    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.shp")

    # Keep compatibility batch outputs at root too.
    try:
        build_batch_pond_points_layers(index_csv, out_root)
    except Exception as e:
        print(f"Warning: failed to write root batch layers: {e}")

    # Figures.
    summary = build_results_summary(idx, cand_df, attempts_df=attempts_df)
    summary_rows = pd.DataFrame([summary])
    summary_rows.to_csv(tables_dir / "run_summary.csv", index=False)
    write_json(tables_dir / "run_summary.json", summary)

    write_simple_bar(
        ["Input chips", "Successful", "Failed/incomplete", "Candidate centers"],
        [
            summary["total_input_points"],
            summary["successful_complete_interpretations"],
            summary["failed_or_incomplete_rows"],
            summary["candidate_pond_center_points"],
        ],
        "Run summary",
        "Count",
        figs_dir / "run_summary_counts.png",
    )

    write_simple_bar(
        ["Surface water yes", "Pond yes", "Access yes", "Likely draftable yes"],
        [
            summary["surface_water_yes"],
            summary["pond_yes"],
            summary["road_or_access_yes"],
            summary["likely_draftable_yes"],
        ],
        "AI interpretation counts",
        "Image-chip count",
        figs_dir / "ai_interpretation_counts.png",
    )

    write_map_figure(all_gdf, cand_gdf, figs_dir / "candidate_pond_centers_map.png")

    # Review-pack image copies.
    copied_reports = copy_review_reports(idx, cand_df, review_dir)

    # Prompt / config / context docs.
    write_text(docs_dir / "prompt_used.txt", prompt_text)
    write_text(docs_dir / "system_preamble_used.txt", system_preamble)
    write_json(docs_dir / "run_config.json", {
        "points_csv": str(points_csv),
        "out_root": str(out_root),
        "openai_model": OPENAI_MODEL,
        "assumed_gsd_m": ASSUMED_GSD_M,
        "default_zoom_m": DEFAULT_ZOOM_M,
        "default_scene_width_m": DEFAULT_ZOOM_M * 2,
        "default_start": DEFAULT_START,
        "default_end": DEFAULT_END,
        "force_rerun_all": FORCE_RERUN_ALL,
        "rerun_failed_or_invalid": RERUN_FAILED_OR_INVALID,
        "max_attempts_per_case": MAX_ATTEMPTS_PER_CASE,
        "require_crop_tif_for_complete": REQUIRE_CROP_TIF_FOR_COMPLETE,
        "packages": get_pkg_versions(),
    })

    context_md = f"""# Results context for AI writing

This folder contains a production-style output package for the NAIP AI pond/drafting-location screening workflow.

## Main interpretation summary

- Input image chips / grid points: {summary['total_input_points']}
- Successful complete interpretations: {summary['successful_complete_interpretations']}
- Failed or incomplete interpretations after retries: {summary['failed_or_incomplete_rows']}
- Image chips where AI reported surface water = yes: {summary['surface_water_yes']}
- Image chips where AI reported pond = yes: {summary['pond_yes']}
- Image chips where AI reported visible nearby road/access = yes: {summary['road_or_access_yes']}
- Image chips where AI reported likely_draftable = yes: {summary['likely_draftable_yes']}
- Candidate pond center points exported to GIS: {summary['candidate_pond_center_points']}

## How to use this folder to write a Results section

Use these files first:

1. `tables/run_summary.csv` — one-row summary of the run.
2. `tables/final_index_one_row_per_input_point.csv` — one row per input image chip/grid point.
3. `tables/candidate_pond_centers.csv` — one row per mapped AI pond-center point.
4. `gis/candidate_pond_centers.gpkg` or `.geojson` — GIS-ready candidate pond-center layer.
5. `gis/all_interpreted_grid_points.gpkg` or `.geojson` — GIS-ready layer of all interpreted chip centers.
6. `figures/candidate_pond_centers_map.png` — simple map figure for the Results section.
7. `figures/run_summary_counts.png` and `figures/ai_interpretation_counts.png` — simple count figures.
8. `qa/FAILURES_NEED_REVIEW.csv` — any chips that still failed or were incomplete after retries.
9. `review_pack/candidate_reports/` — report images for candidate detections.

## Suggested Results framing

The primary result is a GIS-ready set of candidate pond-based drafting locations, not a confirmed operational drafting inventory. The AI screened fixed-area NAIP image chips for visible open water and nearby vehicle-access context. Candidate points represent approximate pond centers derived from the AI-reported normalized image coordinates. Locations should be described as candidate water-source locations requiring firefighter review, landowner coordination, and field verification.

## Placeholder language

Use `XXXX` for any values that require manual expert review, field confirmation, landowner status, seasonal water persistence, measured depth, or operational safety assessment.
"""
    write_text(results_root / "RESULTS_CONTEXT_FOR_AI.md", context_md)

    readme = f"""# RESULTS folder

This folder was automatically generated by the final NAIP AI pond-screening workflow.

## Folder contents

- `tables/`: CSV tables for writing the Results section.
- `gis/`: GIS-ready outputs for candidate pond centers and all interpreted grid points.
- `figures/`: simple figures for a manuscript/report Results section.
- `review_pack/`: copied report images for manual review.
- `qa/`: retry logs, failed cases, and validation outputs.
- `documents/`: prompt, system preamble, run configuration, and metadata.

## Most important outputs

- Candidate pond centers: `gis/candidate_pond_centers.gpkg`
- All interpreted chip centers: `gis/all_interpreted_grid_points.gpkg`
- Final chip-level table: `tables/final_index_one_row_per_input_point.csv`
- Candidate point table: `tables/candidate_pond_centers.csv`
- Summary table: `tables/run_summary.csv`
- AI writing context: `RESULTS_CONTEXT_FOR_AI.md`

## Run completion

- Input image chips / grid points: {summary['total_input_points']}
- Successful complete interpretations: {summary['successful_complete_interpretations']}
- Failed or incomplete after retries: {summary['failed_or_incomplete_rows']}
- Candidate pond center points: {summary['candidate_pond_center_points']}

If `qa/FAILURES_NEED_REVIEW.csv` has rows, those rows did not fully complete after retries. They were not silently skipped.
"""
    write_text(results_root / "README_RESULTS.md", readme)

    validation_report = {
        "summary": summary,
        "failures_csv": str(qa_dir / "FAILURES_NEED_REVIEW.csv"),
        "candidate_reports_copied": len(copied_reports),
        "results_root": str(results_root),
    }
    write_json(qa_dir / "validation_report.json", validation_report)

    print("Wrote RESULTS folder:", results_root)
    print(json.dumps(summary, indent=2))
    return str(results_root)


# =============================================================================
# BATCH RUNNER
# =============================================================================

def run_one_case_with_retries(row, out_root, question, system_preamble, temperature, export_tif, export_case_shapefile, attempts_path, log_path):
    point_id = row["point_id"]
    lon = float(row["lon"])
    lat = float(row["lat"])
    zoom_m = float(row["zoom_m"])
    start = row["start"]
    end = row["end"]
    case_id = row["case_id"]

    if not FORCE_RERUN_ALL:
        complete, issues = case_is_complete(out_root, case_id)
        if complete:
            print(f"Skipping complete case: {case_id}")
            return reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0)
        elif not RERUN_FAILED_OR_INVALID:
            print(f"Existing incomplete case will not be rerun because RERUN_FAILED_OR_INVALID=False: {case_id}")
            return reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0)
        else:
            print(f"Rerunning incomplete/failed case: {case_id} | issues={';'.join(issues)}")

    last_res = None
    for attempt_num in range(1, MAX_ATTEMPTS_PER_CASE + 1):
        print(f"Running case {case_id} | attempt {attempt_num}/{MAX_ATTEMPTS_PER_CASE}")
        try:
            res = naip_qa_case(
                point_id=point_id,
                lon=lon,
                lat=lat,
                out_root=out_root,
                question=question,
                zoom_m=zoom_m,
                system_preamble=system_preamble,
                temperature=temperature,
                export_tif=export_tif,
                export_case_shapefile=export_case_shapefile,
                start=start,
                end=end,
                attempt_num=attempt_num,
            )
        except Exception as e:
            tb = traceback.format_exc()
            err = f"{type(e).__name__}: {e}"
            with LOG_FILE_LOCK:
                with Path(log_path).open("a", encoding="utf-8") as f_log:
                    f_log.write(f"\n[{datetime.now().isoformat(timespec='seconds')}] case_id={case_id} attempt={attempt_num}\n{err}\n{tb}\n")
            res = make_error_result(
                point_id=point_id,
                lon=lon,
                lat=lat,
                zoom_m=zoom_m,
                case_id=case_id,
                case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
                attempt_num=attempt_num,
                error=err,
            )

        append_attempt_csv(attempts_path, res)
        last_res = res

        ok, issues = validate_case_result(res)
        res["validation_ok"] = bool(ok)
        res["validation_issues"] = ";".join(issues)
        res["final_complete"] = bool(ok)
        if ok:
            res["ok"] = True
            res["error"] = ""
            return coerce_result_fields(res)

        print(f"Case did not validate: {case_id} | attempt {attempt_num} | issues={';'.join(issues)}")
        if attempt_num < MAX_ATTEMPTS_PER_CASE:
            time.sleep(SLEEP_BETWEEN_ATTEMPTS_SEC)

    if last_res is None:
        last_res = make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
            attempt_num=MAX_ATTEMPTS_PER_CASE,
            error="No attempts were completed.",
        )

    final_ok, final_issues = validate_case_result(last_res)
    last_res["validation_ok"] = bool(final_ok)
    last_res["validation_issues"] = ";".join(final_issues)
    last_res["final_complete"] = bool(final_ok)
    last_res["ok"] = bool(final_ok)
    if not final_ok and not last_res.get("error"):
        last_res["error"] = "Case failed final validation: " + ";".join(final_issues)
    return coerce_result_fields(last_res)


def run_batch_points_csv(
    points_csv,
    out_root,
    question,
    zoom_m_default=DEFAULT_ZOOM_M,
    system_preamble=None,
    temperature=0,
    export_tif=True,
    export_case_shapefile=True,
    export_batch_shapefile=True,
    start=DEFAULT_START,
    end=DEFAULT_END,
    max_parallel_workers=MAX_PARALLEL_WORKERS,
):
    out_root = ensure_dir(out_root)
    ensure_dir(out_root / CASE_DIR_NAME)

    index_path = out_root / "index.csv"
    attempts_path = out_root / "index_attempts.csv"
    log_path = out_root / "batch_log.txt"

    points_df = read_points_csv(points_csv, zoom_m_default=zoom_m_default, start=start, end=end)
    points_df.to_csv(out_root / "input_points_parsed.csv", index=False)

    # Store final rows by input_order so index.csv remains in original input order,
    # even when cases finish out of order in parallel.
    final_rows_by_order = {}

    invalid_inputs = points_df[~points_df["input_valid"]]
    for _, bad in invalid_inputs.iterrows():
        input_order = int(bad.get("input_order", len(final_rows_by_order) + 1))
        res = make_error_result(
            point_id=bad.get("point_id", ""),
            lon="",
            lat="",
            zoom_m=bad.get("zoom_m", ""),
            case_id="",
            case_dir="",
            attempt_num=0,
            error=bad.get("input_error", "Invalid input row."),
        )
        append_attempt_csv(attempts_path, res)
        final_rows_by_order[input_order] = res

    valid_points = points_df[points_df["input_valid"]].copy()
    n = len(valid_points)

    workers = int(max_parallel_workers or 1)
    workers = max(1, workers)

    def ordered_final_rows():
        return [final_rows_by_order[k] for k in sorted(final_rows_by_order.keys())]

    print(f"\nStarting batch run with MAX_PARALLEL_WORKERS={workers}")
    print(f"Valid input points: {n} | Invalid input rows: {len(invalid_inputs)}")

    if workers == 1:
        # Sequential mode. This is useful for debugging.
        for i, (_, row) in enumerate(valid_points.iterrows(), start=1):
            print(f"\n========== {i}/{n} | point_id={row['point_id']} | case_id={row['case_id']} ==========")
            res = run_one_case_with_retries(
                row=row,
                out_root=out_root,
                question=question,
                system_preamble=system_preamble,
                temperature=temperature,
                export_tif=export_tif,
                export_case_shapefile=export_case_shapefile,
                attempts_path=attempts_path,
                log_path=log_path,
            )
            final_rows_by_order[int(row["input_order"])] = res
            write_index_csv(index_path, ordered_final_rows())
            print_cost_summary(prefix="RUNNING TOTAL")
    else:
        # Parallel mode. Each worker processes one full case, including its retries.
        futures = {}
        with ThreadPoolExecutor(max_workers=workers) as executor:
            for i, (_, row) in enumerate(valid_points.iterrows(), start=1):
                print(f"Submitting {i}/{n} | point_id={row['point_id']} | case_id={row['case_id']}")
                fut = executor.submit(
                    run_one_case_with_retries,
                    row=row,
                    out_root=out_root,
                    question=question,
                    system_preamble=system_preamble,
                    temperature=temperature,
                    export_tif=export_tif,
                    export_case_shapefile=export_case_shapefile,
                    attempts_path=attempts_path,
                    log_path=log_path,
                )
                futures[fut] = (i, int(row["input_order"]), row["point_id"], row["case_id"])

            completed = 0
            for fut in as_completed(futures):
                i, input_order, point_id, case_id = futures[fut]
                completed += 1
                try:
                    res = fut.result()
                except Exception as e:
                    tb = traceback.format_exc()
                    err = f"{type(e).__name__}: {e}"
                    with LOG_FILE_LOCK:
                        with Path(log_path).open("a", encoding="utf-8") as f_log:
                            f_log.write(f"\n[{datetime.now().isoformat(timespec='seconds')}] case_id={case_id} future_failed\n{err}\n{tb}\n")
                    # Pull lon/lat/zoom from the input row for a useful final failure row.
                    row_match = valid_points[valid_points["input_order"].astype(int) == int(input_order)].iloc[0]
                    res = make_error_result(
                        point_id=point_id,
                        lon=row_match.get("lon", ""),
                        lat=row_match.get("lat", ""),
                        zoom_m=row_match.get("zoom_m", ""),
                        case_id=case_id,
                        case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
                        attempt_num=MAX_ATTEMPTS_PER_CASE,
                        error=err,
                    )
                    append_attempt_csv(attempts_path, res)

                final_rows_by_order[input_order] = res
                write_index_csv(index_path, ordered_final_rows())

                ok_txt = "OK" if parse_boolish(res.get("final_complete")) else "FAILED"
                print(f"\nCompleted {completed}/{n} | original_submit={i}/{n} | {ok_txt} | point_id={point_id} | case_id={case_id}")
                print_cost_summary(prefix="RUNNING TOTAL")

    write_index_csv(index_path, ordered_final_rows())

    if export_batch_shapefile:
        try:
            batch_shp = build_batch_pond_points_layers(index_path, out_root)
            print("Wrote batch pond shapefile:", batch_shp)
        except Exception as e:
            print(f"Warning: failed to build batch shapefile: {e}")

    write_text(
        out_root / "README_STUDENTS.txt",
        "Student workflow:\n"
        "1) Open RESULTS/README_RESULTS.md first.\n"
        "2) Open RESULTS/tables/final_index_one_row_per_input_point.csv for chip-level results.\n"
        "3) Open RESULTS/gis/candidate_pond_centers.gpkg for mapped candidate pond centers.\n"
        "4) Open report.png files in CASES folders or RESULTS/review_pack/candidate_reports for visual review.\n"
        "5) Treat all mapped points as candidate drafting locations only. Field/local verification is required.\n",
    )

    build_results_folder(
        out_root=out_root,
        points_csv=points_csv,
        prompt_text=question,
        system_preamble=system_preamble or "",
    )

    return str(index_path)


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    MODE = "batch"

    out_root = DEFAULT_OUT_ROOT
    points_csv = DEFAULT_POINTS_CSV
    temperature = 0

    scene_width_m = int(DEFAULT_ZOOM_M * 2)
    question = build_pond_prompt(scene_width_m=scene_width_m)
    system_preamble = build_system_preamble()

    if MODE == "single":
        point_id = "pt001test23"
        lon, lat = -105.3227622, 40.55704649

        res = naip_qa_case(
            point_id=point_id,
            lon=lon,
            lat=lat,
            out_root=out_root,
            question=question,
            zoom_m=DEFAULT_ZOOM_M,
            system_preamble=system_preamble,
            temperature=temperature,
            export_tif=True,
            export_case_shapefile=True,
            start=DEFAULT_START,
            end=DEFAULT_END,
            attempt_num=1,
        )
        print(json.dumps(res, indent=2))
        build_results_folder(
            out_root=out_root,
            points_csv=points_csv,
            prompt_text=question,
            system_preamble=system_preamble,
        )
        print_cost_summary(prefix="SESSION TOTAL")

    else:
        idx = run_batch_points_csv(
            points_csv=points_csv,
            out_root=out_root,
            question=question,
            zoom_m_default=DEFAULT_ZOOM_M,
            system_preamble=system_preamble,
            temperature=temperature,
            export_tif=True,
            export_case_shapefile=True,
            export_batch_shapefile=True,
            start=DEFAULT_START,
            end=DEFAULT_END,
            max_parallel_workers=MAX_PARALLEL_WORKERS,
        )
        print("Wrote index:", idx)
        print_cost_summary(prefix="SESSION TOTAL")


In [ ]:
# This script converts the main index.csv rows into map point
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# =============================================================================
# INPUTS
# =============================================================================

index_csv = r"C:\Users\index.csv"
out_dir = r"C:\Users\"
os.makedirs(out_dir, exist_ok=True)

out_shp_all = os.path.join(out_dir, "ai_pond_points_all_500m.shp")
out_shp_yes = os.path.join(out_dir, "ai_pond_points_yes_500m.shp")
out_shp_no = os.path.join(out_dir, "ai_pond_points_no_500m.shp")
out_plot = os.path.join(out_dir, "ai_is_there_a_pond_map_500m.png")
out_csv_clean = os.path.join(out_dir, "ai_pond_points_clean_500m.csv")

# =============================================================================
# READ CSV
# =============================================================================

df = pd.read_csv(index_csv)

# Keep only rows with valid lat/lon
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df = df.dropna(subset=["lat", "lon"]).copy()

# Normalize the pond field
df["ai_is_there_a_pond"] = (
    df["ai_is_there_a_pond"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Optional: keep only yes/no rows for plotting/classification
valid_labels = ["yes", "no"]
df["pond_class"] = df["ai_is_there_a_pond"].where(df["ai_is_there_a_pond"].isin(valid_labels), "other")

# Save cleaned CSV
df.to_csv(out_csv_clean, index=False)

# =============================================================================
# CREATE GEODATAFRAME
# =============================================================================

geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Shapefile field names should be short to avoid truncation issues
gdf["pond_ai"] = gdf["pond_class"]

# Save all points
gdf.to_file(out_shp_all)

# Save yes/no subsets
gdf_yes = gdf[gdf["pond_ai"] == "yes"].copy()
gdf_no = gdf[gdf["pond_ai"] == "no"].copy()

if len(gdf_yes) > 0:
    gdf_yes.to_file(out_shp_yes)

if len(gdf_no) > 0:
    gdf_no.to_file(out_shp_no)

# =============================================================================
# PLOT
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Plot "no" first so "yes" draws on top
if len(gdf_no) > 0:
    gdf_no.plot(ax=ax, markersize=12, label="No pond", alpha=0.7)

if len(gdf_yes) > 0:
    gdf_yes.plot(ax=ax, markersize=18, label="Yes pond", alpha=0.9)

gdf_other = gdf[gdf["pond_ai"] == "other"].copy()
if len(gdf_other) > 0:
    gdf_other.plot(ax=ax, markersize=10, label="Other/blank", alpha=0.5)

ax.set_title("AI Pond Classification from index.csv")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(out_plot, dpi=300)
plt.show()

# =============================================================================
# SUMMARY
# =============================================================================

print("\nDone.")
print(f"Input CSV: {index_csv}")
print(f"Clean CSV: {out_csv_clean}")
print(f"All-points shapefile: {out_shp_all}")
print(f"Yes-points shapefile: {out_shp_yes if len(gdf_yes) > 0 else 'No YES records to save'}")
print(f"No-points shapefile: {out_shp_no if len(gdf_no) > 0 else 'No NO records to save'}")
print(f"Plot: {out_plot}")
print("\nCounts:")
print(gdf["pond_ai"].value_counts(dropna=False))

In [ ]:
# THIS script exports 2 - all pond centers and only likely draftable pond centers
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path

# =============================================================================
# INPUTS
# =============================================================================

INDEX_CSV = r"C:\Users\index.csv"

OUT_DIR = r"C:\Users"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DRAFTABLE_SHP = os.path.join(OUT_DIR, "ai_draftable_pond_centers_500m.shp")
OUT_DRAFTABLE_GEOJSON = os.path.join(OUT_DIR, "ai_draftable_pond_centers_500m.geojson")
OUT_DRAFTABLE_CSV = os.path.join(OUT_DIR, "ai_draftable_pond_centers_500m.csv")

OUT_ALL_PONDS_WITH_STATUS_SHP = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_500m.shp")
OUT_ALL_PONDS_WITH_STATUS_GEOJSON = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_500m.geojson")
OUT_ALL_PONDS_WITH_STATUS_CSV = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_500m.csv")

# =============================================================================
# HELPERS
# =============================================================================

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def short_shp_columns(gdf):
    """
    Rename columns to shapefile-safe names <= 10 characters.
    """
    rename = {
        "case_id": "case_id",
        "point_id": "point_id",
        "src_lon": "src_lon",
        "src_lat": "src_lat",
        "naip_id": "naip_id",
        "pond_lbl": "pond_lbl",
        "pond_label": "pond_lbl",
        "conf": "conf",
        "confidence": "conf",
        "norm_x": "norm_x",
        "norm_y": "norm_y",
        "pix_x": "pix_x",
        "pixel_x": "pix_x",
        "pix_y": "pix_y",
        "pixel_y": "pix_y",
        "ai_is_there_surface_water": "ai_sw",
        "ai_is_there_a_pond": "ai_pond",
        "ai_road_or_access_nearby": "ai_access",
        "ai_likely_draftable": "ai_draft",
        "ai_pond_count": "ai_pcnt",
        "report_png": "report_png",
        "image_png": "image_png",
        "ai_json": "ai_json",
        "pond_points_shp": "src_shp",
        "pond_points_geojson": "src_geojs",
        "pond_points_csv": "src_csv",
    }

    out = gdf.copy()
    out = out.rename(columns={c: rename[c] for c in out.columns if c in rename})

    # Shapefile is picky about object fields and long strings
    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)

    return out

# =============================================================================
# READ INDEX
# =============================================================================

if not os.path.exists(INDEX_CSV):
    raise FileNotFoundError(f"Index CSV not found:\n{INDEX_CSV}")

idx = pd.read_csv(INDEX_CSV)

required_cols = [
    "pond_points_shp",
    "ai_likely_draftable",
    "ai_is_there_a_pond",
]

missing = [c for c in required_cols if c not in idx.columns]
if missing:
    raise ValueError(
        "The index.csv is missing required columns:\n"
        + "\n".join(missing)
        + "\n\nAvailable columns are:\n"
        + "\n".join(idx.columns)
    )

idx["ai_likely_draftable_norm"] = idx["ai_likely_draftable"].apply(norm_text)
idx["ai_is_there_a_pond_norm"] = idx["ai_is_there_a_pond"].apply(norm_text)

# Keep only rows where AI said likely_draftable == yes
draftable_idx = idx[idx["ai_likely_draftable_norm"] == "yes"].copy()

print("Index rows:", len(idx))
print("Rows where ai_likely_draftable == yes:", len(draftable_idx))

# =============================================================================
# MERGE ALL POND POINTS WITH STATUS
# =============================================================================

all_pond_gdfs = []
draftable_gdfs = []

for _, row in idx.iterrows():
    shp_path = str(row.get("pond_points_shp", "")).strip()

    if not shp_path or not os.path.exists(shp_path):
        continue

    try:
        ponds = gpd.read_file(shp_path)
    except Exception as e:
        print(f"Could not read: {shp_path}")
        print(f"  {type(e).__name__}: {e}")
        continue

    if ponds.empty:
        continue

    # Add classification fields from index.csv to each pond center point
    ponds["ai_is_there_surface_water"] = row.get("ai_is_there_surface_water", "")
    ponds["ai_is_there_a_pond"] = row.get("ai_is_there_a_pond", "")
    ponds["ai_road_or_access_nearby"] = row.get("ai_road_or_access_nearby", "")
    ponds["ai_likely_draftable"] = row.get("ai_likely_draftable", "")
    ponds["ai_pond_count"] = row.get("ai_pond_count", "")
    ponds["report_png"] = row.get("report_png", "")
    ponds["image_png"] = row.get("image_png", "")
    ponds["ai_json"] = row.get("ai_json", "")
    ponds["pond_points_shp"] = shp_path
    ponds["pond_points_geojson"] = row.get("pond_points_geojson", "")
    ponds["pond_points_csv"] = row.get("pond_points_csv", "")

    all_pond_gdfs.append(ponds)

    if norm_text(row.get("ai_likely_draftable", "")) == "yes":
        draftable_gdfs.append(ponds.copy())

# =============================================================================
# EXPORT ALL POND CENTERS WITH DRAFTABLE STATUS
# =============================================================================

if len(all_pond_gdfs) > 0:
    all_ponds = gpd.GeoDataFrame(
        pd.concat(all_pond_gdfs, ignore_index=True),
        geometry="geometry",
        crs=all_pond_gdfs[0].crs
    )

    if all_ponds.crs is None:
        all_ponds = all_ponds.set_crs("EPSG:4326")

    all_ponds_wgs84 = all_ponds.to_crs("EPSG:4326")

    all_ponds_wgs84["lon"] = all_ponds_wgs84.geometry.x
    all_ponds_wgs84["lat"] = all_ponds_wgs84.geometry.y

    all_ponds_wgs84.drop(columns="geometry").to_csv(OUT_ALL_PONDS_WITH_STATUS_CSV, index=False)
    all_ponds_wgs84.to_file(OUT_ALL_PONDS_WITH_STATUS_GEOJSON, driver="GeoJSON")

    all_ponds_shp = short_shp_columns(all_ponds_wgs84)
    all_ponds_shp.to_file(OUT_ALL_PONDS_WITH_STATUS_SHP)

    print("\nWrote all pond centers with draftable status:")
    print(OUT_ALL_PONDS_WITH_STATUS_SHP)
    print(OUT_ALL_PONDS_WITH_STATUS_GEOJSON)
    print(OUT_ALL_PONDS_WITH_STATUS_CSV)

else:
    print("\nNo pond center shapefiles found to merge.")

# =============================================================================
# EXPORT ONLY DRAFTABLE POND CENTERS
# =============================================================================

if len(draftable_gdfs) > 0:
    draftable = gpd.GeoDataFrame(
        pd.concat(draftable_gdfs, ignore_index=True),
        geometry="geometry",
        crs=draftable_gdfs[0].crs
    )

    if draftable.crs is None:
        draftable = draftable.set_crs("EPSG:4326")

    draftable_wgs84 = draftable.to_crs("EPSG:4326")

    draftable_wgs84["lon"] = draftable_wgs84.geometry.x
    draftable_wgs84["lat"] = draftable_wgs84.geometry.y

    draftable_wgs84.drop(columns="geometry").to_csv(OUT_DRAFTABLE_CSV, index=False)
    draftable_wgs84.to_file(OUT_DRAFTABLE_GEOJSON, driver="GeoJSON")

    draftable_shp = short_shp_columns(draftable_wgs84)
    draftable_shp.to_file(OUT_DRAFTABLE_SHP)

    print("\nWrote draftable pond centers only:")
    print(OUT_DRAFTABLE_SHP)
    print(OUT_DRAFTABLE_GEOJSON)
    print(OUT_DRAFTABLE_CSV)

    print("\nDraftable pond count:")
    print(len(draftable_wgs84))

else:
    print("\nNo records where ai_likely_draftable == yes were found.")
    print("No draftable pond shapefile was created.")

# =============================================================================
# SUMMARY
# =============================================================================

print("\nSummary from index.csv:")
print(idx["ai_likely_draftable_norm"].value_counts(dropna=False))

print("\nDone.")

In [ ]:
# SHAPEFILE TO GPKG


import os
import geopandas as gpd

# ============================================================
# INPUTS
# ============================================================

input_shp = r"C:\Users\RCFVD_BNDRY.shp"
output_gpkg = r"C:\Users\RCFVD_BNDRY.gpkg"

# Optional: name of the layer inside the GeoPackage
layer_name = "converted_shapefile"

# ============================================================
# CONVERT SHAPEFILE TO GEOPACKAGE
# ============================================================

# Check input exists
if not os.path.exists(input_shp):
    raise FileNotFoundError(f"Input shapefile not found: {input_shp}")

# Create output folder if needed
out_dir = os.path.dirname(output_gpkg)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)

# Read shapefile
gdf = gpd.read_file(input_shp)

# Save to GeoPackage
gdf.to_file(output_gpkg, layer=layer_name, driver="GPKG")

print("Done.")
print(f"Input features: {len(gdf)}")
print(f"Output GeoPackage: {output_gpkg}")
print(f"Layer name: {layer_name}")

# 1000 m sensitivity analysis 

In [ ]:
# THIS MAKES A GRID OF POINTS
import os
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pyproj import CRS

# =========================
# USER SETTINGS
# =========================
IN_SHP = r"C:\Users\RCFVD_BNDRY.shp"
OUT_DIR = r"C:\Users\"
SPACING_M = 1000.0  # desired spacing in METERS (always)

# If your boundary is actually EPSG:2876 but the shapefile is missing/incorrectly labeled,
# set this to True to force-assign EPSG:2876 without reprojecting.
FORCE_ASSIGN_EPSG2876_IF_MISSING = True

# Outputs
OUT_POINTS_SHP_SRC = os.path.join(OUT_DIR, "RCVFD_grid_points_spacing1000m_SRC_CRS.shp")
OUT_POINTS_SHP_4326 = os.path.join(OUT_DIR, "RCVFD_grid_points_spacing1000m_WGS84.shp")
OUT_POINTS_CSV = os.path.join(OUT_DIR, "RCVFD_grid_points_spacing1000m.csv")

os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# HELPERS
# =========================
def _crs_units_and_to_crs_units_per_meter(crs: CRS):
    """
    Returns:
      units_name (str): e.g. "metre", "US survey foot", "foot"
      crs_units_per_meter (float): multiplier such that:
         spacing_in_crs_units = spacing_meters * crs_units_per_meter
    """
    # Default assumption: meters
    units_name = "metre"
    crs_units_per_meter = 1.0

    try:
        # For projected CRS, axis_info[0].unit_name often exists
        axis = crs.axis_info[0]
        units_name = (axis.unit_name or "").lower()

        # Use explicit conversions for common cases
        if "metre" in units_name or "meter" in units_name:
            crs_units_per_meter = 1.0
            units_name = axis.unit_name
        elif "us survey foot" in units_name or "foot_us" in units_name or "u.s. survey foot" in units_name:
            # 1 US survey foot = 1200/3937 meters => feet per meter = 3937/1200
            crs_units_per_meter = 3937.0 / 1200.0
            units_name = axis.unit_name
        elif units_name.strip() == "foot" or "international foot" in units_name:
            # 1 international foot = 0.3048 meters => feet per meter = 1/0.3048
            crs_units_per_meter = 1.0 / 0.3048
            units_name = axis.unit_name
        else:
            # Fallback: try to infer from unit conversion factor if available
            # pyproj doesn't always expose a direct factor cleanly; keep meters if unknown
            units_name = axis.unit_name or "unknown"
            crs_units_per_meter = 1.0
    except Exception:
        units_name = "unknown"
        crs_units_per_meter = 1.0

    return units_name, crs_units_per_meter

# =========================
# READ + PREP BOUNDARY
# =========================
bndry = gpd.read_file(IN_SHP)
if bndry.empty:
    raise RuntimeError("Boundary shapefile read as empty.")

# Fix/assign CRS if needed
if bndry.crs is None:
    if FORCE_ASSIGN_EPSG2876_IF_MISSING:
        bndry = bndry.set_crs(epsg=2876, allow_override=True)
    else:
        raise RuntimeError("Input shapefile has no CRS. Set it in GIS or set FORCE_ASSIGN_EPSG2876_IF_MISSING=True.")

src_crs = CRS.from_user_input(bndry.crs)
units_name, crs_units_per_meter = _crs_units_and_to_crs_units_per_meter(src_crs)

spacing_crs_units = SPACING_M * crs_units_per_meter

print("Boundary CRS:", src_crs.to_string())
print("Detected linear units:", units_name)
print(f"Requested spacing: {SPACING_M} m")
print(f"Using spacing in CRS units: {spacing_crs_units:.6f} ({units_name})")

# Dissolve to a single geometry for filtering
bndry_union = bndry.geometry.unary_union

# =========================
# BUILD REGULAR GRID IN SOURCE CRS UNITS
# =========================
minx, miny, maxx, maxy = bndry_union.bounds

xs = np.arange(minx, maxx + spacing_crs_units, spacing_crs_units)
ys = np.arange(miny, maxy + spacing_crs_units, spacing_crs_units)

pts = [Point(x, y) for y in ys for x in xs]
gpts = gpd.GeoDataFrame({"geometry": pts}, crs=bndry.crs)

# Filter to within boundary (strict). Use intersects if you want boundary-inclusive.
mask_within = gpts.within(bndry_union)
gpts_in = gpts.loc[mask_within].copy()

# Add stable ID and coordinates in source CRS
gpts_in.reset_index(drop=True, inplace=True)
gpts_in["id"] = [f"pt_{i:06d}" for i in range(len(gpts_in))]
gpts_in["x_src"] = gpts_in.geometry.x
gpts_in["y_src"] = gpts_in.geometry.y

# =========================
# WRITE OUTPUTS
# =========================
# 1) Shapefile in source CRS
gpts_in.to_file(OUT_POINTS_SHP_SRC)

# 2) Reproject to WGS84, add lon/lat, save shapefile
gpts_wgs84 = gpts_in.to_crs(epsg=4326)
gpts_wgs84["lon"] = gpts_wgs84.geometry.x
gpts_wgs84["lat"] = gpts_wgs84.geometry.y
gpts_wgs84.to_file(OUT_POINTS_SHP_4326)

# 3) CSV for NAIP batch (id, lon, lat)
gpts_wgs84[["id", "lon", "lat"]].to_csv(OUT_POINTS_CSV, index=False)

print("Done.")
print(f"Points created (within boundary): {len(gpts_in):,}")
print("Wrote:")
print(" -", OUT_POINTS_SHP_SRC)
print(" -", OUT_POINTS_SHP_4326)
print(" -", OUT_POINTS_CSV)

In [ ]:
# THIS IS THE MODEL RUN - 1000m 
# FINAL NAIP POND / DRAFTING LOCATION WORKFLOW
# =============================================================================
# Purpose:
#   1) Run NAIP image chips through a conservative multimodal AI pond-screening prompt.
#   2) Prevent silent skipped failures: cases are skipped only when they are truly complete.
#   3) Retry failed, invalid, or ok=False cases.
#   4) Build a clean RESULTS folder that can be handed to another AI or reviewer to write Results.
#
# Main outputs:
#   OUT_ROOT/
#       CASES/                         per-chip outputs
#       index.csv                      one final row per input point
#       index_attempts.csv             every attempt, including retries
#       batch_log.txt                  errors and traceback log
#       RESULTS/
#           README_RESULTS.md
#           RESULTS_CONTEXT_FOR_AI.md
#           tables/
#           gis/
#           figures/
#           review_pack/
#           qa/
#           documents/
# =============================================================================

import os
import csv
import json
import time
import base64
import hashlib
import traceback
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from io import BytesIO

import pandas as pd
import numpy as np
from PIL import Image

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import rasterio
from rasterio.mask import mask
from rasterio.transform import xy as rio_xy

import geopandas as gpd
from shapely.geometry import shape, Point, box, mapping

import planetary_computer
from openai import OpenAI
import matplotlib.pyplot as plt

try:
    from importlib.metadata import version as pkg_version
except Exception:
    pkg_version = None


# =============================================================================
# CONFIG
# =============================================================================

OPENAI_MODEL = "gpt-5.1"
# OPENAI_MODEL = "gpt-4o"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Set your OPENAI_API_KEY in the environment"
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_PRICING_PER_1M = {
    "gpt-5.1": {"input": 1.25, "output": 10.00},
    "gpt-5": {"input": 1.25, "output": 10.00},
    "gpt-4o": {"input": 2.50, "output": 10.00},
}

RUN_COST_TRACKER = {
    "calls": 0,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
}

# Thread locks used when MAX_PARALLEL_WORKERS > 1.
# These prevent shared CSV/log/cost/matplotlib outputs from colliding across workers.
RUN_COST_LOCK = threading.Lock()
ATTEMPTS_CSV_LOCK = threading.Lock()
LOG_FILE_LOCK = threading.Lock()
MATPLOTLIB_LOCK = threading.Lock()
PRINT_LOCK = threading.Lock()

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SEARCH_URL = STAC_URL.rstrip("/") + "/search"

ASSUMED_GSD_M = 0.30

# zoom_m is half the scene width.
# 250 = 500 m x 500 m chips.
# 500 = 1000 m x 1000 m chips.
DEFAULT_ZOOM_M = 500
DEFAULT_START = "2023-01-01"
DEFAULT_END = "2024-12-31"

# Change this to a new folder for your final production run.
DEFAULT_OUT_ROOT = r"C:\Users\"

# Input CSV must have lon/lat fields, or longitude/latitude, or x/y.
DEFAULT_POINTS_CSV = r"C:\Users\RCVFD_grid_points_spacing1000m.csv"

CASE_DIR_NAME = "CASES"
RESULTS_DIR_NAME = "RESULTS"

# Final-run safeguards.
FORCE_RERUN_ALL = False
RERUN_FAILED_OR_INVALID = True
MAX_ATTEMPTS_PER_CASE = 3
SLEEP_BETWEEN_ATTEMPTS_SEC = 3

# Parallel processing.
# 1 = sequential. 3 is a good safe default for OpenAI + NAIP + local file writing.
MAX_PARALLEL_WORKERS = 1

# Outputs required for a case to be treated as complete.
REQUIRE_CROP_TIF_FOR_COMPLETE = True
REQUIRE_CASE_SHAPEFILE_FOR_COMPLETE = False

# Results packaging.
MAX_REVIEW_REPORTS_TO_COPY = 999999
MAX_EXAMPLE_REPORTS_PER_GROUP = 999999


# =============================================================================
# COST HELPERS
# =============================================================================

def get_model_pricing(model_name):
    return MODEL_PRICING_PER_1M.get(model_name, {"input": 0.0, "output": 0.0})


def estimate_cost_usd(model_name, prompt_tokens, completion_tokens):
    pricing = get_model_pricing(model_name)
    in_cost = (prompt_tokens or 0) / 1_000_000.0 * pricing["input"]
    out_cost = (completion_tokens or 0) / 1_000_000.0 * pricing["output"]
    return in_cost + out_cost


def update_run_cost_tracker(prompt_tokens, completion_tokens, total_tokens, cost_usd):
    with RUN_COST_LOCK:
        RUN_COST_TRACKER["calls"] += 1
        RUN_COST_TRACKER["prompt_tokens"] += int(prompt_tokens or 0)
        RUN_COST_TRACKER["completion_tokens"] += int(completion_tokens or 0)
        RUN_COST_TRACKER["total_tokens"] += int(total_tokens or 0)
        RUN_COST_TRACKER["estimated_cost_usd"] += float(cost_usd or 0.0)


def print_cost_summary(prefix="RUN"):
    with RUN_COST_LOCK:
        snap = RUN_COST_TRACKER.copy()
    print(
        f"[{prefix}] "
        f"calls={snap['calls']} | "
        f"prompt_tokens={snap['prompt_tokens']} | "
        f"completion_tokens={snap['completion_tokens']} | "
        f"total_tokens={snap['total_tokens']} | "
        f"estimated_cost_usd=${snap['estimated_cost_usd']:.6f}"
    )


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def write_json(path, obj):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def write_text(path, text):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")


def safe_lower(v):
    return "" if pd.isna(v) else str(v).strip().lower()


def parse_boolish(v):
    s = safe_lower(v)
    if s in {"true", "1", "yes", "y", "ok"}:
        return True
    if s in {"false", "0", "no", "n", ""}:
        return False
    return False


def clean_filename(s, max_len=120):
    s = "" if s is None else str(s)
    keep = []
    for ch in s:
        if ch.isalnum() or ch in "._-":
            keep.append(ch)
        else:
            keep.append("_")
    out = "".join(keep).strip("_")
    while "__" in out:
        out = out.replace("__", "_")
    return out[:max_len] if len(out) > max_len else out


def safe_case_id(row_id, lon, lat):
    rid = "point" if row_id in (None, "") else str(row_id)
    base = f"{rid}__{lat:.6f}__{lon:.6f}"
    h = hashlib.sha1(base.encode("utf-8")).hexdigest()[:8]
    safe = clean_filename(base.replace("-", "m").replace(".", "p"), max_len=140)
    return f"{safe}__{h}"


def get_pkg_versions():
    keys = [
        "numpy", "pandas", "rasterio", "geopandas", "shapely",
        "planetary-computer", "requests", "Pillow", "matplotlib", "openai"
    ]
    out = {}
    if pkg_version is None:
        return out
    for k in keys:
        try:
            out[k] = pkg_version(k)
        except Exception:
            pass
    return out


# =============================================================================
# NETWORK
# =============================================================================

def make_retry_session(
    total=10,
    backoff_factor=0.8,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET", "POST"),
    timeout=(10, 90),
):
    s = requests.Session()
    retry = Retry(
        total=total,
        connect=total,
        read=total,
        status=total,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=set(allowed_methods),
        raise_on_status=False,
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update(
        {
            "User-Agent": "naip-pond-final-results-workflow/1.0 (+python requests)",
            "Accept": "application/geo+json, application/json",
            "Content-Type": "application/json",
        }
    )
    return s, timeout


SESSION, REQ_TIMEOUT = make_retry_session()


# =============================================================================
# STAC SEARCH
# =============================================================================

def stac_search_naip(intersects_geojson, datetime_range, limit=200):
    payload = {
        "collections": ["naip"],
        "intersects": intersects_geojson,
        "datetime": datetime_range,
        "limit": limit,
    }
    last_err = None
    for attempt in range(1, 6):
        try:
            r = SESSION.post(SEARCH_URL, data=json.dumps(payload), timeout=REQ_TIMEOUT)
            r.raise_for_status()
            fc = r.json()
            return fc.get("features", []) or []
        except Exception as e:
            last_err = e
            time.sleep(min(2**attempt, 20))
    raise RuntimeError(f"STAC search failed after retries: {last_err}")


def find_best_naip_item(lon, lat, start=DEFAULT_START, end=DEFAULT_END, pad_deg=0.001):
    aoi = {
        "type": "Polygon",
        "coordinates": [[
            [lon - pad_deg, lat - pad_deg],
            [lon + pad_deg, lat - pad_deg],
            [lon + pad_deg, lat + pad_deg],
            [lon - pad_deg, lat + pad_deg],
            [lon - pad_deg, lat - pad_deg],
        ]],
    }

    feats = stac_search_naip(intersects_geojson=aoi, datetime_range=f"{start}/{end}", limit=200)
    if not feats:
        return None, aoi

    aoi_shape = shape(aoi)
    feats_sorted = sorted(
        feats,
        key=lambda f: shape(f["geometry"]).intersection(aoi_shape).area,
        reverse=True,
    )
    return feats_sorted[0], aoi


def extract_asset_href(feat):
    if not isinstance(feat, dict):
        return None

    assets = feat.get("assets")
    if not isinstance(assets, dict):
        assets = feat.get("properties", {}).get("assets", None)

    if not isinstance(assets, dict) or not assets:
        return None

    if "image" in assets and isinstance(assets["image"], dict) and "href" in assets["image"]:
        return assets["image"]["href"]

    for a in assets.values():
        if isinstance(a, dict) and str(a.get("href", "")).lower().endswith((".tif", ".tiff")):
            return a["href"]

    return None


# =============================================================================
# IMAGE HELPERS
# =============================================================================

def pct_scale_to_u8(rgb_arr):
    if rgb_arr.dtype == np.uint8:
        return rgb_arr
    out = []
    for b in range(rgb_arr.shape[0]):
        band = rgb_arr[b].astype(np.float32)
        if not np.isfinite(band).any():
            out.append(np.zeros_like(band, dtype=np.uint8))
            continue
        lo, hi = np.nanpercentile(band, [0.0, 99.5])
        if not np.isfinite(lo):
            lo = 0.0
        if not np.isfinite(hi) or hi <= lo:
            hi = lo + 1.0
        band = (np.clip((band - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
        out.append(band)
    return np.stack(out, axis=0)


def try_parse_json(text):
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.strip("`").strip()
        if t.lower().startswith("json"):
            t = t[4:].strip()
    if "{" in t and "}" in t:
        t2 = t[t.find("{"): t.rfind("}") + 1]
    else:
        t2 = t
    try:
        return json.loads(t2), t2
    except Exception:
        return None, t2


def export_crop_tif(href_signed, crop_geom, out_tif_path):
    with rasterio.open(href_signed) as src:
        data, out_transform = mask(src, crop_geom, crop=True)
        out_meta = src.meta.copy()
        out_meta.update(
            {
                "driver": "GTiff",
                "height": data.shape[1],
                "width": data.shape[2],
                "transform": out_transform,
                "count": data.shape[0],
                "compress": "deflate",
                "tiled": True,
                "blockxsize": 256,
                "blockysize": 256,
            }
        )
        out_tif_path = Path(out_tif_path)
        ensure_dir(out_tif_path.parent)
        with rasterio.open(out_tif_path, "w", **out_meta) as dst:
            dst.write(data)
    return str(out_tif_path)


def sanitize_point_list(points, img_w, img_h):
    clean = []
    if not isinstance(points, list):
        return clean

    for i, p in enumerate(points):
        if not isinstance(p, dict):
            continue
        try:
            x = float(p.get("x", np.nan))
            y = float(p.get("y", np.nan))
        except Exception:
            continue
        if not all(np.isfinite([x, y])):
            continue
        x = max(0.0, min(1.0, x))
        y = max(0.0, min(1.0, y))
        px = int(round(x * (img_w - 1)))
        py = int(round(y * (img_h - 1)))
        conf = p.get("confidence", None)
        try:
            conf = None if conf is None else max(0.0, min(1.0, float(conf)))
        except Exception:
            conf = None
        clean.append(
            {
                "label": str(p.get("label", f"pond_{i+1}")),
                "confidence": conf,
                "x": x,
                "y": y,
                "pixel_x": px,
                "pixel_y": py,
            }
        )
    return clean


def normalized_points_to_geodataframe(point_list, crop_transform, crop_crs, point_id, case_id, lon, lat, naip_id):
    rows = []
    if not point_list:
        return gpd.GeoDataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id",
            "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y"
        ], geometry=[], crs=crop_crs)

    img_h = int(crop_transform["img_h"])
    img_w = int(crop_transform["img_w"])
    aff = crop_transform["transform"]

    for p in point_list:
        px = max(0, min(img_w - 1, int(p["pixel_x"])))
        py = max(0, min(img_h - 1, int(p["pixel_y"])))
        map_x, map_y = rio_xy(aff, py, px, offset="center")
        rows.append(
            {
                "case_id": case_id,
                "point_id": point_id,
                "src_lon": lon,
                "src_lat": lat,
                "naip_id": naip_id,
                "pond_label": p.get("label", ""),
                "confidence": p.get("confidence", None),
                "norm_x": p.get("x", None),
                "norm_y": p.get("y", None),
                "pixel_x": px,
                "pixel_y": py,
                "geometry": Point(map_x, map_y),
            }
        )
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crop_crs)


def empty_pond_points_gdf(crs="EPSG:4326"):
    return gpd.GeoDataFrame(
        columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id",
            "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y"
        ],
        geometry=[],
        crs=crs,
    )


def write_vector_safe(gdf, path, driver=None):
    path = Path(path)
    ensure_dir(path.parent)
    out = gdf.copy() if gdf is not None else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    if out.crs is None:
        out = out.set_crs(4326)

    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)

    if path.suffix.lower() == ".shp":
        rename_map = {
            "pond_label": "pond_lbl",
            "confidence": "conf",
            "pixel_x": "pix_x",
            "pixel_y": "pix_y",
            "likely_draftable": "likely_dr",
            "road_or_access_nearby": "road_acc",
            "is_there_surface_water": "surf_wtr",
            "is_there_a_pond": "pond",
        }
        out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
        out.to_file(path)
    else:
        if driver:
            out.to_file(path, driver=driver)
        elif path.suffix.lower() == ".geojson":
            out.to_file(path, driver="GeoJSON")
        elif path.suffix.lower() == ".gpkg":
            layer_name = clean_filename(path.stem, max_len=60) or "layer"
            out.to_file(path, driver="GPKG", layer=layer_name)
        else:
            out.to_file(path)
    return str(path)


def _write_report_png_unlocked(pil_image, title_text, json_text, out_path, dpi=200):
    obj, norm_text = try_parse_json(json_text)
    display_text = json.dumps(obj, indent=2) if obj is not None else norm_text

    img_w, img_h = pil_image.size
    pond_points = []
    if isinstance(obj, dict):
        pond_points = sanitize_point_list(obj.get("pond_points", []), img_w=img_w, img_h=img_h)

    fig = plt.figure(figsize=(13, 6), dpi=dpi)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.0])
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])

    ax0.imshow(pil_image)
    ax0.axis("off")
    ax0.set_title(title_text, fontsize=10)

    for idx, p in enumerate(pond_points, start=1):
        x = p["pixel_x"]
        y = p["pixel_y"]
        ax0.scatter([x], [y], s=60, c="red", marker="o", edgecolors="white", linewidths=0.8)
        label = f"{idx}"
        conf = p.get("confidence", None)
        if conf is not None:
            label = f"{idx} ({float(conf):.2f})"
        ax0.text(
            x + 4,
            max(0, y - 4),
            label,
            color="white",
            fontsize=8,
            fontweight="bold",
            bbox=dict(facecolor="red", edgecolor="red", boxstyle="round,pad=0.2"),
        )

    ax1.axis("off")
    ax1.text(
        0.0,
        1.0,
        display_text,
        va="top",
        ha="left",
        family="monospace",
        fontsize=8.5,
        wrap=True,
    )

    plt.tight_layout()
    out_path = Path(out_path)
    ensure_dir(out_path.parent)
    plt.savefig(str(out_path), bbox_inches="tight")
    plt.close(fig)


def write_report_png(pil_image, title_text, json_text, out_path, dpi=200):
    # Matplotlib uses global state, so protect figure creation/saving when running threads.
    with MATPLOTLIB_LOCK:
        return _write_report_png_unlocked(pil_image, title_text, json_text, out_path, dpi=dpi)


# =============================================================================
# OPENAI VISION
# =============================================================================

def ask_image_question(pil_image, question, system_preamble=None, model=OPENAI_MODEL, temperature=0):
    buf = BytesIO()
    pil_image.save(buf, format="PNG")
    img_data_url = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("utf-8")

    sys_msg = system_preamble or "You are a careful remote sensing analyst. Answer concisely and only from the image."

    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": sys_msg},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": img_data_url}},
                ],
            },
        ],
        temperature=temperature,
    )

    answer_text = resp.choices[0].message.content.strip()

    usage = getattr(resp, "usage", None)
    prompt_tokens = getattr(usage, "prompt_tokens", 0) if usage else 0
    completion_tokens = getattr(usage, "completion_tokens", 0) if usage else 0
    total_tokens = getattr(usage, "total_tokens", 0) if usage else 0

    est_cost_usd = estimate_cost_usd(
        model_name=model,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
    )

    update_run_cost_tracker(
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        cost_usd=est_cost_usd,
    )

    print(
        f"[API CALL] model={model} | "
        f"prompt_tokens={prompt_tokens} | "
        f"completion_tokens={completion_tokens} | "
        f"total_tokens={total_tokens} | "
        f"estimated_cost_usd=${est_cost_usd:.6f}"
    )

    usage_info = {
        "model": model,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": est_cost_usd,
    }
    return answer_text, usage_info


def try_parse_json_strict_or_retry(pil, question, system_preamble, temperature=0, model=OPENAI_MODEL):
    ans1, usage1 = ask_image_question(
        pil,
        question,
        system_preamble=system_preamble,
        temperature=temperature,
        model=model,
    )
    obj1, _ = try_parse_json(ans1)
    if obj1 is not None:
        combined_usage = {
            "model": model,
            "prompt_tokens": usage1["prompt_tokens"],
            "completion_tokens": usage1["completion_tokens"],
            "total_tokens": usage1["total_tokens"],
            "estimated_cost_usd": usage1["estimated_cost_usd"],
            "num_api_calls_for_case": 1,
        }
        return ans1, True, obj1, combined_usage

    retry_q = (
        "Your previous response was not valid JSON.\n\n"
        "Return ONLY valid JSON matching the exact schema requested. "
        "Do not include markdown, prose, or comments.\n\n"
        + question
    )

    ans2, usage2 = ask_image_question(
        pil,
        retry_q,
        system_preamble=system_preamble,
        temperature=temperature,
        model=model,
    )
    obj2, _ = try_parse_json(ans2)

    combined_usage = {
        "model": model,
        "prompt_tokens": usage1["prompt_tokens"] + usage2["prompt_tokens"],
        "completion_tokens": usage1["completion_tokens"] + usage2["completion_tokens"],
        "total_tokens": usage1["total_tokens"] + usage2["total_tokens"],
        "estimated_cost_usd": usage1["estimated_cost_usd"] + usage2["estimated_cost_usd"],
        "num_api_calls_for_case": 2,
    }
    return ans2, (obj2 is not None), obj2, combined_usage


# =============================================================================
# PROMPT
# =============================================================================

def build_pond_prompt(scene_width_m=None):
    scene_width_m = scene_width_m or int(DEFAULT_ZOOM_M * 2)
    return rf"""
Analyze this NAIP RGB aerial image (~{ASSUMED_GSD_M:.2f} m / 30 cm ground sample distance).
Use ONLY what is visible in the image. Do not assume anything not directly observable.

This task must be conservative. False positives are worse than false negatives.
The primary goal is to identify visible open-water ponds that could plausibly be wildfire drafting sources.

Return ONLY valid JSON.

Context:
This image is a fixed-area crop (~{scene_width_m} m × {scene_width_m} m). The goal is to identify true visible open surface water
and specifically ponds or standing water bodies that might serve as potential drafting sources.

Whole-scene context rule:
- Do not judge a candidate feature in isolation.
- Compare the candidate to the rest of the image before deciding it is water.
- Use the whole scene to determine whether the dark area is more consistent with shadow, terrain shading, tree shadow, rock shadow, or other non-water dark features that appear elsewhere in the image.
- If similar dark shapes, tones, or textures occur throughout the scene in obvious shadows or shaded terrain, prefer "no" for pond.

Definitions (image-only):

Surface water:
Visible open water such as pond, lake, reservoir, stream reach with visible water, canal holding water,
or wetland/open water patch.

Water surface appearance rule:
- Open water usually appears smoother and more internally uniform than mud, grass, sediment, or disturbed ground.
- If the interior shows mottled texture, vegetation patches, hoof-disturbed mud, exposed sediment, or land-like texture across most of the basin, do NOT classify it as open water.

Pond:
A distinct, bounded, mostly standing open-water body with a visible open-water surface and visible edges or shoreline.

Pond basin / impoundment footprint:
A depression, stock tank, basin, or pond-shaped feature that may be dry, muddy, vegetated, only faintly damp,
or may contain too little visible water to matter operationally. A pond basin is NOT the same as visible open water.

Draftable pond:
A visible open-water pond that appears, from imagery alone, to be plausibly usable as a wildfire drafting source.
Positive evidence includes visible open water, a distinct shoreline, nearby road or vehicle access, and a feature that is not obviously too small.
This is only an image-based screening judgment and does not confirm depth, volume, bank stability, legal access, seasonal persistence, or actual field operability.

Manmade rectangular feature exclusion rules:
- Do NOT consider anything perfectly square or perfectly rectangular to be a pond unless unmistakable open water is clearly visible.
- Perfect geometric shapes are strong negative evidence for ponds in this task and should usually be treated as manmade features, not open-water ponds.
- Small dark rectangular or square features near houses, sheds, driveways, pads, or developed areas should NOT be classified as ponds unless clear open water is unmistakably visible.
- Roofs, sheds, covered tanks, liners, tarps, equipment pads, shadowed structures, and other manmade site features can appear dark and water-like from overhead.
- A neat rectangular feature is more likely to be a manmade object than a natural or draftable pond unless there is strong visible evidence of true open water.
- If a feature is adjacent to buildings or a maintained homesite, be especially cautious and prefer "no" unless water is obvious.

Shadow comparison rule:
- Before classifying a feature as pond, compare it to other dark areas in the image.
- If the candidate has similar darkness, texture, edge quality, or orientation as nearby shadows from trees, rocks, cliffs, buildings, or terrain, do NOT classify it as a pond.
- If the dark feature blends gradually into surrounding shadow or appears connected to shadowed terrain, choose "no" for pond.

Scene consistency rule:
- A true pond should remain visually distinct from the broader shadow pattern of the scene.
- If the feature can be explained by the same lighting and shadow behavior seen elsewhere in the image, it is not strong evidence for water.
- Use surrounding illumination, terrain, and nearby shadows to interpret ambiguous dark features conservatively.

Contextual shadow rule for trees and rock:
- In wooded, rocky, or mountainous scenes, many dark features are caused by tree shadow, terrain shadow, or rock relief.
- If a candidate dark feature resembles the shadow behavior of nearby trees, ridges, boulders, or slopes, do not classify it as a pond unless open water is unmistakable.

Important interpretation rules:
1. Darkness alone is NOT evidence of water.
2. Do NOT confuse shadow, vegetation, wet ground, mud, burn scar, terrain shading, roofs, tanks, troughs, containers,
   disturbed pads, equipment, or structures with ponds.
3. A true pond with open water requires BOTH:
   - a visible bounded shoreline or edge, AND
   - an interior that visibly looks like open water
4. A coherent basin shape alone is NOT enough.
5. If a feature appears mostly dry, muddy, vegetated, shallow, or only faintly damp, do NOT classify it as a pond with open water.
6. If a pond-shaped basin is visible but open water is not clearly visible, choose "no" for pond.
7. If a basin or stock tank footprint is visible without clear open water, do NOT classify it as a pond.
8. If uncertain between water and shadow, vegetation, wet ground, mud, structure, or tank, choose "no" for pond.
9. Small dark rectangular features in developed or disturbed areas are often NOT ponds.
10. Tiny dark features may be houses, sheds, tanks, pads, containers, shadows, or other manmade features. Do not call them ponds unless open water is unmistakable.
11. For wildfire drafting, size matters. If a feature looks too small, too narrow, too shallow-looking, or operationally insignificant, do not mark it draftable.
12. If the feature may contain water but appears too small to realistically support drafting operations, set "likely_draftable" to "no" or "uncertain".
13. If uncertain, choose "no" for pond and explain why.
14. If a feature is uncertain, do NOT include a point.
15. If there are no visible ponds, return an empty list for pond_points.
16. In steep, rocky, alpine, or heavily textured terrain, dark enclosed features are often shadows, rock hollows, or terrain depressions rather than open water.
17. In rocky terrain, do NOT classify a feature as a pond unless the interior appears smooth and water-like, with a clearly bounded shoreline that is distinct from surrounding rock texture and shadow.
18. If a dark feature contains visible internal texture, irregular rock pattern, or tonal variation similar to surrounding rock/shadow, do NOT classify it as open water.
19. Small dark pockets embedded in broken rock or cliff-like terrain should usually be treated as shadow or rock unless open water is unmistakable.
20. If the feature could be explained by terrain shadow or rock geometry, choose "no" for pond.

Output JSON with exactly these keys:

{{
  "is_there_surface_water": "<yes|no>",
  "is_there_a_pond": "<yes|no>",
  "road_or_access_nearby": "yes|no|uncertain",
  "likely_draftable": "<yes|no|uncertain>",
  "pond_count": <integer>,
  "evidence": "<=90 words",
  "notes": "<concise operational note>",
  "pond_points": [
    {{
      "label": "<pond>",
      "confidence": <float 0-1>,
      "x": <float 0-1>,
      "y": <float 0-1>
    }}
  ]
}}

Rules for pond_points:
- Coordinates must be normalized to the displayed crop:
  x = column / img_width
  y = row / img_height
- All coordinates must be between 0 and 1.
- Use one center point per clearly visible pond/open-water feature.
- The point should be at the approximate CENTER of the visible pond/open-water body.
- If a feature is uncertain, do NOT include a point.
- If there are no visible ponds, return [].

Decision rules:
- If there is no clearly visible open water, then "is_there_a_pond" must be "no".
- If shoreline is not visible, "is_there_a_pond" should usually be "no".
- If the interior does not look like open water, "is_there_a_pond" must be "no".
- If the feature could reasonably be shadow, vegetation, mud, wet ground, structure, container, equipment, roof, or tank, "is_there_a_pond" must be "no".
- If a pond-shaped basin or impoundment is visible but clear open water is not, "is_there_a_pond" must be "no".
- If "pond_count" is 0, "pond_points" must be [].
- If a pond appears too small or operationally insignificant for drafting, "likely_draftable" should be "no" or "uncertain".
- Do not set "likely_draftable" to "yes" unless the feature appears plausibly large enough and operationally usable from the image.
- "road_or_access_nearby" refers only to what is visible in the image, such as a road, driveway, turnout, pull-off, or clear vehicle approach immediately adjacent to the pond.
- If a clearly visible road, driveway, turnout, or open vehicle-access area reaches or lies immediately adjacent to a clearly visible pond, set "road_or_access_nearby" to "yes".
- If access is not clearly visible, set "road_or_access_nearby" to "no" or "uncertain".
- A "yes" for road_or_access_nearby does not guarantee draftability by itself, but it is strong positive evidence.
- If there is a clearly visible pond with open water AND road/access immediately adjacent to it AND the pond is not obviously tiny, then prefer "likely_draftable": "yes".
- If there is a clearly visible pond with open water AND road/access immediately adjacent, but the pond may be too small or operationally marginal, set "likely_draftable": "uncertain".
- If there is no visible access near the pond, "likely_draftable" should usually be "no" or "uncertain".
- In rocky or mountainous terrain, darkness plus enclosure is not enough; require a smooth open-water appearance and a distinct shoreline.
- If a feature is small, dark, irregular, and embedded in rocky textured terrain, prefer "is_there_a_pond": "no" unless open water is unmistakable.
- If confidence is not high that the feature is true open water, set "pond_count" to 0 and return no point.
- Be conservative, but do not ignore obvious direct road access when judging likely draftability.

THE NUMBER ONE RULE IS TO NOT HALLUCINATE
""".strip()


def build_system_preamble():
    return (
        "You are a remote sensing analyst specializing in aerial imagery interpretation for wildfire water-source screening. "
        "Use only what is directly visible in the image. "
        "Assume NAIP is about 30 cm GSD. "
        "Be conservative and prefer false negatives over false positives. "
        "Carefully distinguish true open water from shadow, vegetation, wet ground, mud, dark soil, tanks, roofs, and structures. "
        "Evaluate road or vehicle access separately from pond presence. "
        "If a visible road, driveway, turnout, or vehicle-access area lies immediately adjacent to a clearly visible pond, mark road_or_access_nearby as yes. "
        "If a clearly visible pond has direct visible access and is not obviously tiny, likely_draftable may be yes. "
        "Only return pond center points for clearly visible pond/open-water features. "
        "Each point must represent the approximate center of a clearly visible pond. "
        "Coordinates must be normalized from 0 to 1. "
        "If there is uncertainty, do not return a point. "
        "Return only valid JSON with the exact keys requested."
    )


# =============================================================================
# AI OUTPUT VALIDATION
# =============================================================================

REQUIRED_AI_KEYS = [
    "is_there_surface_water",
    "is_there_a_pond",
    "road_or_access_nearby",
    "likely_draftable",
    "pond_count",
    "evidence",
    "notes",
    "pond_points",
]


def normalize_ai_obj(ai_obj, img_w=None, img_h=None):
    if not isinstance(ai_obj, dict):
        return None

    out = dict(ai_obj)
    for k in REQUIRED_AI_KEYS:
        if k not in out:
            if k == "road_or_access_nearby":
                out[k] = "uncertain"
            elif k == "pond_points":
                out[k] = []
            elif k == "pond_count":
                out[k] = 0
            else:
                out[k] = ""

    out["is_there_surface_water"] = safe_lower(out.get("is_there_surface_water"))
    out["is_there_a_pond"] = safe_lower(out.get("is_there_a_pond"))
    out["road_or_access_nearby"] = safe_lower(out.get("road_or_access_nearby"))
    out["likely_draftable"] = safe_lower(out.get("likely_draftable"))

    if out["is_there_surface_water"] not in {"yes", "no"}:
        out["is_there_surface_water"] = "no"
    if out["is_there_a_pond"] not in {"yes", "no"}:
        out["is_there_a_pond"] = "no"
    if out["road_or_access_nearby"] not in {"yes", "no", "uncertain"}:
        out["road_or_access_nearby"] = "uncertain"
    if out["likely_draftable"] not in {"yes", "no", "uncertain"}:
        out["likely_draftable"] = "uncertain"

    try:
        out["pond_count"] = int(out.get("pond_count", 0))
    except Exception:
        out["pond_count"] = 0

    if img_w is not None and img_h is not None:
        out["pond_points"] = sanitize_point_list(out.get("pond_points", []), img_w=img_w, img_h=img_h)
    elif not isinstance(out.get("pond_points", []), list):
        out["pond_points"] = []

    if out["pond_count"] <= 0:
        out["pond_count"] = 0
        out["pond_points"] = []
    elif len(out["pond_points"]) == 0:
        # Conservative: no usable point means no mapped pond center.
        out["pond_count"] = 0
        out["is_there_a_pond"] = "no"
        out["likely_draftable"] = "no"

    out["evidence"] = "" if out.get("evidence") is None else str(out.get("evidence"))[:600]
    out["notes"] = "" if out.get("notes") is None else str(out.get("notes"))[:600]

    return out


def validate_ai_json_file(ai_json_path):
    ai_json_path = Path(ai_json_path)
    issues = []
    if not ai_json_path.exists():
        return False, ["missing_ai_json"]
    try:
        obj = json.loads(ai_json_path.read_text(encoding="utf-8"))
    except Exception as e:
        return False, [f"ai_json_read_error:{type(e).__name__}:{e}"]
    if not isinstance(obj, dict):
        return False, ["ai_json_not_dict"]
    for k in REQUIRED_AI_KEYS:
        if k not in obj:
            issues.append(f"missing_key:{k}")
    if safe_lower(obj.get("is_there_surface_water")) not in {"yes", "no"}:
        issues.append("bad_is_there_surface_water")
    if safe_lower(obj.get("is_there_a_pond")) not in {"yes", "no"}:
        issues.append("bad_is_there_a_pond")
    if safe_lower(obj.get("road_or_access_nearby")) not in {"yes", "no", "uncertain"}:
        issues.append("bad_road_or_access_nearby")
    if safe_lower(obj.get("likely_draftable")) not in {"yes", "no", "uncertain"}:
        issues.append("bad_likely_draftable")
    try:
        pc = int(obj.get("pond_count"))
        if pc < 0:
            issues.append("negative_pond_count")
    except Exception:
        issues.append("bad_pond_count")
    if not isinstance(obj.get("pond_points"), list):
        issues.append("bad_pond_points")
    return len(issues) == 0, issues


def ai_fields_for_index(ai_obj):
    if not isinstance(ai_obj, dict):
        return {
            "ai_is_there_surface_water": "",
            "ai_is_there_a_pond": "",
            "ai_road_or_access_nearby": "",
            "ai_likely_draftable": "",
            "ai_pond_count": "",
        }
    return {
        "ai_is_there_surface_water": "" if ai_obj.get("is_there_surface_water") is None else str(ai_obj.get("is_there_surface_water")),
        "ai_is_there_a_pond": "" if ai_obj.get("is_there_a_pond") is None else str(ai_obj.get("is_there_a_pond")),
        "ai_road_or_access_nearby": "" if ai_obj.get("road_or_access_nearby") is None else str(ai_obj.get("road_or_access_nearby")),
        "ai_likely_draftable": "" if ai_obj.get("likely_draftable") is None else str(ai_obj.get("likely_draftable")),
        "ai_pond_count": "" if ai_obj.get("pond_count") is None else str(ai_obj.get("pond_count")),
    }


def make_error_result(point_id, lon, lat, zoom_m, error, case_id="", case_dir="", attempt_num=""):
    return {
        "ok": False,
        "final_complete": False,
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": case_dir,
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": "",
        "parse_ok": "",
        "validation_ok": False,
        "validation_issues": "",
        "ai_is_there_surface_water": "",
        "ai_is_there_a_pond": "",
        "ai_road_or_access_nearby": "",
        "ai_likely_draftable": "",
        "ai_pond_count": "",
        "api_calls_for_case": "",
        "prompt_tokens": "",
        "completion_tokens": "",
        "total_tokens": "",
        "estimated_cost_usd": "",
        "report_png": "",
        "image_png": "",
        "ai_json": "",
        "ai_raw": "",
        "crop_tif": "",
        "pond_points_shp": "",
        "pond_points_geojson": "",
        "pond_points_csv": "",
        "error": str(error),
    }


INDEX_FIELDNAMES = [
    "ok",
    "final_complete",
    "attempt_num",
    "case_id",
    "case_dir",
    "point_id",
    "lon",
    "lat",
    "zoom_m",
    "naip_id",
    "parse_ok",
    "validation_ok",
    "validation_issues",
    "ai_is_there_surface_water",
    "ai_is_there_a_pond",
    "ai_road_or_access_nearby",
    "ai_likely_draftable",
    "ai_pond_count",
    "api_calls_for_case",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "estimated_cost_usd",
    "report_png",
    "image_png",
    "ai_json",
    "ai_raw",
    "crop_tif",
    "pond_points_shp",
    "pond_points_geojson",
    "pond_points_csv",
    "error",
]


def coerce_result_fields(res):
    out = {k: res.get(k, "") for k in INDEX_FIELDNAMES}
    return out


# =============================================================================
# CASE COMPLETENESS CHECKING
# =============================================================================

def get_expected_case_paths(out_root, case_id):
    case_dir = Path(out_root) / CASE_DIR_NAME / case_id
    return {
        "case_dir": case_dir,
        "crop_png": case_dir / "crop.png",
        "report_png": case_dir / "report.png",
        "ai_json": case_dir / "ai.json",
        "ai_raw": case_dir / "ai_raw.txt",
        "meta_json": case_dir / "meta.json",
        "crop_tif": case_dir / "crop.tif",
        "pond_points_shp": case_dir / "ai_pond_points.shp",
        "pond_points_geojson": case_dir / "ai_pond_points.geojson",
        "pond_points_csv": case_dir / "ai_pond_points.csv",
    }


def case_is_complete(out_root, case_id):
    paths = get_expected_case_paths(out_root, case_id)
    issues = []

    if not paths["case_dir"].exists():
        issues.append("missing_case_dir")
    if not paths["crop_png"].exists():
        issues.append("missing_crop_png")
    if not paths["report_png"].exists():
        issues.append("missing_report_png")
    if not paths["ai_json"].exists():
        issues.append("missing_ai_json")
    if not paths["meta_json"].exists():
        issues.append("missing_meta_json")
    if REQUIRE_CROP_TIF_FOR_COMPLETE and not paths["crop_tif"].exists():
        issues.append("missing_crop_tif")
    if REQUIRE_CASE_SHAPEFILE_FOR_COMPLETE and not paths["pond_points_shp"].exists():
        issues.append("missing_pond_points_shp")
    if not paths["pond_points_geojson"].exists():
        issues.append("missing_pond_points_geojson")
    if not paths["pond_points_csv"].exists():
        issues.append("missing_pond_points_csv")

    ai_ok, ai_issues = validate_ai_json_file(paths["ai_json"])
    if not ai_ok:
        issues.extend(ai_issues)

    return len(issues) == 0, issues


def reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0):
    paths = get_expected_case_paths(out_root, case_id)
    validation_ok, validation_issues = case_is_complete(out_root, case_id)
    ai_obj = None
    try:
        ai_obj = json.loads(paths["ai_json"].read_text(encoding="utf-8"))
    except Exception:
        ai_obj = None
    ai_fields = ai_fields_for_index(ai_obj)

    naip_id = ""
    parse_ok = validation_ok
    try:
        meta = json.loads(paths["meta_json"].read_text(encoding="utf-8"))
        naip_id = meta.get("naip_id", "")
        parse_ok = bool(meta.get("parse_ok", validation_ok))
    except Exception:
        pass

    return coerce_result_fields({
        "ok": bool(validation_ok),
        "final_complete": bool(validation_ok),
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": str(paths["case_dir"]),
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": naip_id,
        "parse_ok": bool(parse_ok),
        "validation_ok": bool(validation_ok),
        "validation_issues": ";".join(validation_issues),
        **ai_fields,
        "api_calls_for_case": 0,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "estimated_cost_usd": 0,
        "report_png": str(paths["report_png"]) if paths["report_png"].exists() else "",
        "image_png": str(paths["crop_png"]) if paths["crop_png"].exists() else "",
        "ai_json": str(paths["ai_json"]) if paths["ai_json"].exists() else "",
        "ai_raw": str(paths["ai_raw"]) if paths["ai_raw"].exists() else "",
        "crop_tif": str(paths["crop_tif"]) if paths["crop_tif"].exists() else "",
        "pond_points_shp": str(paths["pond_points_shp"]) if paths["pond_points_shp"].exists() else "",
        "pond_points_geojson": str(paths["pond_points_geojson"]) if paths["pond_points_geojson"].exists() else "",
        "pond_points_csv": str(paths["pond_points_csv"]) if paths["pond_points_csv"].exists() else "",
        "error": "" if validation_ok else "Incomplete existing case: " + ";".join(validation_issues),
    })


def validate_case_result(res):
    if not parse_boolish(res.get("ok")):
        return False, ["result_ok_false"]
    if not parse_boolish(res.get("parse_ok")):
        return False, ["parse_ok_false"]
    case_id = res.get("case_id", "")
    case_dir = res.get("case_dir", "")
    issues = []
    if not case_id:
        issues.append("missing_case_id")
    if not case_dir or not Path(case_dir).exists():
        issues.append("missing_case_dir")
    for key in ["report_png", "image_png", "ai_json", "pond_points_geojson", "pond_points_csv"]:
        p = res.get(key, "")
        if not p or not Path(p).exists():
            issues.append(f"missing_{key}")
    if REQUIRE_CROP_TIF_FOR_COMPLETE:
        p = res.get("crop_tif", "")
        if not p or not Path(p).exists():
            issues.append("missing_crop_tif")
    ai_ok, ai_issues = validate_ai_json_file(res.get("ai_json", ""))
    if not ai_ok:
        issues.extend(ai_issues)
    return len(issues) == 0, issues


# =============================================================================
# STUDENT / REVIEW TEMPLATE
# =============================================================================

def student_review_template(case_meta):
    return {
        "case_id": case_meta.get("case_id"),
        "point_id": case_meta.get("point_id"),
        "lon": case_meta.get("lon"),
        "lat": case_meta.get("lat"),
        "zoom_m": case_meta.get("zoom_m"),
        "reviewer_name": "",
        "review_date_local": "",
        "labels": {
            "is_there_surface_water": "",
            "is_there_a_pond": "",
            "road_or_access_nearby": "",
            "pond_count_estimate": None,
            "likely_draftable": "",
            "notes": ""
        },
        "agreement_with_ai": {
            "agree_overall": "",
            "disagreement_fields": [],
            "why": ""
        },
        "qc_flags": {
            "image_quality_issue": False,
            "shadow_confusion": False,
            "seasonal_dryness": False,
            "other": ""
        }
    }


# =============================================================================
# CORE CASE RUNNER
# =============================================================================

def naip_qa_case(
    point_id,
    lon,
    lat,
    out_root,
    question,
    zoom_m=DEFAULT_ZOOM_M,
    system_preamble=None,
    temperature=0,
    export_tif=True,
    export_case_shapefile=True,
    start=DEFAULT_START,
    end=DEFAULT_END,
    attempt_num=1,
):
    out_root = Path(out_root)
    cases_root = ensure_dir(out_root / CASE_DIR_NAME)

    case_id = safe_case_id(point_id, lon, lat)
    case_dir = ensure_dir(cases_root / case_id)

    feat, _aoi = find_best_naip_item(lon, lat, start=start, end=end)
    if feat is None:
        return make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(case_dir),
            attempt_num=attempt_num,
            error="No NAIP scene found in window.",
        )

    item_id = feat.get("id")
    href = extract_asset_href(feat)
    if href is None:
        res = make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(case_dir),
            attempt_num=attempt_num,
            error="NAIP item missing usable image href.",
        )
        res["naip_id"] = item_id or ""
        return res

    href_signed = planetary_computer.sign(href)

    with rasterio.open(href_signed) as src:
        pt = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs=4326).to_crs(src.crs)
        cx, cy = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
        crop_geom_srccrs = [mapping(box(cx - zoom_m, cy - zoom_m, cx + zoom_m, cy + zoom_m))]

        data, crop_transform_affine = mask(src, crop_geom_srccrs, crop=True)
        if data.shape[0] >= 3:
            rgb = data[:3, :, :]
        else:
            rgb = np.repeat(data[0:1, :, :], 3, axis=0)

        rgb_u8 = pct_scale_to_u8(rgb)
        pil = Image.fromarray(np.transpose(rgb_u8, (1, 2, 0)))

        src_crs = src.crs
        src_crs_wkt = src.crs.to_wkt() if src.crs else None
        src_res = getattr(src, "res", None)
        src_bounds = tuple(src.bounds) if getattr(src, "bounds", None) else None

    crop_png = case_dir / "crop.png"
    pil.save(str(crop_png), format="PNG")

    answer_text, parse_ok, ai_obj, usage_info = try_parse_json_strict_or_retry(
        pil,
        question,
        system_preamble=system_preamble,
        temperature=temperature,
        model=OPENAI_MODEL,
    )

    img_w, img_h = pil.size
    ai_obj = normalize_ai_obj(ai_obj, img_w=img_w, img_h=img_h) if isinstance(ai_obj, dict) else None
    ai_fields = ai_fields_for_index(ai_obj if parse_ok else None)

    ai_json_path = case_dir / "ai.json"
    ai_raw_path = case_dir / "ai_raw.txt"

    if isinstance(ai_obj, dict):
        write_json(ai_json_path, ai_obj)
        if ai_raw_path.exists():
            ai_raw_path.unlink()
        answer_text_for_report = json.dumps(ai_obj)
    else:
        obj, norm_text = try_parse_json(answer_text)
        obj = normalize_ai_obj(obj, img_w=img_w, img_h=img_h) if isinstance(obj, dict) else None
        if isinstance(obj, dict):
            write_json(ai_json_path, obj)
            if ai_raw_path.exists():
                ai_raw_path.unlink()
            answer_text_for_report = json.dumps(obj)
            ai_obj = obj
            parse_ok = True
            ai_fields = ai_fields_for_index(ai_obj)
        else:
            write_text(ai_raw_path, norm_text)
            if ai_json_path.exists():
                ai_json_path.unlink()
            answer_text_for_report = norm_text

    report_png = case_dir / "report.png"
    title_text = f"NAIP (~{ASSUMED_GSD_M:.2f} m) @ {lat:.6f}, {lon:.6f}\n{item_id} | scene ~{int(zoom_m * 2)} m x {int(zoom_m * 2)} m"
    write_report_png(pil, title_text, answer_text_for_report, report_png, dpi=200)

    crop_tif_path = ""
    if export_tif:
        crop_tif = case_dir / "crop.tif"
        crop_tif_path = export_crop_tif(href_signed=href_signed, crop_geom=crop_geom_srccrs, out_tif_path=crop_tif)

    pond_points_shp = ""
    pond_points_geojson = ""
    pond_points_csv = ""

    if isinstance(ai_obj, dict):
        pond_points = sanitize_point_list(ai_obj.get("pond_points", []), img_w=img_w, img_h=img_h)
        crop_transform = {
            "transform": crop_transform_affine,
            "img_w": img_w,
            "img_h": img_h,
        }

        case_points_gdf = normalized_points_to_geodataframe(
            point_list=pond_points,
            crop_transform=crop_transform,
            crop_crs=src_crs,
            point_id=point_id,
            case_id=case_id,
            lon=lon,
            lat=lat,
            naip_id=item_id,
        )

        if case_points_gdf is not None and len(case_points_gdf) > 0:
            case_points_gdf_wgs84 = case_points_gdf.to_crs(4326)
        else:
            case_points_gdf_wgs84 = empty_pond_points_gdf(crs="EPSG:4326")

        if export_case_shapefile:
            pond_points_shp = str(case_dir / "ai_pond_points.shp")
            write_vector_safe(case_points_gdf_wgs84, pond_points_shp)

        pond_points_geojson = str(case_dir / "ai_pond_points.geojson")
        write_vector_safe(case_points_gdf_wgs84, pond_points_geojson, driver="GeoJSON")

        pond_points_csv = str(case_dir / "ai_pond_points.csv")
        if len(case_points_gdf_wgs84) > 0:
            tmp = case_points_gdf_wgs84.copy()
            tmp["lon"] = tmp.geometry.x
            tmp["lat"] = tmp.geometry.y
            tmp.drop(columns=["geometry"]).to_csv(pond_points_csv, index=False)
        else:
            pd.DataFrame(columns=[
                "case_id", "point_id", "src_lon", "src_lat", "naip_id",
                "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat"
            ]).to_csv(pond_points_csv, index=False)

    props = feat.get("properties", {}) if isinstance(feat, dict) else {}
    case_meta = {
        "case_id": case_id,
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "scene_width_m": zoom_m * 2,
        "datetime_local": datetime.now().isoformat(timespec="seconds"),
        "assumed_gsd_m_prompt": ASSUMED_GSD_M,
        "naip_id": item_id,
        "naip_href_signed_runtime": href_signed,
        "stac_properties_subset": {
            "datetime": props.get("datetime"),
            "naip:year": props.get("naip:year"),
            "gsd": props.get("gsd"),
            "proj:epsg": props.get("proj:epsg"),
        },
        "src_crs_wkt": src_crs_wkt,
        "src_res": src_res,
        "src_bounds": src_bounds,
        "packages": get_pkg_versions(),
        "parse_ok": bool(parse_ok),
        "openai_model": OPENAI_MODEL,
        "temperature": temperature,
        "start": start,
        "end": end,
        "ai_index_fields": ai_fields,
        "openai_usage": usage_info,
        "run_cost_tracker_snapshot": RUN_COST_TRACKER.copy(),
        "exports": {
            "case_point_shapefile": pond_points_shp,
            "case_point_geojson": pond_points_geojson,
            "case_point_csv": pond_points_csv,
        }
    }
    write_json(case_dir / "meta.json", case_meta)
    write_json(case_dir / "student_review.json", student_review_template(case_meta))
    write_text(
        case_dir / "README.txt",
        "Open report.png first.\n"
        "Red points indicate AI-detected pond/open-water centers.\n"
        "Open ai_pond_points.shp or ai_pond_points.geojson for GIS use.\n"
        "Then fill student_review.json if this case is selected for review.\n"
        "Do NOT treat this as a confirmed drafting source without local/field verification.\n",
    )

    res = {
        "ok": True,
        "final_complete": False,
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": str(case_dir),
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": item_id or "",
        "parse_ok": bool(parse_ok),
        "validation_ok": False,
        "validation_issues": "",
        **ai_fields,
        "api_calls_for_case": usage_info["num_api_calls_for_case"],
        "prompt_tokens": usage_info["prompt_tokens"],
        "completion_tokens": usage_info["completion_tokens"],
        "total_tokens": usage_info["total_tokens"],
        "estimated_cost_usd": round(usage_info["estimated_cost_usd"], 6),
        "report_png": str(report_png),
        "image_png": str(crop_png),
        "ai_json": str(ai_json_path) if ai_json_path.exists() else "",
        "ai_raw": str(ai_raw_path) if ai_raw_path.exists() else "",
        "crop_tif": crop_tif_path or "",
        "pond_points_shp": pond_points_shp,
        "pond_points_geojson": pond_points_geojson,
        "pond_points_csv": pond_points_csv,
        "error": "",
    }

    validation_ok, validation_issues = validate_case_result(res)
    res["validation_ok"] = bool(validation_ok)
    res["validation_issues"] = ";".join(validation_issues)
    res["final_complete"] = bool(validation_ok)
    if not validation_ok:
        res["ok"] = False
        res["error"] = "Case failed validation: " + ";".join(validation_issues)

    return coerce_result_fields(res)


# =============================================================================
# INPUT CSV READING
# =============================================================================

def read_points_csv(points_csv, zoom_m_default=DEFAULT_ZOOM_M, start=DEFAULT_START, end=DEFAULT_END):
    points_csv = Path(points_csv)
    if not points_csv.exists():
        raise FileNotFoundError(f"Input points CSV does not exist: {points_csv}")

    with points_csv.open("r", encoding="utf-8-sig", newline="") as f_in:
        sample = f_in.read(8192)
        f_in.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";", "|"])
        except Exception:
            dialect = csv.get_dialect("excel")
            dialect.delimiter = "\t" if "\t" in sample and "," not in sample else ","
        reader = csv.DictReader(f_in, dialect=dialect)
        if not reader.fieldnames:
            raise RuntimeError("Could not read header row from points file.")

        raw_fields = list(reader.fieldnames)
        norm_fields = [((h or "").strip().lower()) for h in raw_fields]
        header_map = {raw: norm for raw, norm in zip(raw_fields, norm_fields)}

        def get_value(row, *keys):
            keys = set(keys)
            for k in keys:
                if k in row and row[k] not in (None, ""):
                    return row[k]
            for raw_k, norm_k in header_map.items():
                if norm_k in keys:
                    v = row.get(raw_k, "")
                    if v not in (None, ""):
                        return v
            return ""

        rows = []
        for i, row in enumerate(reader, start=1):
            point_id = get_value(row, "id", "point_id", "name", "fid") or str(i)
            lon_s = get_value(row, "lon", "longitude", "x")
            lat_s = get_value(row, "lat", "latitude", "y")
            if lon_s == "" or lat_s == "":
                rows.append({
                    "input_order": i,
                    "input_valid": False,
                    "point_id": point_id,
                    "lon": np.nan,
                    "lat": np.nan,
                    "zoom_m": zoom_m_default,
                    "start": start,
                    "end": end,
                    "input_error": f"Missing lon/lat in row {i}: {row}",
                })
                continue
            try:
                lon = float(str(lon_s).strip())
                lat = float(str(lat_s).strip())
            except Exception as e:
                rows.append({
                    "input_order": i,
                    "input_valid": False,
                    "point_id": point_id,
                    "lon": np.nan,
                    "lat": np.nan,
                    "zoom_m": zoom_m_default,
                    "start": start,
                    "end": end,
                    "input_error": f"Bad lon/lat in row {i}: {type(e).__name__}: {e}",
                })
                continue

            zoom_s = get_value(row, "zoom_m", "zoom", "buffer_m")
            try:
                zoom_m = float(str(zoom_s).strip()) if zoom_s not in ("", None) else float(zoom_m_default)
            except Exception:
                zoom_m = float(zoom_m_default)

            row_start = (get_value(row, "start") or start).strip()
            row_end = (get_value(row, "end") or end).strip()

            rows.append({
                "input_order": i,
                "input_valid": True,
                "point_id": point_id,
                "lon": lon,
                "lat": lat,
                "zoom_m": zoom_m,
                "start": row_start,
                "end": row_end,
                "input_error": "",
            })

    df = pd.DataFrame(rows)
    df["case_id"] = df.apply(
        lambda r: safe_case_id(r["point_id"], r["lon"], r["lat"]) if bool(r["input_valid"]) else "",
        axis=1,
    )
    return df


# =============================================================================
# BATCH SHAPEFILE / MERGED OUTPUT HELPERS
# =============================================================================

def build_batch_pond_points_layers(index_csv_path, out_root):
    out_root = Path(out_root)
    index_csv_path = Path(index_csv_path)
    merged = []

    if not index_csv_path.exists():
        return ""

    idx = pd.read_csv(index_csv_path, dtype=str).fillna("")
    for _, row in idx.iterrows():
        csv_path = str(row.get("pond_points_csv", "")).strip()
        if csv_path and Path(csv_path).exists():
            try:
                pts = pd.read_csv(csv_path)
                if len(pts) > 0 and {"lon", "lat"}.issubset(pts.columns):
                    for f in [
                        "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
                        "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json"
                    ]:
                        pts[f] = row.get(f, "")
                    merged.append(pts)
            except Exception as e:
                print(f"Warning: could not read pond point CSV {csv_path}: {e}")

    batch_shp = out_root / "batch_ai_pond_points.shp"
    batch_geojson = out_root / "batch_ai_pond_points.geojson"
    batch_gpkg = out_root / "batch_ai_pond_points.gpkg"
    batch_csv = out_root / "batch_ai_pond_points.csv"

    if len(merged) == 0:
        out_csv = pd.DataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id", "pond_label", "confidence",
            "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat",
            "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
            "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json"
        ])
        out_csv.to_csv(batch_csv, index=False)
        empty = empty_pond_points_gdf(crs="EPSG:4326")
        write_vector_safe(empty, batch_shp)
        write_vector_safe(empty, batch_geojson, driver="GeoJSON")
        write_vector_safe(empty, batch_gpkg)
        return str(batch_shp)

    all_pts = pd.concat(merged, ignore_index=True)
    all_pts.to_csv(batch_csv, index=False)
    gdf = gpd.GeoDataFrame(
        all_pts,
        geometry=gpd.points_from_xy(all_pts["lon"].astype(float), all_pts["lat"].astype(float)),
        crs="EPSG:4326",
    )
    write_vector_safe(gdf, batch_shp)
    write_vector_safe(gdf, batch_geojson, driver="GeoJSON")
    write_vector_safe(gdf, batch_gpkg)
    return str(batch_shp)


def write_index_csv(index_path, rows):
    index_path = Path(index_path)
    ensure_dir(index_path.parent)
    with index_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=INDEX_FIELDNAMES)
        writer.writeheader()
        for row in rows:
            writer.writerow(coerce_result_fields(row))
    return str(index_path)


def append_attempt_csv(attempts_path, row):
    attempts_path = Path(attempts_path)
    ensure_dir(attempts_path.parent)
    with ATTEMPTS_CSV_LOCK:
        write_header = not attempts_path.exists()
        with attempts_path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=INDEX_FIELDNAMES)
            if write_header:
                writer.writeheader()
            writer.writerow(coerce_result_fields(row))


# =============================================================================
# RESULTS PACKAGING
# =============================================================================

def safe_read_index(index_csv):
    if not Path(index_csv).exists():
        return pd.DataFrame(columns=INDEX_FIELDNAMES)
    return pd.read_csv(index_csv, dtype=str).fillna("")


def make_all_grid_gdf(idx):
    if len(idx) == 0:
        return gpd.GeoDataFrame(columns=list(idx.columns), geometry=[], crs="EPSG:4326")
    df = idx.copy()
    df["lon_num"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat_num"] = pd.to_numeric(df["lat"], errors="coerce")
    df = df.dropna(subset=["lon_num", "lat_num"])
    if len(df) == 0:
        return gpd.GeoDataFrame(columns=list(idx.columns), geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(
        df.drop(columns=["lon_num", "lat_num"]),
        geometry=gpd.points_from_xy(df["lon_num"], df["lat_num"]),
        crs="EPSG:4326",
    )


def read_candidate_points_from_index(idx):
    merged = []
    for _, row in idx.iterrows():
        p = str(row.get("pond_points_csv", "")).strip()
        if not p or not Path(p).exists():
            continue
        try:
            pts = pd.read_csv(p)
            if len(pts) == 0:
                continue
            for f in [
                "ok", "final_complete", "validation_ok", "validation_issues",
                "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
                "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json", "case_dir"
            ]:
                pts[f] = row.get(f, "")
            merged.append(pts)
        except Exception as e:
            print(f"Warning: failed reading candidate points {p}: {e}")

    if len(merged) == 0:
        return pd.DataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id", "pond_label", "confidence",
            "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat",
            "ai_likely_draftable", "ai_road_or_access_nearby", "report_png", "image_png", "ai_json"
        ])
    return pd.concat(merged, ignore_index=True)


def build_results_summary(idx, cand_df, attempts_df=None):
    total_input = len(idx)
    ok = idx["ok"].map(parse_boolish).sum() if "ok" in idx else 0
    complete = idx["final_complete"].map(parse_boolish).sum() if "final_complete" in idx else 0
    failures = total_input - complete

    def count_value(field, val):
        if field not in idx.columns:
            return 0
        return int((idx[field].map(safe_lower) == val).sum())

    summary = {
        "created_local": datetime.now().isoformat(timespec="seconds"),
        "model": OPENAI_MODEL,
        "assumed_gsd_m": ASSUMED_GSD_M,
        "total_input_points": int(total_input),
        "successful_complete_interpretations": int(complete),
        "ok_rows": int(ok),
        "failed_or_incomplete_rows": int(failures),
        "surface_water_yes": count_value("ai_is_there_surface_water", "yes"),
        "pond_yes": count_value("ai_is_there_a_pond", "yes"),
        "road_or_access_yes": count_value("ai_road_or_access_nearby", "yes"),
        "road_or_access_uncertain": count_value("ai_road_or_access_nearby", "uncertain"),
        "likely_draftable_yes": count_value("ai_likely_draftable", "yes"),
        "likely_draftable_uncertain": count_value("ai_likely_draftable", "uncertain"),
        "candidate_pond_center_points": int(len(cand_df)),
        "run_cost_tracker": RUN_COST_TRACKER.copy(),
    }
    if attempts_df is not None and len(attempts_df) > 0:
        summary["total_attempt_rows"] = int(len(attempts_df))
        summary["attempt_rows_ok"] = int(attempts_df["ok"].map(parse_boolish).sum()) if "ok" in attempts_df else 0
    return summary


def write_simple_bar(labels, values, title, ylabel, out_png):
    out_png = Path(out_png)
    ensure_dir(out_png.parent)
    fig, ax = plt.subplots(figsize=(8, 5), dpi=200)
    ax.bar(labels, values)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=25)
    for i, v in enumerate(values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def write_map_figure(all_gdf, cand_gdf, out_png):
    out_png = Path(out_png)
    ensure_dir(out_png.parent)
    fig, ax = plt.subplots(figsize=(8, 8), dpi=220)
    if all_gdf is not None and len(all_gdf) > 0:
        all_gdf.plot(ax=ax, markersize=5, color="lightgray", edgecolor="none", label="Interpreted image chips")
    if cand_gdf is not None and len(cand_gdf) > 0:
        cand_gdf.plot(ax=ax, markersize=18, color="red", edgecolor="black", linewidth=0.3, label="AI candidate pond centers")
    ax.set_title("AI-screened candidate pond centers")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def copy_review_reports(idx, cand_df, review_dir):
    review_dir = ensure_dir(review_dir)
    copied = []

    # Copy candidate reports first.
    if len(cand_df) > 0:
        tmp = cand_df.copy()
        if "confidence" in tmp.columns:
            tmp["confidence_num"] = pd.to_numeric(tmp["confidence"], errors="coerce").fillna(-1)
            tmp = tmp.sort_values("confidence_num", ascending=False)
        seen = set()
        for _, row in tmp.iterrows():
            case_id = str(row.get("case_id", ""))
            if not case_id or case_id in seen:
                continue
            seen.add(case_id)
            src = str(row.get("report_png", ""))
            if src and Path(src).exists():
                dst = review_dir / "candidate_reports" / f"candidate_{len(copied)+1:04d}_{clean_filename(case_id, 80)}.png"
                ensure_dir(dst.parent)
                shutil.copy2(src, dst)
                copied.append(str(dst))
                if len(copied) >= MAX_REVIEW_REPORTS_TO_COPY:
                    break

    # Copy a small set of negative reports for comparison.
    neg_dir = review_dir / "negative_examples"
    neg_count = 0
    if len(idx) > 0:
        neg = idx[idx.get("ai_is_there_a_pond", pd.Series([""] * len(idx))).map(safe_lower) == "no"].head(MAX_EXAMPLE_REPORTS_PER_GROUP)
        for _, row in neg.iterrows():
            src = str(row.get("report_png", ""))
            case_id = str(row.get("case_id", ""))
            if src and Path(src).exists():
                dst = neg_dir / f"negative_{neg_count+1:04d}_{clean_filename(case_id, 80)}.png"
                ensure_dir(dst.parent)
                shutil.copy2(src, dst)
                neg_count += 1

    return copied


def build_results_folder(out_root, points_csv, prompt_text, system_preamble):
    out_root = Path(out_root)
    results_root = ensure_dir(out_root / RESULTS_DIR_NAME)
    tables_dir = ensure_dir(results_root / "tables")
    gis_dir = ensure_dir(results_root / "gis")
    figs_dir = ensure_dir(results_root / "figures")
    review_dir = ensure_dir(results_root / "review_pack")
    qa_dir = ensure_dir(results_root / "qa")
    docs_dir = ensure_dir(results_root / "documents")

    index_csv = out_root / "index.csv"
    attempts_csv = out_root / "index_attempts.csv"
    idx = safe_read_index(index_csv)
    attempts_df = safe_read_index(attempts_csv) if attempts_csv.exists() else pd.DataFrame()
    cand_df = read_candidate_points_from_index(idx)

    # Tables.
    idx.to_csv(tables_dir / "final_index_one_row_per_input_point.csv", index=False)
    cand_df.to_csv(tables_dir / "candidate_pond_centers.csv", index=False)
    if len(attempts_df) > 0:
        attempts_df.to_csv(qa_dir / "all_attempts_including_retries.csv", index=False)

    failures = idx[~idx["final_complete"].map(parse_boolish)].copy() if len(idx) > 0 else idx.copy()
    failures.to_csv(qa_dir / "FAILURES_NEED_REVIEW.csv", index=False)
    failures.to_csv(tables_dir / "failures_need_review.csv", index=False)

    # GIS outputs.
    all_gdf = make_all_grid_gdf(idx)
    if len(all_gdf) > 0:
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.geojson", driver="GeoJSON")
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.gpkg")
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.shp")

    if len(cand_df) > 0:
        cand_gdf = gpd.GeoDataFrame(
            cand_df.copy(),
            geometry=gpd.points_from_xy(cand_df["lon"].astype(float), cand_df["lat"].astype(float)),
            crs="EPSG:4326",
        )
    else:
        cand_gdf = empty_pond_points_gdf(crs="EPSG:4326")

    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.geojson", driver="GeoJSON")
    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.gpkg")
    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.shp")

    # Keep compatibility batch outputs at root too.
    try:
        build_batch_pond_points_layers(index_csv, out_root)
    except Exception as e:
        print(f"Warning: failed to write root batch layers: {e}")

    # Figures.
    summary = build_results_summary(idx, cand_df, attempts_df=attempts_df)
    summary_rows = pd.DataFrame([summary])
    summary_rows.to_csv(tables_dir / "run_summary.csv", index=False)
    write_json(tables_dir / "run_summary.json", summary)

    write_simple_bar(
        ["Input chips", "Successful", "Failed/incomplete", "Candidate centers"],
        [
            summary["total_input_points"],
            summary["successful_complete_interpretations"],
            summary["failed_or_incomplete_rows"],
            summary["candidate_pond_center_points"],
        ],
        "Run summary",
        "Count",
        figs_dir / "run_summary_counts.png",
    )

    write_simple_bar(
        ["Surface water yes", "Pond yes", "Access yes", "Likely draftable yes"],
        [
            summary["surface_water_yes"],
            summary["pond_yes"],
            summary["road_or_access_yes"],
            summary["likely_draftable_yes"],
        ],
        "AI interpretation counts",
        "Image-chip count",
        figs_dir / "ai_interpretation_counts.png",
    )

    write_map_figure(all_gdf, cand_gdf, figs_dir / "candidate_pond_centers_map.png")

    # Review-pack image copies.
    copied_reports = copy_review_reports(idx, cand_df, review_dir)

    # Prompt / config / context docs.
    write_text(docs_dir / "prompt_used.txt", prompt_text)
    write_text(docs_dir / "system_preamble_used.txt", system_preamble)
    write_json(docs_dir / "run_config.json", {
        "points_csv": str(points_csv),
        "out_root": str(out_root),
        "openai_model": OPENAI_MODEL,
        "assumed_gsd_m": ASSUMED_GSD_M,
        "default_zoom_m": DEFAULT_ZOOM_M,
        "default_scene_width_m": DEFAULT_ZOOM_M * 2,
        "default_start": DEFAULT_START,
        "default_end": DEFAULT_END,
        "force_rerun_all": FORCE_RERUN_ALL,
        "rerun_failed_or_invalid": RERUN_FAILED_OR_INVALID,
        "max_attempts_per_case": MAX_ATTEMPTS_PER_CASE,
        "require_crop_tif_for_complete": REQUIRE_CROP_TIF_FOR_COMPLETE,
        "packages": get_pkg_versions(),
    })

    context_md = f"""# Results context for AI writing

This folder contains a production-style output package for the NAIP AI pond/drafting-location screening workflow.

## Main interpretation summary

- Input image chips / grid points: {summary['total_input_points']}
- Successful complete interpretations: {summary['successful_complete_interpretations']}
- Failed or incomplete interpretations after retries: {summary['failed_or_incomplete_rows']}
- Image chips where AI reported surface water = yes: {summary['surface_water_yes']}
- Image chips where AI reported pond = yes: {summary['pond_yes']}
- Image chips where AI reported visible nearby road/access = yes: {summary['road_or_access_yes']}
- Image chips where AI reported likely_draftable = yes: {summary['likely_draftable_yes']}
- Candidate pond center points exported to GIS: {summary['candidate_pond_center_points']}

## How to use this folder to write a Results section

Use these files first:

1. `tables/run_summary.csv` — one-row summary of the run.
2. `tables/final_index_one_row_per_input_point.csv` — one row per input image chip/grid point.
3. `tables/candidate_pond_centers.csv` — one row per mapped AI pond-center point.
4. `gis/candidate_pond_centers.gpkg` or `.geojson` — GIS-ready candidate pond-center layer.
5. `gis/all_interpreted_grid_points.gpkg` or `.geojson` — GIS-ready layer of all interpreted chip centers.
6. `figures/candidate_pond_centers_map.png` — simple map figure for the Results section.
7. `figures/run_summary_counts.png` and `figures/ai_interpretation_counts.png` — simple count figures.
8. `qa/FAILURES_NEED_REVIEW.csv` — any chips that still failed or were incomplete after retries.
9. `review_pack/candidate_reports/` — report images for candidate detections.

## Suggested Results framing

The primary result is a GIS-ready set of candidate pond-based drafting locations, not a confirmed operational drafting inventory. The AI screened fixed-area NAIP image chips for visible open water and nearby vehicle-access context. Candidate points represent approximate pond centers derived from the AI-reported normalized image coordinates. Locations should be described as candidate water-source locations requiring firefighter review, landowner coordination, and field verification.

## Placeholder language

Use `XXXX` for any values that require manual expert review, field confirmation, landowner status, seasonal water persistence, measured depth, or operational safety assessment.
"""
    write_text(results_root / "RESULTS_CONTEXT_FOR_AI.md", context_md)

    readme = f"""# RESULTS folder

This folder was automatically generated by the final NAIP AI pond-screening workflow.

## Folder contents

- `tables/`: CSV tables for writing the Results section.
- `gis/`: GIS-ready outputs for candidate pond centers and all interpreted grid points.
- `figures/`: simple figures for a manuscript/report Results section.
- `review_pack/`: copied report images for manual review.
- `qa/`: retry logs, failed cases, and validation outputs.
- `documents/`: prompt, system preamble, run configuration, and metadata.

## Most important outputs

- Candidate pond centers: `gis/candidate_pond_centers.gpkg`
- All interpreted chip centers: `gis/all_interpreted_grid_points.gpkg`
- Final chip-level table: `tables/final_index_one_row_per_input_point.csv`
- Candidate point table: `tables/candidate_pond_centers.csv`
- Summary table: `tables/run_summary.csv`
- AI writing context: `RESULTS_CONTEXT_FOR_AI.md`

## Run completion

- Input image chips / grid points: {summary['total_input_points']}
- Successful complete interpretations: {summary['successful_complete_interpretations']}
- Failed or incomplete after retries: {summary['failed_or_incomplete_rows']}
- Candidate pond center points: {summary['candidate_pond_center_points']}

If `qa/FAILURES_NEED_REVIEW.csv` has rows, those rows did not fully complete after retries. They were not silently skipped.
"""
    write_text(results_root / "README_RESULTS.md", readme)

    validation_report = {
        "summary": summary,
        "failures_csv": str(qa_dir / "FAILURES_NEED_REVIEW.csv"),
        "candidate_reports_copied": len(copied_reports),
        "results_root": str(results_root),
    }
    write_json(qa_dir / "validation_report.json", validation_report)

    print("Wrote RESULTS folder:", results_root)
    print(json.dumps(summary, indent=2))
    return str(results_root)


# =============================================================================
# BATCH RUNNER
# =============================================================================

def run_one_case_with_retries(row, out_root, question, system_preamble, temperature, export_tif, export_case_shapefile, attempts_path, log_path):
    point_id = row["point_id"]
    lon = float(row["lon"])
    lat = float(row["lat"])
    zoom_m = float(row["zoom_m"])
    start = row["start"]
    end = row["end"]
    case_id = row["case_id"]

    if not FORCE_RERUN_ALL:
        complete, issues = case_is_complete(out_root, case_id)
        if complete:
            print(f"Skipping complete case: {case_id}")
            return reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0)
        elif not RERUN_FAILED_OR_INVALID:
            print(f"Existing incomplete case will not be rerun because RERUN_FAILED_OR_INVALID=False: {case_id}")
            return reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0)
        else:
            print(f"Rerunning incomplete/failed case: {case_id} | issues={';'.join(issues)}")

    last_res = None
    for attempt_num in range(1, MAX_ATTEMPTS_PER_CASE + 1):
        print(f"Running case {case_id} | attempt {attempt_num}/{MAX_ATTEMPTS_PER_CASE}")
        try:
            res = naip_qa_case(
                point_id=point_id,
                lon=lon,
                lat=lat,
                out_root=out_root,
                question=question,
                zoom_m=zoom_m,
                system_preamble=system_preamble,
                temperature=temperature,
                export_tif=export_tif,
                export_case_shapefile=export_case_shapefile,
                start=start,
                end=end,
                attempt_num=attempt_num,
            )
        except Exception as e:
            tb = traceback.format_exc()
            err = f"{type(e).__name__}: {e}"
            with LOG_FILE_LOCK:
                with Path(log_path).open("a", encoding="utf-8") as f_log:
                    f_log.write(f"\n[{datetime.now().isoformat(timespec='seconds')}] case_id={case_id} attempt={attempt_num}\n{err}\n{tb}\n")
            res = make_error_result(
                point_id=point_id,
                lon=lon,
                lat=lat,
                zoom_m=zoom_m,
                case_id=case_id,
                case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
                attempt_num=attempt_num,
                error=err,
            )

        append_attempt_csv(attempts_path, res)
        last_res = res

        ok, issues = validate_case_result(res)
        res["validation_ok"] = bool(ok)
        res["validation_issues"] = ";".join(issues)
        res["final_complete"] = bool(ok)
        if ok:
            res["ok"] = True
            res["error"] = ""
            return coerce_result_fields(res)

        print(f"Case did not validate: {case_id} | attempt {attempt_num} | issues={';'.join(issues)}")
        if attempt_num < MAX_ATTEMPTS_PER_CASE:
            time.sleep(SLEEP_BETWEEN_ATTEMPTS_SEC)

    if last_res is None:
        last_res = make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
            attempt_num=MAX_ATTEMPTS_PER_CASE,
            error="No attempts were completed.",
        )

    final_ok, final_issues = validate_case_result(last_res)
    last_res["validation_ok"] = bool(final_ok)
    last_res["validation_issues"] = ";".join(final_issues)
    last_res["final_complete"] = bool(final_ok)
    last_res["ok"] = bool(final_ok)
    if not final_ok and not last_res.get("error"):
        last_res["error"] = "Case failed final validation: " + ";".join(final_issues)
    return coerce_result_fields(last_res)


def run_batch_points_csv(
    points_csv,
    out_root,
    question,
    zoom_m_default=DEFAULT_ZOOM_M,
    system_preamble=None,
    temperature=0,
    export_tif=True,
    export_case_shapefile=True,
    export_batch_shapefile=True,
    start=DEFAULT_START,
    end=DEFAULT_END,
    max_parallel_workers=MAX_PARALLEL_WORKERS,
):
    out_root = ensure_dir(out_root)
    ensure_dir(out_root / CASE_DIR_NAME)

    index_path = out_root / "index.csv"
    attempts_path = out_root / "index_attempts.csv"
    log_path = out_root / "batch_log.txt"

    points_df = read_points_csv(points_csv, zoom_m_default=zoom_m_default, start=start, end=end)
    points_df.to_csv(out_root / "input_points_parsed.csv", index=False)

    # Store final rows by input_order so index.csv remains in original input order,
    # even when cases finish out of order in parallel.
    final_rows_by_order = {}

    invalid_inputs = points_df[~points_df["input_valid"]]
    for _, bad in invalid_inputs.iterrows():
        input_order = int(bad.get("input_order", len(final_rows_by_order) + 1))
        res = make_error_result(
            point_id=bad.get("point_id", ""),
            lon="",
            lat="",
            zoom_m=bad.get("zoom_m", ""),
            case_id="",
            case_dir="",
            attempt_num=0,
            error=bad.get("input_error", "Invalid input row."),
        )
        append_attempt_csv(attempts_path, res)
        final_rows_by_order[input_order] = res

    valid_points = points_df[points_df["input_valid"]].copy()
    n = len(valid_points)

    workers = int(max_parallel_workers or 1)
    workers = max(1, workers)

    def ordered_final_rows():
        return [final_rows_by_order[k] for k in sorted(final_rows_by_order.keys())]

    print(f"\nStarting batch run with MAX_PARALLEL_WORKERS={workers}")
    print(f"Valid input points: {n} | Invalid input rows: {len(invalid_inputs)}")

    if workers == 1:
        # Sequential mode. This is useful for debugging.
        for i, (_, row) in enumerate(valid_points.iterrows(), start=1):
            print(f"\n========== {i}/{n} | point_id={row['point_id']} | case_id={row['case_id']} ==========")
            res = run_one_case_with_retries(
                row=row,
                out_root=out_root,
                question=question,
                system_preamble=system_preamble,
                temperature=temperature,
                export_tif=export_tif,
                export_case_shapefile=export_case_shapefile,
                attempts_path=attempts_path,
                log_path=log_path,
            )
            final_rows_by_order[int(row["input_order"])] = res
            write_index_csv(index_path, ordered_final_rows())
            print_cost_summary(prefix="RUNNING TOTAL")
    else:
        # Parallel mode. Each worker processes one full case, including its retries.
        futures = {}
        with ThreadPoolExecutor(max_workers=workers) as executor:
            for i, (_, row) in enumerate(valid_points.iterrows(), start=1):
                print(f"Submitting {i}/{n} | point_id={row['point_id']} | case_id={row['case_id']}")
                fut = executor.submit(
                    run_one_case_with_retries,
                    row=row,
                    out_root=out_root,
                    question=question,
                    system_preamble=system_preamble,
                    temperature=temperature,
                    export_tif=export_tif,
                    export_case_shapefile=export_case_shapefile,
                    attempts_path=attempts_path,
                    log_path=log_path,
                )
                futures[fut] = (i, int(row["input_order"]), row["point_id"], row["case_id"])

            completed = 0
            for fut in as_completed(futures):
                i, input_order, point_id, case_id = futures[fut]
                completed += 1
                try:
                    res = fut.result()
                except Exception as e:
                    tb = traceback.format_exc()
                    err = f"{type(e).__name__}: {e}"
                    with LOG_FILE_LOCK:
                        with Path(log_path).open("a", encoding="utf-8") as f_log:
                            f_log.write(f"\n[{datetime.now().isoformat(timespec='seconds')}] case_id={case_id} future_failed\n{err}\n{tb}\n")
                    # Pull lon/lat/zoom from the input row for a useful final failure row.
                    row_match = valid_points[valid_points["input_order"].astype(int) == int(input_order)].iloc[0]
                    res = make_error_result(
                        point_id=point_id,
                        lon=row_match.get("lon", ""),
                        lat=row_match.get("lat", ""),
                        zoom_m=row_match.get("zoom_m", ""),
                        case_id=case_id,
                        case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
                        attempt_num=MAX_ATTEMPTS_PER_CASE,
                        error=err,
                    )
                    append_attempt_csv(attempts_path, res)

                final_rows_by_order[input_order] = res
                write_index_csv(index_path, ordered_final_rows())

                ok_txt = "OK" if parse_boolish(res.get("final_complete")) else "FAILED"
                print(f"\nCompleted {completed}/{n} | original_submit={i}/{n} | {ok_txt} | point_id={point_id} | case_id={case_id}")
                print_cost_summary(prefix="RUNNING TOTAL")

    write_index_csv(index_path, ordered_final_rows())

    if export_batch_shapefile:
        try:
            batch_shp = build_batch_pond_points_layers(index_path, out_root)
            print("Wrote batch pond shapefile:", batch_shp)
        except Exception as e:
            print(f"Warning: failed to build batch shapefile: {e}")

    write_text(
        out_root / "README_STUDENTS.txt",
        "Student workflow:\n"
        "1) Open RESULTS/README_RESULTS.md first.\n"
        "2) Open RESULTS/tables/final_index_one_row_per_input_point.csv for chip-level results.\n"
        "3) Open RESULTS/gis/candidate_pond_centers.gpkg for mapped candidate pond centers.\n"
        "4) Open report.png files in CASES folders or RESULTS/review_pack/candidate_reports for visual review.\n"
        "5) Treat all mapped points as candidate drafting locations only. Field/local verification is required.\n",
    )

    build_results_folder(
        out_root=out_root,
        points_csv=points_csv,
        prompt_text=question,
        system_preamble=system_preamble or "",
    )

    return str(index_path)


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    MODE = "batch"

    out_root = DEFAULT_OUT_ROOT
    points_csv = DEFAULT_POINTS_CSV
    temperature = 0

    scene_width_m = int(DEFAULT_ZOOM_M * 2)
    question = build_pond_prompt(scene_width_m=scene_width_m)
    system_preamble = build_system_preamble()

    if MODE == "single":
        point_id = "pt001test23"
        lon, lat = -105.3227622, 40.55704649

        res = naip_qa_case(
            point_id=point_id,
            lon=lon,
            lat=lat,
            out_root=out_root,
            question=question,
            zoom_m=DEFAULT_ZOOM_M,
            system_preamble=system_preamble,
            temperature=temperature,
            export_tif=True,
            export_case_shapefile=True,
            start=DEFAULT_START,
            end=DEFAULT_END,
            attempt_num=1,
        )
        print(json.dumps(res, indent=2))
        build_results_folder(
            out_root=out_root,
            points_csv=points_csv,
            prompt_text=question,
            system_preamble=system_preamble,
        )
        print_cost_summary(prefix="SESSION TOTAL")

    else:
        idx = run_batch_points_csv(
            points_csv=points_csv,
            out_root=out_root,
            question=question,
            zoom_m_default=DEFAULT_ZOOM_M,
            system_preamble=system_preamble,
            temperature=temperature,
            export_tif=True,
            export_case_shapefile=True,
            export_batch_shapefile=True,
            start=DEFAULT_START,
            end=DEFAULT_END,
            max_parallel_workers=MAX_PARALLEL_WORKERS,
        )
        print("Wrote index:", idx)
        print_cost_summary(prefix="SESSION TOTAL")


In [ ]:
# THIS script converts the main index.csv rows into map point
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# =============================================================================
# INPUTS
# =============================================================================

index_csv = r"C:\Users\index.csv"
out_dir = r"C:\Users\"
os.makedirs(out_dir, exist_ok=True)

out_shp_all = os.path.join(out_dir, "ai_pond_points_all_1000m.shp")
out_shp_yes = os.path.join(out_dir, "ai_pond_points_yes_1000m.shp")
out_shp_no = os.path.join(out_dir, "ai_pond_points_no_1000m.shp")
out_plot = os.path.join(out_dir, "ai_is_there_a_pond_map_1000m.png")
out_csv_clean = os.path.join(out_dir, "ai_pond_points_clean_1000m.csv")

# =============================================================================
# READ CSV
# =============================================================================

df = pd.read_csv(index_csv)

# Keep only rows with valid lat/lon
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df = df.dropna(subset=["lat", "lon"]).copy()

# Normalize the pond field
df["ai_is_there_a_pond"] = (
    df["ai_is_there_a_pond"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Optional: keep only yes/no rows for plotting/classification
valid_labels = ["yes", "no"]
df["pond_class"] = df["ai_is_there_a_pond"].where(df["ai_is_there_a_pond"].isin(valid_labels), "other")

# Save cleaned CSV
df.to_csv(out_csv_clean, index=False)

# =============================================================================
# CREATE GEODATAFRAME
# =============================================================================

geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Shapefile field names should be short to avoid truncation issues
gdf["pond_ai"] = gdf["pond_class"]

# Save all points
gdf.to_file(out_shp_all)

# Save yes/no subsets
gdf_yes = gdf[gdf["pond_ai"] == "yes"].copy()
gdf_no = gdf[gdf["pond_ai"] == "no"].copy()

if len(gdf_yes) > 0:
    gdf_yes.to_file(out_shp_yes)

if len(gdf_no) > 0:
    gdf_no.to_file(out_shp_no)

# =============================================================================
# PLOT
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Plot "no" first so "yes" draws on top
if len(gdf_no) > 0:
    gdf_no.plot(ax=ax, markersize=12, label="No pond", alpha=0.7)

if len(gdf_yes) > 0:
    gdf_yes.plot(ax=ax, markersize=18, label="Yes pond", alpha=0.9)

gdf_other = gdf[gdf["pond_ai"] == "other"].copy()
if len(gdf_other) > 0:
    gdf_other.plot(ax=ax, markersize=10, label="Other/blank", alpha=0.5)

ax.set_title("AI Pond Classification from index.csv")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(out_plot, dpi=300)
plt.show()

# =============================================================================
# SUMMARY
# =============================================================================

print("\nDone.")
print(f"Input CSV: {index_csv}")
print(f"Clean CSV: {out_csv_clean}")
print(f"All-points shapefile: {out_shp_all}")
print(f"Yes-points shapefile: {out_shp_yes if len(gdf_yes) > 0 else 'No YES records to save'}")
print(f"No-points shapefile: {out_shp_no if len(gdf_no) > 0 else 'No NO records to save'}")
print(f"Plot: {out_plot}")
print("\nCounts:")
print(gdf["pond_ai"].value_counts(dropna=False))

In [ ]:
# THIS script exports 2 - all pond centers and only likely draftable pond centers
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path

# =============================================================================
# INPUTS
# =============================================================================

INDEX_CSV = r"C:\Users\index.csv"

OUT_DIR = r"C:\Users\"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DRAFTABLE_SHP = os.path.join(OUT_DIR, "ai_draftable_pond_centers_1000m.shp")
OUT_DRAFTABLE_GEOJSON = os.path.join(OUT_DIR, "ai_draftable_pond_centers_1000m.geojson")
OUT_DRAFTABLE_CSV = os.path.join(OUT_DIR, "ai_draftable_pond_centers_1000m.csv")

OUT_ALL_PONDS_WITH_STATUS_SHP = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_1000m.shp")
OUT_ALL_PONDS_WITH_STATUS_GEOJSON = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_1000m.geojson")
OUT_ALL_PONDS_WITH_STATUS_CSV = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_1000m.csv")

# =============================================================================
# HELPERS
# =============================================================================

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def short_shp_columns(gdf):
    """
    Rename columns to shapefile-safe names <= 10 characters.
    """
    rename = {
        "case_id": "case_id",
        "point_id": "point_id",
        "src_lon": "src_lon",
        "src_lat": "src_lat",
        "naip_id": "naip_id",
        "pond_lbl": "pond_lbl",
        "pond_label": "pond_lbl",
        "conf": "conf",
        "confidence": "conf",
        "norm_x": "norm_x",
        "norm_y": "norm_y",
        "pix_x": "pix_x",
        "pixel_x": "pix_x",
        "pix_y": "pix_y",
        "pixel_y": "pix_y",
        "ai_is_there_surface_water": "ai_sw",
        "ai_is_there_a_pond": "ai_pond",
        "ai_road_or_access_nearby": "ai_access",
        "ai_likely_draftable": "ai_draft",
        "ai_pond_count": "ai_pcnt",
        "report_png": "report_png",
        "image_png": "image_png",
        "ai_json": "ai_json",
        "pond_points_shp": "src_shp",
        "pond_points_geojson": "src_geojs",
        "pond_points_csv": "src_csv",
    }

    out = gdf.copy()
    out = out.rename(columns={c: rename[c] for c in out.columns if c in rename})

    # Shapefile is picky about object fields and long strings
    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)

    return out

# =============================================================================
# READ INDEX
# =============================================================================

if not os.path.exists(INDEX_CSV):
    raise FileNotFoundError(f"Index CSV not found:\n{INDEX_CSV}")

idx = pd.read_csv(INDEX_CSV)

required_cols = [
    "pond_points_shp",
    "ai_likely_draftable",
    "ai_is_there_a_pond",
]

missing = [c for c in required_cols if c not in idx.columns]
if missing:
    raise ValueError(
        "The index.csv is missing required columns:\n"
        + "\n".join(missing)
        + "\n\nAvailable columns are:\n"
        + "\n".join(idx.columns)
    )

idx["ai_likely_draftable_norm"] = idx["ai_likely_draftable"].apply(norm_text)
idx["ai_is_there_a_pond_norm"] = idx["ai_is_there_a_pond"].apply(norm_text)

# Keep only rows where AI said likely_draftable == yes
draftable_idx = idx[idx["ai_likely_draftable_norm"] == "yes"].copy()

print("Index rows:", len(idx))
print("Rows where ai_likely_draftable == yes:", len(draftable_idx))

# =============================================================================
# MERGE ALL POND POINTS WITH STATUS
# =============================================================================

all_pond_gdfs = []
draftable_gdfs = []

for _, row in idx.iterrows():
    shp_path = str(row.get("pond_points_shp", "")).strip()

    if not shp_path or not os.path.exists(shp_path):
        continue

    try:
        ponds = gpd.read_file(shp_path)
    except Exception as e:
        print(f"Could not read: {shp_path}")
        print(f"  {type(e).__name__}: {e}")
        continue

    if ponds.empty:
        continue

    # Add classification fields from index.csv to each pond center point
    ponds["ai_is_there_surface_water"] = row.get("ai_is_there_surface_water", "")
    ponds["ai_is_there_a_pond"] = row.get("ai_is_there_a_pond", "")
    ponds["ai_road_or_access_nearby"] = row.get("ai_road_or_access_nearby", "")
    ponds["ai_likely_draftable"] = row.get("ai_likely_draftable", "")
    ponds["ai_pond_count"] = row.get("ai_pond_count", "")
    ponds["report_png"] = row.get("report_png", "")
    ponds["image_png"] = row.get("image_png", "")
    ponds["ai_json"] = row.get("ai_json", "")
    ponds["pond_points_shp"] = shp_path
    ponds["pond_points_geojson"] = row.get("pond_points_geojson", "")
    ponds["pond_points_csv"] = row.get("pond_points_csv", "")

    all_pond_gdfs.append(ponds)

    if norm_text(row.get("ai_likely_draftable", "")) == "yes":
        draftable_gdfs.append(ponds.copy())

# =============================================================================
# EXPORT ALL POND CENTERS WITH DRAFTABLE STATUS
# =============================================================================

if len(all_pond_gdfs) > 0:
    all_ponds = gpd.GeoDataFrame(
        pd.concat(all_pond_gdfs, ignore_index=True),
        geometry="geometry",
        crs=all_pond_gdfs[0].crs
    )

    if all_ponds.crs is None:
        all_ponds = all_ponds.set_crs("EPSG:4326")

    all_ponds_wgs84 = all_ponds.to_crs("EPSG:4326")

    all_ponds_wgs84["lon"] = all_ponds_wgs84.geometry.x
    all_ponds_wgs84["lat"] = all_ponds_wgs84.geometry.y

    all_ponds_wgs84.drop(columns="geometry").to_csv(OUT_ALL_PONDS_WITH_STATUS_CSV, index=False)
    all_ponds_wgs84.to_file(OUT_ALL_PONDS_WITH_STATUS_GEOJSON, driver="GeoJSON")

    all_ponds_shp = short_shp_columns(all_ponds_wgs84)
    all_ponds_shp.to_file(OUT_ALL_PONDS_WITH_STATUS_SHP)

    print("\nWrote all pond centers with draftable status:")
    print(OUT_ALL_PONDS_WITH_STATUS_SHP)
    print(OUT_ALL_PONDS_WITH_STATUS_GEOJSON)
    print(OUT_ALL_PONDS_WITH_STATUS_CSV)

else:
    print("\nNo pond center shapefiles found to merge.")

# =============================================================================
# EXPORT ONLY DRAFTABLE POND CENTERS
# =============================================================================

if len(draftable_gdfs) > 0:
    draftable = gpd.GeoDataFrame(
        pd.concat(draftable_gdfs, ignore_index=True),
        geometry="geometry",
        crs=draftable_gdfs[0].crs
    )

    if draftable.crs is None:
        draftable = draftable.set_crs("EPSG:4326")

    draftable_wgs84 = draftable.to_crs("EPSG:4326")

    draftable_wgs84["lon"] = draftable_wgs84.geometry.x
    draftable_wgs84["lat"] = draftable_wgs84.geometry.y

    draftable_wgs84.drop(columns="geometry").to_csv(OUT_DRAFTABLE_CSV, index=False)
    draftable_wgs84.to_file(OUT_DRAFTABLE_GEOJSON, driver="GeoJSON")

    draftable_shp = short_shp_columns(draftable_wgs84)
    draftable_shp.to_file(OUT_DRAFTABLE_SHP)

    print("\nWrote draftable pond centers only:")
    print(OUT_DRAFTABLE_SHP)
    print(OUT_DRAFTABLE_GEOJSON)
    print(OUT_DRAFTABLE_CSV)

    print("\nDraftable pond count:")
    print(len(draftable_wgs84))

else:
    print("\nNo records where ai_likely_draftable == yes were found.")
    print("No draftable pond shapefile was created.")

# =============================================================================
# SUMMARY
# =============================================================================

print("\nSummary from index.csv:")
print(idx["ai_likely_draftable_norm"].value_counts(dropna=False))

print("\nDone.")

# Claude - 500m

In [ ]:
# CLAUDE VERSION OF NAIP POND / DRAFTING LOCATION WORKFLOW
# =============================================================================
# Converted from the uploaded OpenAI workflow.
# The pond prompt, system preamble, NAIP/STAC logic, output packaging,
# retries, shapefile/GeoJSON/CSV outputs, and RESULTS folder logic are preserved.
# The model call has been changed to Anthropic Claude Messages API.
#
# Required setup in your conda environment:
#   pip install anthropic
#   set ANTHROPIC_API_KEY=your_key_here
#
# Recommended first test:
#   MODE = "single"
#   MAX_PARALLEL_WORKERS = 1
#
# Then run batch:
#   MODE = "batch"
#   MAX_PARALLEL_WORKERS = 1 to 3
# =============================================================================


import os
import csv
import json
import time
import base64
import hashlib
import traceback
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from io import BytesIO

import pandas as pd
import numpy as np
from PIL import Image

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import rasterio
from rasterio.mask import mask
from rasterio.transform import xy as rio_xy

import geopandas as gpd
from shapely.geometry import shape, Point, box, mapping

import planetary_computer
from anthropic import Anthropic
import matplotlib.pyplot as plt

try:
    from importlib.metadata import version as pkg_version
except Exception:
    pkg_version = None


# =============================================================================
# CONFIG
# =============================================================================

CLAUDE_MODEL = "claude-sonnet-4-6"
# Good alternatives:
# CLAUDE_MODEL = "claude-haiku-4-5-20251001"  # cheaper/faster, likely less accurate
# CLAUDE_MODEL = "claude-opus-4-8"            # stronger, more expensive

CLAUDE_API_KEY = os.getenv("ANTHROPIC_API_KEY")
assert CLAUDE_API_KEY, "Set your ANTHROPIC_API_KEY in the environment"
client = Anthropic(api_key=CLAUDE_API_KEY)

MODEL_PRICING_PER_1M = {
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-opus-4-8": {"input": 5.00, "output": 25.00},
}

RUN_COST_TRACKER = {
    "calls": 0,
    "prompt_tokens": 0,
    "completion_tokens": 0,
    "total_tokens": 0,
    "estimated_cost_usd": 0.0,
}

# Thread locks used when MAX_PARALLEL_WORKERS > 1.
# These prevent shared CSV/log/cost/matplotlib outputs from colliding across workers.
RUN_COST_LOCK = threading.Lock()
ATTEMPTS_CSV_LOCK = threading.Lock()
LOG_FILE_LOCK = threading.Lock()
MATPLOTLIB_LOCK = threading.Lock()
PRINT_LOCK = threading.Lock()

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
SEARCH_URL = STAC_URL.rstrip("/") + "/search"

ASSUMED_GSD_M = 0.30

# zoom_m is half the scene width.
# 250 = 500 m x 500 m chips.
# 500 = 1000 m x 1000 m chips.
DEFAULT_ZOOM_M = 250
DEFAULT_START = "2023-01-01"
DEFAULT_END = "2024-12-31"

# Change this to a new folder for your final production run.
DEFAULT_OUT_ROOT = r"C:\Users\"

# Input CSV must have lon/lat fields, or longitude/latitude, or x/y.
DEFAULT_POINTS_CSV = r"C:\Users\RCVFD_grid_points_spacing500m.csv"

CASE_DIR_NAME = "CASES"
RESULTS_DIR_NAME = "RESULTS"

# Final-run safeguards.
FORCE_RERUN_ALL = False
RERUN_FAILED_OR_INVALID = True
MAX_ATTEMPTS_PER_CASE = 3
SLEEP_BETWEEN_ATTEMPTS_SEC = 3

# Parallel processing.
# 1 = sequential. 3 is a good safe default for OpenAI + NAIP + local file writing.
MAX_PARALLEL_WORKERS = 3

# Outputs required for a case to be treated as complete.
REQUIRE_CROP_TIF_FOR_COMPLETE = True
REQUIRE_CASE_SHAPEFILE_FOR_COMPLETE = False

# Results packaging.
MAX_REVIEW_REPORTS_TO_COPY = 999999
MAX_EXAMPLE_REPORTS_PER_GROUP = 999999


# =============================================================================
# COST HELPERS
# =============================================================================

def get_model_pricing(model_name):
    return MODEL_PRICING_PER_1M.get(model_name, {"input": 0.0, "output": 0.0})


def estimate_cost_usd(model_name, prompt_tokens, completion_tokens):
    pricing = get_model_pricing(model_name)
    in_cost = (prompt_tokens or 0) / 1_000_000.0 * pricing["input"]
    out_cost = (completion_tokens or 0) / 1_000_000.0 * pricing["output"]
    return in_cost + out_cost


def update_run_cost_tracker(prompt_tokens, completion_tokens, total_tokens, cost_usd):
    with RUN_COST_LOCK:
        RUN_COST_TRACKER["calls"] += 1
        RUN_COST_TRACKER["prompt_tokens"] += int(prompt_tokens or 0)
        RUN_COST_TRACKER["completion_tokens"] += int(completion_tokens or 0)
        RUN_COST_TRACKER["total_tokens"] += int(total_tokens or 0)
        RUN_COST_TRACKER["estimated_cost_usd"] += float(cost_usd or 0.0)


def print_cost_summary(prefix="RUN"):
    with RUN_COST_LOCK:
        snap = RUN_COST_TRACKER.copy()
    print(
        f"[{prefix}] "
        f"calls={snap['calls']} | "
        f"prompt_tokens={snap['prompt_tokens']} | "
        f"completion_tokens={snap['completion_tokens']} | "
        f"total_tokens={snap['total_tokens']} | "
        f"estimated_cost_usd=${snap['estimated_cost_usd']:.6f}"
    )


# =============================================================================
# GENERAL HELPERS
# =============================================================================

def ensure_dir(p):
    p = Path(p)
    p.mkdir(parents=True, exist_ok=True)
    return p


def write_json(path, obj):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")


def write_text(path, text):
    path = Path(path)
    ensure_dir(path.parent)
    path.write_text(str(text), encoding="utf-8")


def safe_lower(v):
    return "" if pd.isna(v) else str(v).strip().lower()


def parse_boolish(v):
    s = safe_lower(v)
    if s in {"true", "1", "yes", "y", "ok"}:
        return True
    if s in {"false", "0", "no", "n", ""}:
        return False
    return False


def clean_filename(s, max_len=120):
    s = "" if s is None else str(s)
    keep = []
    for ch in s:
        if ch.isalnum() or ch in "._-":
            keep.append(ch)
        else:
            keep.append("_")
    out = "".join(keep).strip("_")
    while "__" in out:
        out = out.replace("__", "_")
    return out[:max_len] if len(out) > max_len else out


def safe_case_id(row_id, lon, lat):
    rid = "point" if row_id in (None, "") else str(row_id)
    base = f"{rid}__{lat:.6f}__{lon:.6f}"
    h = hashlib.sha1(base.encode("utf-8")).hexdigest()[:8]
    safe = clean_filename(base.replace("-", "m").replace(".", "p"), max_len=140)
    return f"{safe}__{h}"


def get_pkg_versions():
    keys = [
        "numpy", "pandas", "rasterio", "geopandas", "shapely",
        "planetary-computer", "requests", "Pillow", "matplotlib", "anthropic"
    ]
    out = {}
    if pkg_version is None:
        return out
    for k in keys:
        try:
            out[k] = pkg_version(k)
        except Exception:
            pass
    return out


# =============================================================================
# NETWORK
# =============================================================================

def make_retry_session(
    total=10,
    backoff_factor=0.8,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET", "POST"),
    timeout=(10, 90),
):
    s = requests.Session()
    retry = Retry(
        total=total,
        connect=total,
        read=total,
        status=total,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=set(allowed_methods),
        raise_on_status=False,
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update(
        {
            "User-Agent": "naip-pond-final-results-workflow/1.0 (+python requests)",
            "Accept": "application/geo+json, application/json",
            "Content-Type": "application/json",
        }
    )
    return s, timeout


SESSION, REQ_TIMEOUT = make_retry_session()


# =============================================================================
# STAC SEARCH
# =============================================================================

def stac_search_naip(intersects_geojson, datetime_range, limit=200):
    payload = {
        "collections": ["naip"],
        "intersects": intersects_geojson,
        "datetime": datetime_range,
        "limit": limit,
    }
    last_err = None
    for attempt in range(1, 6):
        try:
            r = SESSION.post(SEARCH_URL, data=json.dumps(payload), timeout=REQ_TIMEOUT)
            r.raise_for_status()
            fc = r.json()
            return fc.get("features", []) or []
        except Exception as e:
            last_err = e
            time.sleep(min(2**attempt, 20))
    raise RuntimeError(f"STAC search failed after retries: {last_err}")


def find_best_naip_item(lon, lat, start=DEFAULT_START, end=DEFAULT_END, pad_deg=0.001):
    aoi = {
        "type": "Polygon",
        "coordinates": [[
            [lon - pad_deg, lat - pad_deg],
            [lon + pad_deg, lat - pad_deg],
            [lon + pad_deg, lat + pad_deg],
            [lon - pad_deg, lat + pad_deg],
            [lon - pad_deg, lat - pad_deg],
        ]],
    }

    feats = stac_search_naip(intersects_geojson=aoi, datetime_range=f"{start}/{end}", limit=200)
    if not feats:
        return None, aoi

    aoi_shape = shape(aoi)
    feats_sorted = sorted(
        feats,
        key=lambda f: shape(f["geometry"]).intersection(aoi_shape).area,
        reverse=True,
    )
    return feats_sorted[0], aoi


def extract_asset_href(feat):
    if not isinstance(feat, dict):
        return None

    assets = feat.get("assets")
    if not isinstance(assets, dict):
        assets = feat.get("properties", {}).get("assets", None)

    if not isinstance(assets, dict) or not assets:
        return None

    if "image" in assets and isinstance(assets["image"], dict) and "href" in assets["image"]:
        return assets["image"]["href"]

    for a in assets.values():
        if isinstance(a, dict) and str(a.get("href", "")).lower().endswith((".tif", ".tiff")):
            return a["href"]

    return None


# =============================================================================
# IMAGE HELPERS
# =============================================================================

def pct_scale_to_u8(rgb_arr):
    if rgb_arr.dtype == np.uint8:
        return rgb_arr
    out = []
    for b in range(rgb_arr.shape[0]):
        band = rgb_arr[b].astype(np.float32)
        if not np.isfinite(band).any():
            out.append(np.zeros_like(band, dtype=np.uint8))
            continue
        lo, hi = np.nanpercentile(band, [0.0, 99.5])
        if not np.isfinite(lo):
            lo = 0.0
        if not np.isfinite(hi) or hi <= lo:
            hi = lo + 1.0
        band = (np.clip((band - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
        out.append(band)
    return np.stack(out, axis=0)


def try_parse_json(text):
    t = (text or "").strip()
    if t.startswith("```"):
        t = t.strip("`").strip()
        if t.lower().startswith("json"):
            t = t[4:].strip()
    if "{" in t and "}" in t:
        t2 = t[t.find("{"): t.rfind("}") + 1]
    else:
        t2 = t
    try:
        return json.loads(t2), t2
    except Exception:
        return None, t2


def export_crop_tif(href_signed, crop_geom, out_tif_path):
    with rasterio.open(href_signed) as src:
        data, out_transform = mask(src, crop_geom, crop=True)
        out_meta = src.meta.copy()
        out_meta.update(
            {
                "driver": "GTiff",
                "height": data.shape[1],
                "width": data.shape[2],
                "transform": out_transform,
                "count": data.shape[0],
                "compress": "deflate",
                "tiled": True,
                "blockxsize": 256,
                "blockysize": 256,
            }
        )
        out_tif_path = Path(out_tif_path)
        ensure_dir(out_tif_path.parent)
        with rasterio.open(out_tif_path, "w", **out_meta) as dst:
            dst.write(data)
    return str(out_tif_path)


def sanitize_point_list(points, img_w, img_h):
    clean = []
    if not isinstance(points, list):
        return clean

    for i, p in enumerate(points):
        if not isinstance(p, dict):
            continue
        try:
            x = float(p.get("x", np.nan))
            y = float(p.get("y", np.nan))
        except Exception:
            continue
        if not all(np.isfinite([x, y])):
            continue
        x = max(0.0, min(1.0, x))
        y = max(0.0, min(1.0, y))
        px = int(round(x * (img_w - 1)))
        py = int(round(y * (img_h - 1)))
        conf = p.get("confidence", None)
        try:
            conf = None if conf is None else max(0.0, min(1.0, float(conf)))
        except Exception:
            conf = None
        clean.append(
            {
                "label": str(p.get("label", f"pond_{i+1}")),
                "confidence": conf,
                "x": x,
                "y": y,
                "pixel_x": px,
                "pixel_y": py,
            }
        )
    return clean


def normalized_points_to_geodataframe(point_list, crop_transform, crop_crs, point_id, case_id, lon, lat, naip_id):
    rows = []
    if not point_list:
        return gpd.GeoDataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id",
            "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y"
        ], geometry=[], crs=crop_crs)

    img_h = int(crop_transform["img_h"])
    img_w = int(crop_transform["img_w"])
    aff = crop_transform["transform"]

    for p in point_list:
        px = max(0, min(img_w - 1, int(p["pixel_x"])))
        py = max(0, min(img_h - 1, int(p["pixel_y"])))
        map_x, map_y = rio_xy(aff, py, px, offset="center")
        rows.append(
            {
                "case_id": case_id,
                "point_id": point_id,
                "src_lon": lon,
                "src_lat": lat,
                "naip_id": naip_id,
                "pond_label": p.get("label", ""),
                "confidence": p.get("confidence", None),
                "norm_x": p.get("x", None),
                "norm_y": p.get("y", None),
                "pixel_x": px,
                "pixel_y": py,
                "geometry": Point(map_x, map_y),
            }
        )
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crop_crs)


def empty_pond_points_gdf(crs="EPSG:4326"):
    return gpd.GeoDataFrame(
        columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id",
            "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y"
        ],
        geometry=[],
        crs=crs,
    )


def write_vector_safe(gdf, path, driver=None):
    path = Path(path)
    ensure_dir(path.parent)
    out = gdf.copy() if gdf is not None else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    if out.crs is None:
        out = out.set_crs(4326)

    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)

    if path.suffix.lower() == ".shp":
        rename_map = {
            "pond_label": "pond_lbl",
            "confidence": "conf",
            "pixel_x": "pix_x",
            "pixel_y": "pix_y",
            "likely_draftable": "likely_dr",
            "road_or_access_nearby": "road_acc",
            "is_there_surface_water": "surf_wtr",
            "is_there_a_pond": "pond",
        }
        out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
        out.to_file(path)
    else:
        if driver:
            out.to_file(path, driver=driver)
        elif path.suffix.lower() == ".geojson":
            out.to_file(path, driver="GeoJSON")
        elif path.suffix.lower() == ".gpkg":
            layer_name = clean_filename(path.stem, max_len=60) or "layer"
            out.to_file(path, driver="GPKG", layer=layer_name)
        else:
            out.to_file(path)
    return str(path)


def _write_report_png_unlocked(pil_image, title_text, json_text, out_path, dpi=200):
    obj, norm_text = try_parse_json(json_text)
    display_text = json.dumps(obj, indent=2) if obj is not None else norm_text

    img_w, img_h = pil_image.size
    pond_points = []
    if isinstance(obj, dict):
        pond_points = sanitize_point_list(obj.get("pond_points", []), img_w=img_w, img_h=img_h)

    fig = plt.figure(figsize=(13, 6), dpi=dpi)
    gs = fig.add_gridspec(1, 2, width_ratios=[1.05, 1.0])
    ax0 = fig.add_subplot(gs[0, 0])
    ax1 = fig.add_subplot(gs[0, 1])

    ax0.imshow(pil_image)
    ax0.axis("off")
    ax0.set_title(title_text, fontsize=10)

    for idx, p in enumerate(pond_points, start=1):
        x = p["pixel_x"]
        y = p["pixel_y"]
        ax0.scatter([x], [y], s=60, c="red", marker="o", edgecolors="white", linewidths=0.8)
        label = f"{idx}"
        conf = p.get("confidence", None)
        if conf is not None:
            label = f"{idx} ({float(conf):.2f})"
        ax0.text(
            x + 4,
            max(0, y - 4),
            label,
            color="white",
            fontsize=8,
            fontweight="bold",
            bbox=dict(facecolor="red", edgecolor="red", boxstyle="round,pad=0.2"),
        )

    ax1.axis("off")
    ax1.text(
        0.0,
        1.0,
        display_text,
        va="top",
        ha="left",
        family="monospace",
        fontsize=8.5,
        wrap=True,
    )

    plt.tight_layout()
    out_path = Path(out_path)
    ensure_dir(out_path.parent)
    plt.savefig(str(out_path), bbox_inches="tight")
    plt.close(fig)


def write_report_png(pil_image, title_text, json_text, out_path, dpi=200):
    # Matplotlib uses global state, so protect figure creation/saving when running threads.
    with MATPLOTLIB_LOCK:
        return _write_report_png_unlocked(pil_image, title_text, json_text, out_path, dpi=dpi)


# =============================================================================
# CLAUDE / ANTHROPIC VISION
# =============================================================================

def ask_image_question(pil_image, question, system_preamble=None, model=CLAUDE_MODEL, temperature=0):
    """
    Claude version of the original OpenAI image call.

    Everything upstream and downstream stays the same:
      - same PIL crop image
      - same system_preamble
      - same pond prompt/question
      - same JSON parsing/retry logic
      - same output files/index fields

    Only the API request format changes.
    """
    buf = BytesIO()
    pil_image.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    sys_msg = system_preamble or "You are a careful remote sensing analyst. Answer concisely and only from the image."

    resp = client.messages.create(
        model=model,
        max_tokens=2048,
        temperature=temperature,
        system=sys_msg,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": img_b64,
                        },
                    },
                    {"type": "text", "text": question},
                ],
            }
        ],
    )

    answer_parts = []
    for block in getattr(resp, "content", []) or []:
        if getattr(block, "type", None) == "text":
            answer_parts.append(getattr(block, "text", ""))
    answer_text = "\n".join(answer_parts).strip()

    usage = getattr(resp, "usage", None)
    prompt_tokens = getattr(usage, "input_tokens", 0) if usage else 0
    completion_tokens = getattr(usage, "output_tokens", 0) if usage else 0
    total_tokens = int(prompt_tokens or 0) + int(completion_tokens or 0)

    est_cost_usd = estimate_cost_usd(
        model_name=model,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
    )

    update_run_cost_tracker(
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        total_tokens=total_tokens,
        cost_usd=est_cost_usd,
    )

    print(
        f"[API CALL] model={model} | "
        f"input_tokens={prompt_tokens} | "
        f"output_tokens={completion_tokens} | "
        f"total_tokens={total_tokens} | "
        f"estimated_cost_usd=${est_cost_usd:.6f}"
    )

    usage_info = {
        "model": model,
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
        "total_tokens": total_tokens,
        "estimated_cost_usd": est_cost_usd,
    }
    return answer_text, usage_info


def try_parse_json_strict_or_retry(pil, question, system_preamble, temperature=0, model=CLAUDE_MODEL):
    ans1, usage1 = ask_image_question(
        pil,
        question,
        system_preamble=system_preamble,
        temperature=temperature,
        model=model,
    )
    obj1, _ = try_parse_json(ans1)
    if obj1 is not None:
        combined_usage = {
            "model": model,
            "prompt_tokens": usage1["prompt_tokens"],
            "completion_tokens": usage1["completion_tokens"],
            "total_tokens": usage1["total_tokens"],
            "estimated_cost_usd": usage1["estimated_cost_usd"],
            "num_api_calls_for_case": 1,
        }
        return ans1, True, obj1, combined_usage

    retry_q = (
        "Your previous response was not valid JSON.\n\n"
        "Return ONLY valid JSON matching the exact schema requested. "
        "Do not include markdown, prose, or comments.\n\n"
        + question
    )

    ans2, usage2 = ask_image_question(
        pil,
        retry_q,
        system_preamble=system_preamble,
        temperature=temperature,
        model=model,
    )
    obj2, _ = try_parse_json(ans2)

    combined_usage = {
        "model": model,
        "prompt_tokens": usage1["prompt_tokens"] + usage2["prompt_tokens"],
        "completion_tokens": usage1["completion_tokens"] + usage2["completion_tokens"],
        "total_tokens": usage1["total_tokens"] + usage2["total_tokens"],
        "estimated_cost_usd": usage1["estimated_cost_usd"] + usage2["estimated_cost_usd"],
        "num_api_calls_for_case": 2,
    }
    return ans2, (obj2 is not None), obj2, combined_usage


# =============================================================================
# PROMPT
# =============================================================================

def build_pond_prompt(scene_width_m=None):
    scene_width_m = scene_width_m or int(DEFAULT_ZOOM_M * 2)
    return rf"""
Analyze this NAIP RGB aerial image (~{ASSUMED_GSD_M:.2f} m / 30 cm ground sample distance).
Use ONLY what is visible in the image. Do not assume anything not directly observable.

This task must be conservative. False positives are worse than false negatives.
The primary goal is to identify visible open-water ponds that could plausibly be wildfire drafting sources.

Return ONLY valid JSON.

Context:
This image is a fixed-area crop (~{scene_width_m} m × {scene_width_m} m). The goal is to identify true visible open surface water
and specifically ponds or standing water bodies that might serve as potential drafting sources.

Whole-scene context rule:
- Do not judge a candidate feature in isolation.
- Compare the candidate to the rest of the image before deciding it is water.
- Use the whole scene to determine whether the dark area is more consistent with shadow, terrain shading, tree shadow, rock shadow, or other non-water dark features that appear elsewhere in the image.
- If similar dark shapes, tones, or textures occur throughout the scene in obvious shadows or shaded terrain, prefer "no" for pond.

Definitions (image-only):

Surface water:
Visible open water such as pond, lake, reservoir, stream reach with visible water, canal holding water,
or wetland/open water patch.

Water surface appearance rule:
- Open water usually appears smoother and more internally uniform than mud, grass, sediment, or disturbed ground.
- If the interior shows mottled texture, vegetation patches, hoof-disturbed mud, exposed sediment, or land-like texture across most of the basin, do NOT classify it as open water.

Pond:
A distinct, bounded, mostly standing open-water body with a visible open-water surface and visible edges or shoreline.

Pond basin / impoundment footprint:
A depression, stock tank, basin, or pond-shaped feature that may be dry, muddy, vegetated, only faintly damp,
or may contain too little visible water to matter operationally. A pond basin is NOT the same as visible open water.

Draftable pond:
A visible open-water pond that appears, from imagery alone, to be plausibly usable as a wildfire drafting source.
Positive evidence includes visible open water, a distinct shoreline, nearby road or vehicle access, and a feature that is not obviously too small.
This is only an image-based screening judgment and does not confirm depth, volume, bank stability, legal access, seasonal persistence, or actual field operability.

Manmade rectangular feature exclusion rules:
- Do NOT consider anything perfectly square or perfectly rectangular to be a pond unless unmistakable open water is clearly visible.
- Perfect geometric shapes are strong negative evidence for ponds in this task and should usually be treated as manmade features, not open-water ponds.
- Small dark rectangular or square features near houses, sheds, driveways, pads, or developed areas should NOT be classified as ponds unless clear open water is unmistakably visible.
- Roofs, sheds, covered tanks, liners, tarps, equipment pads, shadowed structures, and other manmade site features can appear dark and water-like from overhead.
- A neat rectangular feature is more likely to be a manmade object than a natural or draftable pond unless there is strong visible evidence of true open water.
- If a feature is adjacent to buildings or a maintained homesite, be especially cautious and prefer "no" unless water is obvious.

Shadow comparison rule:
- Before classifying a feature as pond, compare it to other dark areas in the image.
- If the candidate has similar darkness, texture, edge quality, or orientation as nearby shadows from trees, rocks, cliffs, buildings, or terrain, do NOT classify it as a pond.
- If the dark feature blends gradually into surrounding shadow or appears connected to shadowed terrain, choose "no" for pond.

Scene consistency rule:
- A true pond should remain visually distinct from the broader shadow pattern of the scene.
- If the feature can be explained by the same lighting and shadow behavior seen elsewhere in the image, it is not strong evidence for water.
- Use surrounding illumination, terrain, and nearby shadows to interpret ambiguous dark features conservatively.

Contextual shadow rule for trees and rock:
- In wooded, rocky, or mountainous scenes, many dark features are caused by tree shadow, terrain shadow, or rock relief.
- If a candidate dark feature resembles the shadow behavior of nearby trees, ridges, boulders, or slopes, do not classify it as a pond unless open water is unmistakable.

Important interpretation rules:
1. Darkness alone is NOT evidence of water.
2. Do NOT confuse shadow, vegetation, wet ground, mud, burn scar, terrain shading, roofs, tanks, troughs, containers,
   disturbed pads, equipment, or structures with ponds.
3. A true pond with open water requires BOTH:
   - a visible bounded shoreline or edge, AND
   - an interior that visibly looks like open water
4. A coherent basin shape alone is NOT enough.
5. If a feature appears mostly dry, muddy, vegetated, shallow, or only faintly damp, do NOT classify it as a pond with open water.
6. If a pond-shaped basin is visible but open water is not clearly visible, choose "no" for pond.
7. If a basin or stock tank footprint is visible without clear open water, do NOT classify it as a pond.
8. If uncertain between water and shadow, vegetation, wet ground, mud, structure, or tank, choose "no" for pond.
9. Small dark rectangular features in developed or disturbed areas are often NOT ponds.
10. Tiny dark features may be houses, sheds, tanks, pads, containers, shadows, or other manmade features. Do not call them ponds unless open water is unmistakable.
11. For wildfire drafting, size matters. If a feature looks too small, too narrow, too shallow-looking, or operationally insignificant, do not mark it draftable.
12. If the feature may contain water but appears too small to realistically support drafting operations, set "likely_draftable" to "no" or "uncertain".
13. If uncertain, choose "no" for pond and explain why.
14. If a feature is uncertain, do NOT include a point.
15. If there are no visible ponds, return an empty list for pond_points.
16. In steep, rocky, alpine, or heavily textured terrain, dark enclosed features are often shadows, rock hollows, or terrain depressions rather than open water.
17. In rocky terrain, do NOT classify a feature as a pond unless the interior appears smooth and water-like, with a clearly bounded shoreline that is distinct from surrounding rock texture and shadow.
18. If a dark feature contains visible internal texture, irregular rock pattern, or tonal variation similar to surrounding rock/shadow, do NOT classify it as open water.
19. Small dark pockets embedded in broken rock or cliff-like terrain should usually be treated as shadow or rock unless open water is unmistakable.
20. If the feature could be explained by terrain shadow or rock geometry, choose "no" for pond.

Output JSON with exactly these keys:

{{
  "is_there_surface_water": "<yes|no>",
  "is_there_a_pond": "<yes|no>",
  "road_or_access_nearby": "yes|no|uncertain",
  "likely_draftable": "<yes|no|uncertain>",
  "pond_count": <integer>,
  "evidence": "<=90 words",
  "notes": "<concise operational note>",
  "pond_points": [
    {{
      "label": "<pond>",
      "confidence": <float 0-1>,
      "x": <float 0-1>,
      "y": <float 0-1>
    }}
  ]
}}

Rules for pond_points:
- Coordinates must be normalized to the displayed crop:
  x = column / img_width
  y = row / img_height
- All coordinates must be between 0 and 1.
- Use one center point per clearly visible pond/open-water feature.
- The point should be at the approximate CENTER of the visible pond/open-water body.
- If a feature is uncertain, do NOT include a point.
- If there are no visible ponds, return [].

Decision rules:
- If there is no clearly visible open water, then "is_there_a_pond" must be "no".
- If shoreline is not visible, "is_there_a_pond" should usually be "no".
- If the interior does not look like open water, "is_there_a_pond" must be "no".
- If the feature could reasonably be shadow, vegetation, mud, wet ground, structure, container, equipment, roof, or tank, "is_there_a_pond" must be "no".
- If a pond-shaped basin or impoundment is visible but clear open water is not, "is_there_a_pond" must be "no".
- If "pond_count" is 0, "pond_points" must be [].
- If a pond appears too small or operationally insignificant for drafting, "likely_draftable" should be "no" or "uncertain".
- Do not set "likely_draftable" to "yes" unless the feature appears plausibly large enough and operationally usable from the image.
- "road_or_access_nearby" refers only to what is visible in the image, such as a road, driveway, turnout, pull-off, or clear vehicle approach immediately adjacent to the pond.
- If a clearly visible road, driveway, turnout, or open vehicle-access area reaches or lies immediately adjacent to a clearly visible pond, set "road_or_access_nearby" to "yes".
- If access is not clearly visible, set "road_or_access_nearby" to "no" or "uncertain".
- A "yes" for road_or_access_nearby does not guarantee draftability by itself, but it is strong positive evidence.
- If there is a clearly visible pond with open water AND road/access immediately adjacent to it AND the pond is not obviously tiny, then prefer "likely_draftable": "yes".
- If there is a clearly visible pond with open water AND road/access immediately adjacent, but the pond may be too small or operationally marginal, set "likely_draftable": "uncertain".
- If there is no visible access near the pond, "likely_draftable" should usually be "no" or "uncertain".
- In rocky or mountainous terrain, darkness plus enclosure is not enough; require a smooth open-water appearance and a distinct shoreline.
- If a feature is small, dark, irregular, and embedded in rocky textured terrain, prefer "is_there_a_pond": "no" unless open water is unmistakable.
- If confidence is not high that the feature is true open water, set "pond_count" to 0 and return no point.
- Be conservative, but do not ignore obvious direct road access when judging likely draftability.

THE NUMBER ONE RULE IS TO NOT HALLUCINATE
""".strip()


def build_system_preamble():
    return (
        "You are a remote sensing analyst specializing in aerial imagery interpretation for wildfire water-source screening. "
        "Use only what is directly visible in the image. "
        "Assume NAIP is about 30 cm GSD. "
        "Be conservative and prefer false negatives over false positives. "
        "Carefully distinguish true open water from shadow, vegetation, wet ground, mud, dark soil, tanks, roofs, and structures. "
        "Evaluate road or vehicle access separately from pond presence. "
        "If a visible road, driveway, turnout, or vehicle-access area lies immediately adjacent to a clearly visible pond, mark road_or_access_nearby as yes. "
        "If a clearly visible pond has direct visible access and is not obviously tiny, likely_draftable may be yes. "
        "Only return pond center points for clearly visible pond/open-water features. "
        "Each point must represent the approximate center of a clearly visible pond. "
        "Coordinates must be normalized from 0 to 1. "
        "If there is uncertainty, do not return a point. "
        "Return only valid JSON with the exact keys requested."
    )


# =============================================================================
# AI OUTPUT VALIDATION
# =============================================================================

REQUIRED_AI_KEYS = [
    "is_there_surface_water",
    "is_there_a_pond",
    "road_or_access_nearby",
    "likely_draftable",
    "pond_count",
    "evidence",
    "notes",
    "pond_points",
]


def normalize_ai_obj(ai_obj, img_w=None, img_h=None):
    if not isinstance(ai_obj, dict):
        return None

    out = dict(ai_obj)
    for k in REQUIRED_AI_KEYS:
        if k not in out:
            if k == "road_or_access_nearby":
                out[k] = "uncertain"
            elif k == "pond_points":
                out[k] = []
            elif k == "pond_count":
                out[k] = 0
            else:
                out[k] = ""

    out["is_there_surface_water"] = safe_lower(out.get("is_there_surface_water"))
    out["is_there_a_pond"] = safe_lower(out.get("is_there_a_pond"))
    out["road_or_access_nearby"] = safe_lower(out.get("road_or_access_nearby"))
    out["likely_draftable"] = safe_lower(out.get("likely_draftable"))

    if out["is_there_surface_water"] not in {"yes", "no"}:
        out["is_there_surface_water"] = "no"
    if out["is_there_a_pond"] not in {"yes", "no"}:
        out["is_there_a_pond"] = "no"
    if out["road_or_access_nearby"] not in {"yes", "no", "uncertain"}:
        out["road_or_access_nearby"] = "uncertain"
    if out["likely_draftable"] not in {"yes", "no", "uncertain"}:
        out["likely_draftable"] = "uncertain"

    try:
        out["pond_count"] = int(out.get("pond_count", 0))
    except Exception:
        out["pond_count"] = 0

    if img_w is not None and img_h is not None:
        out["pond_points"] = sanitize_point_list(out.get("pond_points", []), img_w=img_w, img_h=img_h)
    elif not isinstance(out.get("pond_points", []), list):
        out["pond_points"] = []

    if out["pond_count"] <= 0:
        out["pond_count"] = 0
        out["pond_points"] = []
    elif len(out["pond_points"]) == 0:
        # Conservative: no usable point means no mapped pond center.
        out["pond_count"] = 0
        out["is_there_a_pond"] = "no"
        out["likely_draftable"] = "no"

    out["evidence"] = "" if out.get("evidence") is None else str(out.get("evidence"))[:600]
    out["notes"] = "" if out.get("notes") is None else str(out.get("notes"))[:600]

    return out


def validate_ai_json_file(ai_json_path):
    ai_json_path = Path(ai_json_path)
    issues = []
    if not ai_json_path.exists():
        return False, ["missing_ai_json"]
    try:
        obj = json.loads(ai_json_path.read_text(encoding="utf-8"))
    except Exception as e:
        return False, [f"ai_json_read_error:{type(e).__name__}:{e}"]
    if not isinstance(obj, dict):
        return False, ["ai_json_not_dict"]
    for k in REQUIRED_AI_KEYS:
        if k not in obj:
            issues.append(f"missing_key:{k}")
    if safe_lower(obj.get("is_there_surface_water")) not in {"yes", "no"}:
        issues.append("bad_is_there_surface_water")
    if safe_lower(obj.get("is_there_a_pond")) not in {"yes", "no"}:
        issues.append("bad_is_there_a_pond")
    if safe_lower(obj.get("road_or_access_nearby")) not in {"yes", "no", "uncertain"}:
        issues.append("bad_road_or_access_nearby")
    if safe_lower(obj.get("likely_draftable")) not in {"yes", "no", "uncertain"}:
        issues.append("bad_likely_draftable")
    try:
        pc = int(obj.get("pond_count"))
        if pc < 0:
            issues.append("negative_pond_count")
    except Exception:
        issues.append("bad_pond_count")
    if not isinstance(obj.get("pond_points"), list):
        issues.append("bad_pond_points")
    return len(issues) == 0, issues


def ai_fields_for_index(ai_obj):
    if not isinstance(ai_obj, dict):
        return {
            "ai_is_there_surface_water": "",
            "ai_is_there_a_pond": "",
            "ai_road_or_access_nearby": "",
            "ai_likely_draftable": "",
            "ai_pond_count": "",
        }
    return {
        "ai_is_there_surface_water": "" if ai_obj.get("is_there_surface_water") is None else str(ai_obj.get("is_there_surface_water")),
        "ai_is_there_a_pond": "" if ai_obj.get("is_there_a_pond") is None else str(ai_obj.get("is_there_a_pond")),
        "ai_road_or_access_nearby": "" if ai_obj.get("road_or_access_nearby") is None else str(ai_obj.get("road_or_access_nearby")),
        "ai_likely_draftable": "" if ai_obj.get("likely_draftable") is None else str(ai_obj.get("likely_draftable")),
        "ai_pond_count": "" if ai_obj.get("pond_count") is None else str(ai_obj.get("pond_count")),
    }


def make_error_result(point_id, lon, lat, zoom_m, error, case_id="", case_dir="", attempt_num=""):
    return {
        "ok": False,
        "final_complete": False,
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": case_dir,
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": "",
        "parse_ok": "",
        "validation_ok": False,
        "validation_issues": "",
        "ai_is_there_surface_water": "",
        "ai_is_there_a_pond": "",
        "ai_road_or_access_nearby": "",
        "ai_likely_draftable": "",
        "ai_pond_count": "",
        "api_calls_for_case": "",
        "prompt_tokens": "",
        "completion_tokens": "",
        "total_tokens": "",
        "estimated_cost_usd": "",
        "report_png": "",
        "image_png": "",
        "ai_json": "",
        "ai_raw": "",
        "crop_tif": "",
        "pond_points_shp": "",
        "pond_points_geojson": "",
        "pond_points_csv": "",
        "error": str(error),
    }


INDEX_FIELDNAMES = [
    "ok",
    "final_complete",
    "attempt_num",
    "case_id",
    "case_dir",
    "point_id",
    "lon",
    "lat",
    "zoom_m",
    "naip_id",
    "parse_ok",
    "validation_ok",
    "validation_issues",
    "ai_is_there_surface_water",
    "ai_is_there_a_pond",
    "ai_road_or_access_nearby",
    "ai_likely_draftable",
    "ai_pond_count",
    "api_calls_for_case",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "estimated_cost_usd",
    "report_png",
    "image_png",
    "ai_json",
    "ai_raw",
    "crop_tif",
    "pond_points_shp",
    "pond_points_geojson",
    "pond_points_csv",
    "error",
]


def coerce_result_fields(res):
    out = {k: res.get(k, "") for k in INDEX_FIELDNAMES}
    return out


# =============================================================================
# CASE COMPLETENESS CHECKING
# =============================================================================

def get_expected_case_paths(out_root, case_id):
    case_dir = Path(out_root) / CASE_DIR_NAME / case_id
    return {
        "case_dir": case_dir,
        "crop_png": case_dir / "crop.png",
        "report_png": case_dir / "report.png",
        "ai_json": case_dir / "ai.json",
        "ai_raw": case_dir / "ai_raw.txt",
        "meta_json": case_dir / "meta.json",
        "crop_tif": case_dir / "crop.tif",
        "pond_points_shp": case_dir / "ai_pond_points.shp",
        "pond_points_geojson": case_dir / "ai_pond_points.geojson",
        "pond_points_csv": case_dir / "ai_pond_points.csv",
    }


def case_is_complete(out_root, case_id):
    paths = get_expected_case_paths(out_root, case_id)
    issues = []

    if not paths["case_dir"].exists():
        issues.append("missing_case_dir")
    if not paths["crop_png"].exists():
        issues.append("missing_crop_png")
    if not paths["report_png"].exists():
        issues.append("missing_report_png")
    if not paths["ai_json"].exists():
        issues.append("missing_ai_json")
    if not paths["meta_json"].exists():
        issues.append("missing_meta_json")
    if REQUIRE_CROP_TIF_FOR_COMPLETE and not paths["crop_tif"].exists():
        issues.append("missing_crop_tif")
    if REQUIRE_CASE_SHAPEFILE_FOR_COMPLETE and not paths["pond_points_shp"].exists():
        issues.append("missing_pond_points_shp")
    if not paths["pond_points_geojson"].exists():
        issues.append("missing_pond_points_geojson")
    if not paths["pond_points_csv"].exists():
        issues.append("missing_pond_points_csv")

    ai_ok, ai_issues = validate_ai_json_file(paths["ai_json"])
    if not ai_ok:
        issues.extend(ai_issues)

    return len(issues) == 0, issues


def reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0):
    paths = get_expected_case_paths(out_root, case_id)
    validation_ok, validation_issues = case_is_complete(out_root, case_id)
    ai_obj = None
    try:
        ai_obj = json.loads(paths["ai_json"].read_text(encoding="utf-8"))
    except Exception:
        ai_obj = None
    ai_fields = ai_fields_for_index(ai_obj)

    naip_id = ""
    parse_ok = validation_ok
    try:
        meta = json.loads(paths["meta_json"].read_text(encoding="utf-8"))
        naip_id = meta.get("naip_id", "")
        parse_ok = bool(meta.get("parse_ok", validation_ok))
    except Exception:
        pass

    return coerce_result_fields({
        "ok": bool(validation_ok),
        "final_complete": bool(validation_ok),
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": str(paths["case_dir"]),
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": naip_id,
        "parse_ok": bool(parse_ok),
        "validation_ok": bool(validation_ok),
        "validation_issues": ";".join(validation_issues),
        **ai_fields,
        "api_calls_for_case": 0,
        "prompt_tokens": 0,
        "completion_tokens": 0,
        "total_tokens": 0,
        "estimated_cost_usd": 0,
        "report_png": str(paths["report_png"]) if paths["report_png"].exists() else "",
        "image_png": str(paths["crop_png"]) if paths["crop_png"].exists() else "",
        "ai_json": str(paths["ai_json"]) if paths["ai_json"].exists() else "",
        "ai_raw": str(paths["ai_raw"]) if paths["ai_raw"].exists() else "",
        "crop_tif": str(paths["crop_tif"]) if paths["crop_tif"].exists() else "",
        "pond_points_shp": str(paths["pond_points_shp"]) if paths["pond_points_shp"].exists() else "",
        "pond_points_geojson": str(paths["pond_points_geojson"]) if paths["pond_points_geojson"].exists() else "",
        "pond_points_csv": str(paths["pond_points_csv"]) if paths["pond_points_csv"].exists() else "",
        "error": "" if validation_ok else "Incomplete existing case: " + ";".join(validation_issues),
    })


def validate_case_result(res):
    if not parse_boolish(res.get("ok")):
        return False, ["result_ok_false"]
    if not parse_boolish(res.get("parse_ok")):
        return False, ["parse_ok_false"]
    case_id = res.get("case_id", "")
    case_dir = res.get("case_dir", "")
    issues = []
    if not case_id:
        issues.append("missing_case_id")
    if not case_dir or not Path(case_dir).exists():
        issues.append("missing_case_dir")
    for key in ["report_png", "image_png", "ai_json", "pond_points_geojson", "pond_points_csv"]:
        p = res.get(key, "")
        if not p or not Path(p).exists():
            issues.append(f"missing_{key}")
    if REQUIRE_CROP_TIF_FOR_COMPLETE:
        p = res.get("crop_tif", "")
        if not p or not Path(p).exists():
            issues.append("missing_crop_tif")
    ai_ok, ai_issues = validate_ai_json_file(res.get("ai_json", ""))
    if not ai_ok:
        issues.extend(ai_issues)
    return len(issues) == 0, issues


# =============================================================================
# STUDENT / REVIEW TEMPLATE
# =============================================================================

def student_review_template(case_meta):
    return {
        "case_id": case_meta.get("case_id"),
        "point_id": case_meta.get("point_id"),
        "lon": case_meta.get("lon"),
        "lat": case_meta.get("lat"),
        "zoom_m": case_meta.get("zoom_m"),
        "reviewer_name": "",
        "review_date_local": "",
        "labels": {
            "is_there_surface_water": "",
            "is_there_a_pond": "",
            "road_or_access_nearby": "",
            "pond_count_estimate": None,
            "likely_draftable": "",
            "notes": ""
        },
        "agreement_with_ai": {
            "agree_overall": "",
            "disagreement_fields": [],
            "why": ""
        },
        "qc_flags": {
            "image_quality_issue": False,
            "shadow_confusion": False,
            "seasonal_dryness": False,
            "other": ""
        }
    }


# =============================================================================
# CORE CASE RUNNER
# =============================================================================

def naip_qa_case(
    point_id,
    lon,
    lat,
    out_root,
    question,
    zoom_m=DEFAULT_ZOOM_M,
    system_preamble=None,
    temperature=0,
    export_tif=True,
    export_case_shapefile=True,
    start=DEFAULT_START,
    end=DEFAULT_END,
    attempt_num=1,
):
    out_root = Path(out_root)
    cases_root = ensure_dir(out_root / CASE_DIR_NAME)

    case_id = safe_case_id(point_id, lon, lat)
    case_dir = ensure_dir(cases_root / case_id)

    feat, _aoi = find_best_naip_item(lon, lat, start=start, end=end)
    if feat is None:
        return make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(case_dir),
            attempt_num=attempt_num,
            error="No NAIP scene found in window.",
        )

    item_id = feat.get("id")
    href = extract_asset_href(feat)
    if href is None:
        res = make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(case_dir),
            attempt_num=attempt_num,
            error="NAIP item missing usable image href.",
        )
        res["naip_id"] = item_id or ""
        return res

    href_signed = planetary_computer.sign(href)

    with rasterio.open(href_signed) as src:
        pt = gpd.GeoDataFrame(geometry=[Point(lon, lat)], crs=4326).to_crs(src.crs)
        cx, cy = pt.geometry.iloc[0].x, pt.geometry.iloc[0].y
        crop_geom_srccrs = [mapping(box(cx - zoom_m, cy - zoom_m, cx + zoom_m, cy + zoom_m))]

        data, crop_transform_affine = mask(src, crop_geom_srccrs, crop=True)
        if data.shape[0] >= 3:
            rgb = data[:3, :, :]
        else:
            rgb = np.repeat(data[0:1, :, :], 3, axis=0)

        rgb_u8 = pct_scale_to_u8(rgb)
        pil = Image.fromarray(np.transpose(rgb_u8, (1, 2, 0)))

        src_crs = src.crs
        src_crs_wkt = src.crs.to_wkt() if src.crs else None
        src_res = getattr(src, "res", None)
        src_bounds = tuple(src.bounds) if getattr(src, "bounds", None) else None

    crop_png = case_dir / "crop.png"
    pil.save(str(crop_png), format="PNG")

    answer_text, parse_ok, ai_obj, usage_info = try_parse_json_strict_or_retry(
        pil,
        question,
        system_preamble=system_preamble,
        temperature=temperature,
        model=CLAUDE_MODEL,
    )

    img_w, img_h = pil.size
    ai_obj = normalize_ai_obj(ai_obj, img_w=img_w, img_h=img_h) if isinstance(ai_obj, dict) else None
    ai_fields = ai_fields_for_index(ai_obj if parse_ok else None)

    ai_json_path = case_dir / "ai.json"
    ai_raw_path = case_dir / "ai_raw.txt"

    if isinstance(ai_obj, dict):
        write_json(ai_json_path, ai_obj)
        if ai_raw_path.exists():
            ai_raw_path.unlink()
        answer_text_for_report = json.dumps(ai_obj)
    else:
        obj, norm_text = try_parse_json(answer_text)
        obj = normalize_ai_obj(obj, img_w=img_w, img_h=img_h) if isinstance(obj, dict) else None
        if isinstance(obj, dict):
            write_json(ai_json_path, obj)
            if ai_raw_path.exists():
                ai_raw_path.unlink()
            answer_text_for_report = json.dumps(obj)
            ai_obj = obj
            parse_ok = True
            ai_fields = ai_fields_for_index(ai_obj)
        else:
            write_text(ai_raw_path, norm_text)
            if ai_json_path.exists():
                ai_json_path.unlink()
            answer_text_for_report = norm_text

    report_png = case_dir / "report.png"
    title_text = f"NAIP (~{ASSUMED_GSD_M:.2f} m) @ {lat:.6f}, {lon:.6f}\n{item_id} | scene ~{int(zoom_m * 2)} m x {int(zoom_m * 2)} m"
    write_report_png(pil, title_text, answer_text_for_report, report_png, dpi=200)

    crop_tif_path = ""
    if export_tif:
        crop_tif = case_dir / "crop.tif"
        crop_tif_path = export_crop_tif(href_signed=href_signed, crop_geom=crop_geom_srccrs, out_tif_path=crop_tif)

    pond_points_shp = ""
    pond_points_geojson = ""
    pond_points_csv = ""

    if isinstance(ai_obj, dict):
        pond_points = sanitize_point_list(ai_obj.get("pond_points", []), img_w=img_w, img_h=img_h)
        crop_transform = {
            "transform": crop_transform_affine,
            "img_w": img_w,
            "img_h": img_h,
        }

        case_points_gdf = normalized_points_to_geodataframe(
            point_list=pond_points,
            crop_transform=crop_transform,
            crop_crs=src_crs,
            point_id=point_id,
            case_id=case_id,
            lon=lon,
            lat=lat,
            naip_id=item_id,
        )

        if case_points_gdf is not None and len(case_points_gdf) > 0:
            case_points_gdf_wgs84 = case_points_gdf.to_crs(4326)
        else:
            case_points_gdf_wgs84 = empty_pond_points_gdf(crs="EPSG:4326")

        if export_case_shapefile:
            pond_points_shp = str(case_dir / "ai_pond_points.shp")
            write_vector_safe(case_points_gdf_wgs84, pond_points_shp)

        pond_points_geojson = str(case_dir / "ai_pond_points.geojson")
        write_vector_safe(case_points_gdf_wgs84, pond_points_geojson, driver="GeoJSON")

        pond_points_csv = str(case_dir / "ai_pond_points.csv")
        if len(case_points_gdf_wgs84) > 0:
            tmp = case_points_gdf_wgs84.copy()
            tmp["lon"] = tmp.geometry.x
            tmp["lat"] = tmp.geometry.y
            tmp.drop(columns=["geometry"]).to_csv(pond_points_csv, index=False)
        else:
            pd.DataFrame(columns=[
                "case_id", "point_id", "src_lon", "src_lat", "naip_id",
                "pond_label", "confidence", "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat"
            ]).to_csv(pond_points_csv, index=False)

    props = feat.get("properties", {}) if isinstance(feat, dict) else {}
    case_meta = {
        "case_id": case_id,
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "scene_width_m": zoom_m * 2,
        "datetime_local": datetime.now().isoformat(timespec="seconds"),
        "assumed_gsd_m_prompt": ASSUMED_GSD_M,
        "naip_id": item_id,
        "naip_href_signed_runtime": href_signed,
        "stac_properties_subset": {
            "datetime": props.get("datetime"),
            "naip:year": props.get("naip:year"),
            "gsd": props.get("gsd"),
            "proj:epsg": props.get("proj:epsg"),
        },
        "src_crs_wkt": src_crs_wkt,
        "src_res": src_res,
        "src_bounds": src_bounds,
        "packages": get_pkg_versions(),
        "parse_ok": bool(parse_ok),
        "claude_model": CLAUDE_MODEL,
        "temperature": temperature,
        "start": start,
        "end": end,
        "ai_index_fields": ai_fields,
        "claude_usage": usage_info,
        "run_cost_tracker_snapshot": RUN_COST_TRACKER.copy(),
        "exports": {
            "case_point_shapefile": pond_points_shp,
            "case_point_geojson": pond_points_geojson,
            "case_point_csv": pond_points_csv,
        }
    }
    write_json(case_dir / "meta.json", case_meta)
    write_json(case_dir / "student_review.json", student_review_template(case_meta))
    write_text(
        case_dir / "README.txt",
        "Open report.png first.\n"
        "Red points indicate AI-detected pond/open-water centers.\n"
        "Open ai_pond_points.shp or ai_pond_points.geojson for GIS use.\n"
        "Then fill student_review.json if this case is selected for review.\n"
        "Do NOT treat this as a confirmed drafting source without local/field verification.\n",
    )

    res = {
        "ok": True,
        "final_complete": False,
        "attempt_num": attempt_num,
        "case_id": case_id,
        "case_dir": str(case_dir),
        "point_id": point_id,
        "lon": lon,
        "lat": lat,
        "zoom_m": zoom_m,
        "naip_id": item_id or "",
        "parse_ok": bool(parse_ok),
        "validation_ok": False,
        "validation_issues": "",
        **ai_fields,
        "api_calls_for_case": usage_info["num_api_calls_for_case"],
        "prompt_tokens": usage_info["prompt_tokens"],
        "completion_tokens": usage_info["completion_tokens"],
        "total_tokens": usage_info["total_tokens"],
        "estimated_cost_usd": round(usage_info["estimated_cost_usd"], 6),
        "report_png": str(report_png),
        "image_png": str(crop_png),
        "ai_json": str(ai_json_path) if ai_json_path.exists() else "",
        "ai_raw": str(ai_raw_path) if ai_raw_path.exists() else "",
        "crop_tif": crop_tif_path or "",
        "pond_points_shp": pond_points_shp,
        "pond_points_geojson": pond_points_geojson,
        "pond_points_csv": pond_points_csv,
        "error": "",
    }

    validation_ok, validation_issues = validate_case_result(res)
    res["validation_ok"] = bool(validation_ok)
    res["validation_issues"] = ";".join(validation_issues)
    res["final_complete"] = bool(validation_ok)
    if not validation_ok:
        res["ok"] = False
        res["error"] = "Case failed validation: " + ";".join(validation_issues)

    return coerce_result_fields(res)


# =============================================================================
# INPUT CSV READING
# =============================================================================

def read_points_csv(points_csv, zoom_m_default=DEFAULT_ZOOM_M, start=DEFAULT_START, end=DEFAULT_END):
    points_csv = Path(points_csv)
    if not points_csv.exists():
        raise FileNotFoundError(f"Input points CSV does not exist: {points_csv}")

    with points_csv.open("r", encoding="utf-8-sig", newline="") as f_in:
        sample = f_in.read(8192)
        f_in.seek(0)
        try:
            dialect = csv.Sniffer().sniff(sample, delimiters=[",", "\t", ";", "|"])
        except Exception:
            dialect = csv.get_dialect("excel")
            dialect.delimiter = "\t" if "\t" in sample and "," not in sample else ","
        reader = csv.DictReader(f_in, dialect=dialect)
        if not reader.fieldnames:
            raise RuntimeError("Could not read header row from points file.")

        raw_fields = list(reader.fieldnames)
        norm_fields = [((h or "").strip().lower()) for h in raw_fields]
        header_map = {raw: norm for raw, norm in zip(raw_fields, norm_fields)}

        def get_value(row, *keys):
            keys = set(keys)
            for k in keys:
                if k in row and row[k] not in (None, ""):
                    return row[k]
            for raw_k, norm_k in header_map.items():
                if norm_k in keys:
                    v = row.get(raw_k, "")
                    if v not in (None, ""):
                        return v
            return ""

        rows = []
        for i, row in enumerate(reader, start=1):
            point_id = get_value(row, "id", "point_id", "name", "fid") or str(i)
            lon_s = get_value(row, "lon", "longitude", "x")
            lat_s = get_value(row, "lat", "latitude", "y")
            if lon_s == "" or lat_s == "":
                rows.append({
                    "input_order": i,
                    "input_valid": False,
                    "point_id": point_id,
                    "lon": np.nan,
                    "lat": np.nan,
                    "zoom_m": zoom_m_default,
                    "start": start,
                    "end": end,
                    "input_error": f"Missing lon/lat in row {i}: {row}",
                })
                continue
            try:
                lon = float(str(lon_s).strip())
                lat = float(str(lat_s).strip())
            except Exception as e:
                rows.append({
                    "input_order": i,
                    "input_valid": False,
                    "point_id": point_id,
                    "lon": np.nan,
                    "lat": np.nan,
                    "zoom_m": zoom_m_default,
                    "start": start,
                    "end": end,
                    "input_error": f"Bad lon/lat in row {i}: {type(e).__name__}: {e}",
                })
                continue

            zoom_s = get_value(row, "zoom_m", "zoom", "buffer_m")
            try:
                zoom_m = float(str(zoom_s).strip()) if zoom_s not in ("", None) else float(zoom_m_default)
            except Exception:
                zoom_m = float(zoom_m_default)

            row_start = (get_value(row, "start") or start).strip()
            row_end = (get_value(row, "end") or end).strip()

            rows.append({
                "input_order": i,
                "input_valid": True,
                "point_id": point_id,
                "lon": lon,
                "lat": lat,
                "zoom_m": zoom_m,
                "start": row_start,
                "end": row_end,
                "input_error": "",
            })

    df = pd.DataFrame(rows)
    df["case_id"] = df.apply(
        lambda r: safe_case_id(r["point_id"], r["lon"], r["lat"]) if bool(r["input_valid"]) else "",
        axis=1,
    )
    return df


# =============================================================================
# BATCH SHAPEFILE / MERGED OUTPUT HELPERS
# =============================================================================

def build_batch_pond_points_layers(index_csv_path, out_root):
    out_root = Path(out_root)
    index_csv_path = Path(index_csv_path)
    merged = []

    if not index_csv_path.exists():
        return ""

    idx = pd.read_csv(index_csv_path, dtype=str).fillna("")
    for _, row in idx.iterrows():
        csv_path = str(row.get("pond_points_csv", "")).strip()
        if csv_path and Path(csv_path).exists():
            try:
                pts = pd.read_csv(csv_path)
                if len(pts) > 0 and {"lon", "lat"}.issubset(pts.columns):
                    for f in [
                        "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
                        "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json"
                    ]:
                        pts[f] = row.get(f, "")
                    merged.append(pts)
            except Exception as e:
                print(f"Warning: could not read pond point CSV {csv_path}: {e}")

    batch_shp = out_root / "batch_ai_pond_points.shp"
    batch_geojson = out_root / "batch_ai_pond_points.geojson"
    batch_gpkg = out_root / "batch_ai_pond_points.gpkg"
    batch_csv = out_root / "batch_ai_pond_points.csv"

    if len(merged) == 0:
        out_csv = pd.DataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id", "pond_label", "confidence",
            "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat",
            "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
            "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json"
        ])
        out_csv.to_csv(batch_csv, index=False)
        empty = empty_pond_points_gdf(crs="EPSG:4326")
        write_vector_safe(empty, batch_shp)
        write_vector_safe(empty, batch_geojson, driver="GeoJSON")
        write_vector_safe(empty, batch_gpkg)
        return str(batch_shp)

    all_pts = pd.concat(merged, ignore_index=True)
    all_pts.to_csv(batch_csv, index=False)
    gdf = gpd.GeoDataFrame(
        all_pts,
        geometry=gpd.points_from_xy(all_pts["lon"].astype(float), all_pts["lat"].astype(float)),
        crs="EPSG:4326",
    )
    write_vector_safe(gdf, batch_shp)
    write_vector_safe(gdf, batch_geojson, driver="GeoJSON")
    write_vector_safe(gdf, batch_gpkg)
    return str(batch_shp)


def write_index_csv(index_path, rows):
    index_path = Path(index_path)
    ensure_dir(index_path.parent)
    with index_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=INDEX_FIELDNAMES)
        writer.writeheader()
        for row in rows:
            writer.writerow(coerce_result_fields(row))
    return str(index_path)


def append_attempt_csv(attempts_path, row):
    attempts_path = Path(attempts_path)
    ensure_dir(attempts_path.parent)
    with ATTEMPTS_CSV_LOCK:
        write_header = not attempts_path.exists()
        with attempts_path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=INDEX_FIELDNAMES)
            if write_header:
                writer.writeheader()
            writer.writerow(coerce_result_fields(row))


# =============================================================================
# RESULTS PACKAGING
# =============================================================================

def safe_read_index(index_csv):
    if not Path(index_csv).exists():
        return pd.DataFrame(columns=INDEX_FIELDNAMES)
    return pd.read_csv(index_csv, dtype=str).fillna("")


def make_all_grid_gdf(idx):
    if len(idx) == 0:
        return gpd.GeoDataFrame(columns=list(idx.columns), geometry=[], crs="EPSG:4326")
    df = idx.copy()
    df["lon_num"] = pd.to_numeric(df["lon"], errors="coerce")
    df["lat_num"] = pd.to_numeric(df["lat"], errors="coerce")
    df = df.dropna(subset=["lon_num", "lat_num"])
    if len(df) == 0:
        return gpd.GeoDataFrame(columns=list(idx.columns), geometry=[], crs="EPSG:4326")
    return gpd.GeoDataFrame(
        df.drop(columns=["lon_num", "lat_num"]),
        geometry=gpd.points_from_xy(df["lon_num"], df["lat_num"]),
        crs="EPSG:4326",
    )


def read_candidate_points_from_index(idx):
    merged = []
    for _, row in idx.iterrows():
        p = str(row.get("pond_points_csv", "")).strip()
        if not p or not Path(p).exists():
            continue
        try:
            pts = pd.read_csv(p)
            if len(pts) == 0:
                continue
            for f in [
                "ok", "final_complete", "validation_ok", "validation_issues",
                "ai_is_there_surface_water", "ai_is_there_a_pond", "ai_road_or_access_nearby",
                "ai_likely_draftable", "ai_pond_count", "report_png", "image_png", "ai_json", "case_dir"
            ]:
                pts[f] = row.get(f, "")
            merged.append(pts)
        except Exception as e:
            print(f"Warning: failed reading candidate points {p}: {e}")

    if len(merged) == 0:
        return pd.DataFrame(columns=[
            "case_id", "point_id", "src_lon", "src_lat", "naip_id", "pond_label", "confidence",
            "norm_x", "norm_y", "pixel_x", "pixel_y", "lon", "lat",
            "ai_likely_draftable", "ai_road_or_access_nearby", "report_png", "image_png", "ai_json"
        ])
    return pd.concat(merged, ignore_index=True)


def build_results_summary(idx, cand_df, attempts_df=None):
    total_input = len(idx)
    ok = idx["ok"].map(parse_boolish).sum() if "ok" in idx else 0
    complete = idx["final_complete"].map(parse_boolish).sum() if "final_complete" in idx else 0
    failures = total_input - complete

    def count_value(field, val):
        if field not in idx.columns:
            return 0
        return int((idx[field].map(safe_lower) == val).sum())

    summary = {
        "created_local": datetime.now().isoformat(timespec="seconds"),
        "model": CLAUDE_MODEL,
        "assumed_gsd_m": ASSUMED_GSD_M,
        "total_input_points": int(total_input),
        "successful_complete_interpretations": int(complete),
        "ok_rows": int(ok),
        "failed_or_incomplete_rows": int(failures),
        "surface_water_yes": count_value("ai_is_there_surface_water", "yes"),
        "pond_yes": count_value("ai_is_there_a_pond", "yes"),
        "road_or_access_yes": count_value("ai_road_or_access_nearby", "yes"),
        "road_or_access_uncertain": count_value("ai_road_or_access_nearby", "uncertain"),
        "likely_draftable_yes": count_value("ai_likely_draftable", "yes"),
        "likely_draftable_uncertain": count_value("ai_likely_draftable", "uncertain"),
        "candidate_pond_center_points": int(len(cand_df)),
        "run_cost_tracker": RUN_COST_TRACKER.copy(),
    }
    if attempts_df is not None and len(attempts_df) > 0:
        summary["total_attempt_rows"] = int(len(attempts_df))
        summary["attempt_rows_ok"] = int(attempts_df["ok"].map(parse_boolish).sum()) if "ok" in attempts_df else 0
    return summary


def write_simple_bar(labels, values, title, ylabel, out_png):
    out_png = Path(out_png)
    ensure_dir(out_png.parent)
    fig, ax = plt.subplots(figsize=(8, 5), dpi=200)
    ax.bar(labels, values)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=25)
    for i, v in enumerate(values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def write_map_figure(all_gdf, cand_gdf, out_png):
    out_png = Path(out_png)
    ensure_dir(out_png.parent)
    fig, ax = plt.subplots(figsize=(8, 8), dpi=220)
    if all_gdf is not None and len(all_gdf) > 0:
        all_gdf.plot(ax=ax, markersize=5, color="lightgray", edgecolor="none", label="Interpreted image chips")
    if cand_gdf is not None and len(cand_gdf) > 0:
        cand_gdf.plot(ax=ax, markersize=18, color="red", edgecolor="black", linewidth=0.3, label="AI candidate pond centers")
    ax.set_title("AI-screened candidate pond centers")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)


def copy_review_reports(idx, cand_df, review_dir):
    review_dir = ensure_dir(review_dir)
    copied = []

    # Copy candidate reports first.
    if len(cand_df) > 0:
        tmp = cand_df.copy()
        if "confidence" in tmp.columns:
            tmp["confidence_num"] = pd.to_numeric(tmp["confidence"], errors="coerce").fillna(-1)
            tmp = tmp.sort_values("confidence_num", ascending=False)
        seen = set()
        for _, row in tmp.iterrows():
            case_id = str(row.get("case_id", ""))
            if not case_id or case_id in seen:
                continue
            seen.add(case_id)
            src = str(row.get("report_png", ""))
            if src and Path(src).exists():
                dst = review_dir / "candidate_reports" / f"candidate_{len(copied)+1:04d}_{clean_filename(case_id, 80)}.png"
                ensure_dir(dst.parent)
                shutil.copy2(src, dst)
                copied.append(str(dst))
                if len(copied) >= MAX_REVIEW_REPORTS_TO_COPY:
                    break

    # Copy a small set of negative reports for comparison.
    neg_dir = review_dir / "negative_examples"
    neg_count = 0
    if len(idx) > 0:
        neg = idx[idx.get("ai_is_there_a_pond", pd.Series([""] * len(idx))).map(safe_lower) == "no"].head(MAX_EXAMPLE_REPORTS_PER_GROUP)
        for _, row in neg.iterrows():
            src = str(row.get("report_png", ""))
            case_id = str(row.get("case_id", ""))
            if src and Path(src).exists():
                dst = neg_dir / f"negative_{neg_count+1:04d}_{clean_filename(case_id, 80)}.png"
                ensure_dir(dst.parent)
                shutil.copy2(src, dst)
                neg_count += 1

    return copied


def build_results_folder(out_root, points_csv, prompt_text, system_preamble):
    out_root = Path(out_root)
    results_root = ensure_dir(out_root / RESULTS_DIR_NAME)
    tables_dir = ensure_dir(results_root / "tables")
    gis_dir = ensure_dir(results_root / "gis")
    figs_dir = ensure_dir(results_root / "figures")
    review_dir = ensure_dir(results_root / "review_pack")
    qa_dir = ensure_dir(results_root / "qa")
    docs_dir = ensure_dir(results_root / "documents")

    index_csv = out_root / "index.csv"
    attempts_csv = out_root / "index_attempts.csv"
    idx = safe_read_index(index_csv)
    attempts_df = safe_read_index(attempts_csv) if attempts_csv.exists() else pd.DataFrame()
    cand_df = read_candidate_points_from_index(idx)

    # Tables.
    idx.to_csv(tables_dir / "final_index_one_row_per_input_point.csv", index=False)
    cand_df.to_csv(tables_dir / "candidate_pond_centers.csv", index=False)
    if len(attempts_df) > 0:
        attempts_df.to_csv(qa_dir / "all_attempts_including_retries.csv", index=False)

    failures = idx[~idx["final_complete"].map(parse_boolish)].copy() if len(idx) > 0 else idx.copy()
    failures.to_csv(qa_dir / "FAILURES_NEED_REVIEW.csv", index=False)
    failures.to_csv(tables_dir / "failures_need_review.csv", index=False)

    # GIS outputs.
    all_gdf = make_all_grid_gdf(idx)
    if len(all_gdf) > 0:
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.geojson", driver="GeoJSON")
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.gpkg")
        write_vector_safe(all_gdf, gis_dir / "all_interpreted_grid_points.shp")

    if len(cand_df) > 0:
        cand_gdf = gpd.GeoDataFrame(
            cand_df.copy(),
            geometry=gpd.points_from_xy(cand_df["lon"].astype(float), cand_df["lat"].astype(float)),
            crs="EPSG:4326",
        )
    else:
        cand_gdf = empty_pond_points_gdf(crs="EPSG:4326")

    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.geojson", driver="GeoJSON")
    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.gpkg")
    write_vector_safe(cand_gdf, gis_dir / "candidate_pond_centers.shp")

    # Keep compatibility batch outputs at root too.
    try:
        build_batch_pond_points_layers(index_csv, out_root)
    except Exception as e:
        print(f"Warning: failed to write root batch layers: {e}")

    # Figures.
    summary = build_results_summary(idx, cand_df, attempts_df=attempts_df)
    summary_rows = pd.DataFrame([summary])
    summary_rows.to_csv(tables_dir / "run_summary.csv", index=False)
    write_json(tables_dir / "run_summary.json", summary)

    write_simple_bar(
        ["Input chips", "Successful", "Failed/incomplete", "Candidate centers"],
        [
            summary["total_input_points"],
            summary["successful_complete_interpretations"],
            summary["failed_or_incomplete_rows"],
            summary["candidate_pond_center_points"],
        ],
        "Run summary",
        "Count",
        figs_dir / "run_summary_counts.png",
    )

    write_simple_bar(
        ["Surface water yes", "Pond yes", "Access yes", "Likely draftable yes"],
        [
            summary["surface_water_yes"],
            summary["pond_yes"],
            summary["road_or_access_yes"],
            summary["likely_draftable_yes"],
        ],
        "AI interpretation counts",
        "Image-chip count",
        figs_dir / "ai_interpretation_counts.png",
    )

    write_map_figure(all_gdf, cand_gdf, figs_dir / "candidate_pond_centers_map.png")

    # Review-pack image copies.
    copied_reports = copy_review_reports(idx, cand_df, review_dir)

    # Prompt / config / context docs.
    write_text(docs_dir / "prompt_used.txt", prompt_text)
    write_text(docs_dir / "system_preamble_used.txt", system_preamble)
    write_json(docs_dir / "run_config.json", {
        "points_csv": str(points_csv),
        "out_root": str(out_root),
        "claude_model": CLAUDE_MODEL,
        "assumed_gsd_m": ASSUMED_GSD_M,
        "default_zoom_m": DEFAULT_ZOOM_M,
        "default_scene_width_m": DEFAULT_ZOOM_M * 2,
        "default_start": DEFAULT_START,
        "default_end": DEFAULT_END,
        "force_rerun_all": FORCE_RERUN_ALL,
        "rerun_failed_or_invalid": RERUN_FAILED_OR_INVALID,
        "max_attempts_per_case": MAX_ATTEMPTS_PER_CASE,
        "require_crop_tif_for_complete": REQUIRE_CROP_TIF_FOR_COMPLETE,
        "packages": get_pkg_versions(),
    })

    context_md = f"""# Results context for AI writing

This folder contains a production-style output package for the NAIP AI pond/drafting-location screening workflow.

## Main interpretation summary

- Input image chips / grid points: {summary['total_input_points']}
- Successful complete interpretations: {summary['successful_complete_interpretations']}
- Failed or incomplete interpretations after retries: {summary['failed_or_incomplete_rows']}
- Image chips where AI reported surface water = yes: {summary['surface_water_yes']}
- Image chips where AI reported pond = yes: {summary['pond_yes']}
- Image chips where AI reported visible nearby road/access = yes: {summary['road_or_access_yes']}
- Image chips where AI reported likely_draftable = yes: {summary['likely_draftable_yes']}
- Candidate pond center points exported to GIS: {summary['candidate_pond_center_points']}

## How to use this folder to write a Results section

Use these files first:

1. `tables/run_summary.csv` — one-row summary of the run.
2. `tables/final_index_one_row_per_input_point.csv` — one row per input image chip/grid point.
3. `tables/candidate_pond_centers.csv` — one row per mapped AI pond-center point.
4. `gis/candidate_pond_centers.gpkg` or `.geojson` — GIS-ready candidate pond-center layer.
5. `gis/all_interpreted_grid_points.gpkg` or `.geojson` — GIS-ready layer of all interpreted chip centers.
6. `figures/candidate_pond_centers_map.png` — simple map figure for the Results section.
7. `figures/run_summary_counts.png` and `figures/ai_interpretation_counts.png` — simple count figures.
8. `qa/FAILURES_NEED_REVIEW.csv` — any chips that still failed or were incomplete after retries.
9. `review_pack/candidate_reports/` — report images for candidate detections.

## Suggested Results framing

The primary result is a GIS-ready set of candidate pond-based drafting locations, not a confirmed operational drafting inventory. The AI screened fixed-area NAIP image chips for visible open water and nearby vehicle-access context. Candidate points represent approximate pond centers derived from the AI-reported normalized image coordinates. Locations should be described as candidate water-source locations requiring firefighter review, landowner coordination, and field verification.

## Placeholder language

Use `XXXX` for any values that require manual expert review, field confirmation, landowner status, seasonal water persistence, measured depth, or operational safety assessment.
"""
    write_text(results_root / "RESULTS_CONTEXT_FOR_AI.md", context_md)

    readme = f"""# RESULTS folder

This folder was automatically generated by the final NAIP AI pond-screening workflow.

## Folder contents

- `tables/`: CSV tables for writing the Results section.
- `gis/`: GIS-ready outputs for candidate pond centers and all interpreted grid points.
- `figures/`: simple figures for a manuscript/report Results section.
- `review_pack/`: copied report images for manual review.
- `qa/`: retry logs, failed cases, and validation outputs.
- `documents/`: prompt, system preamble, run configuration, and metadata.

## Most important outputs

- Candidate pond centers: `gis/candidate_pond_centers.gpkg`
- All interpreted chip centers: `gis/all_interpreted_grid_points.gpkg`
- Final chip-level table: `tables/final_index_one_row_per_input_point.csv`
- Candidate point table: `tables/candidate_pond_centers.csv`
- Summary table: `tables/run_summary.csv`
- AI writing context: `RESULTS_CONTEXT_FOR_AI.md`

## Run completion

- Input image chips / grid points: {summary['total_input_points']}
- Successful complete interpretations: {summary['successful_complete_interpretations']}
- Failed or incomplete after retries: {summary['failed_or_incomplete_rows']}
- Candidate pond center points: {summary['candidate_pond_center_points']}

If `qa/FAILURES_NEED_REVIEW.csv` has rows, those rows did not fully complete after retries. They were not silently skipped.
"""
    write_text(results_root / "README_RESULTS.md", readme)

    validation_report = {
        "summary": summary,
        "failures_csv": str(qa_dir / "FAILURES_NEED_REVIEW.csv"),
        "candidate_reports_copied": len(copied_reports),
        "results_root": str(results_root),
    }
    write_json(qa_dir / "validation_report.json", validation_report)

    print("Wrote RESULTS folder:", results_root)
    print(json.dumps(summary, indent=2))
    return str(results_root)


# =============================================================================
# BATCH RUNNER
# =============================================================================

def run_one_case_with_retries(row, out_root, question, system_preamble, temperature, export_tif, export_case_shapefile, attempts_path, log_path):
    point_id = row["point_id"]
    lon = float(row["lon"])
    lat = float(row["lat"])
    zoom_m = float(row["zoom_m"])
    start = row["start"]
    end = row["end"]
    case_id = row["case_id"]

    if not FORCE_RERUN_ALL:
        complete, issues = case_is_complete(out_root, case_id)
        if complete:
            print(f"Skipping complete case: {case_id}")
            return reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0)
        elif not RERUN_FAILED_OR_INVALID:
            print(f"Existing incomplete case will not be rerun because RERUN_FAILED_OR_INVALID=False: {case_id}")
            return reconstruct_result_from_completed_case(out_root, point_id, lon, lat, zoom_m, case_id, attempt_num=0)
        else:
            print(f"Rerunning incomplete/failed case: {case_id} | issues={';'.join(issues)}")

    last_res = None
    for attempt_num in range(1, MAX_ATTEMPTS_PER_CASE + 1):
        print(f"Running case {case_id} | attempt {attempt_num}/{MAX_ATTEMPTS_PER_CASE}")
        try:
            res = naip_qa_case(
                point_id=point_id,
                lon=lon,
                lat=lat,
                out_root=out_root,
                question=question,
                zoom_m=zoom_m,
                system_preamble=system_preamble,
                temperature=temperature,
                export_tif=export_tif,
                export_case_shapefile=export_case_shapefile,
                start=start,
                end=end,
                attempt_num=attempt_num,
            )
        except Exception as e:
            tb = traceback.format_exc()
            err = f"{type(e).__name__}: {e}"
            with LOG_FILE_LOCK:
                with Path(log_path).open("a", encoding="utf-8") as f_log:
                    f_log.write(f"\n[{datetime.now().isoformat(timespec='seconds')}] case_id={case_id} attempt={attempt_num}\n{err}\n{tb}\n")
            res = make_error_result(
                point_id=point_id,
                lon=lon,
                lat=lat,
                zoom_m=zoom_m,
                case_id=case_id,
                case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
                attempt_num=attempt_num,
                error=err,
            )

        append_attempt_csv(attempts_path, res)
        last_res = res

        ok, issues = validate_case_result(res)
        res["validation_ok"] = bool(ok)
        res["validation_issues"] = ";".join(issues)
        res["final_complete"] = bool(ok)
        if ok:
            res["ok"] = True
            res["error"] = ""
            return coerce_result_fields(res)

        print(f"Case did not validate: {case_id} | attempt {attempt_num} | issues={';'.join(issues)}")
        if attempt_num < MAX_ATTEMPTS_PER_CASE:
            time.sleep(SLEEP_BETWEEN_ATTEMPTS_SEC)

    if last_res is None:
        last_res = make_error_result(
            point_id=point_id,
            lon=lon,
            lat=lat,
            zoom_m=zoom_m,
            case_id=case_id,
            case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
            attempt_num=MAX_ATTEMPTS_PER_CASE,
            error="No attempts were completed.",
        )

    final_ok, final_issues = validate_case_result(last_res)
    last_res["validation_ok"] = bool(final_ok)
    last_res["validation_issues"] = ";".join(final_issues)
    last_res["final_complete"] = bool(final_ok)
    last_res["ok"] = bool(final_ok)
    if not final_ok and not last_res.get("error"):
        last_res["error"] = "Case failed final validation: " + ";".join(final_issues)
    return coerce_result_fields(last_res)


def run_batch_points_csv(
    points_csv,
    out_root,
    question,
    zoom_m_default=DEFAULT_ZOOM_M,
    system_preamble=None,
    temperature=0,
    export_tif=True,
    export_case_shapefile=True,
    export_batch_shapefile=True,
    start=DEFAULT_START,
    end=DEFAULT_END,
    max_parallel_workers=MAX_PARALLEL_WORKERS,
):
    out_root = ensure_dir(out_root)
    ensure_dir(out_root / CASE_DIR_NAME)

    index_path = out_root / "index.csv"
    attempts_path = out_root / "index_attempts.csv"
    log_path = out_root / "batch_log.txt"

    points_df = read_points_csv(points_csv, zoom_m_default=zoom_m_default, start=start, end=end)
    points_df.to_csv(out_root / "input_points_parsed.csv", index=False)

    # Store final rows by input_order so index.csv remains in original input order,
    # even when cases finish out of order in parallel.
    final_rows_by_order = {}

    invalid_inputs = points_df[~points_df["input_valid"]]
    for _, bad in invalid_inputs.iterrows():
        input_order = int(bad.get("input_order", len(final_rows_by_order) + 1))
        res = make_error_result(
            point_id=bad.get("point_id", ""),
            lon="",
            lat="",
            zoom_m=bad.get("zoom_m", ""),
            case_id="",
            case_dir="",
            attempt_num=0,
            error=bad.get("input_error", "Invalid input row."),
        )
        append_attempt_csv(attempts_path, res)
        final_rows_by_order[input_order] = res

    valid_points = points_df[points_df["input_valid"]].copy()
    n = len(valid_points)

    workers = int(max_parallel_workers or 1)
    workers = max(1, workers)

    def ordered_final_rows():
        return [final_rows_by_order[k] for k in sorted(final_rows_by_order.keys())]

    print(f"\nStarting batch run with MAX_PARALLEL_WORKERS={workers}")
    print(f"Valid input points: {n} | Invalid input rows: {len(invalid_inputs)}")

    if workers == 1:
        # Sequential mode. This is useful for debugging.
        for i, (_, row) in enumerate(valid_points.iterrows(), start=1):
            print(f"\n========== {i}/{n} | point_id={row['point_id']} | case_id={row['case_id']} ==========")
            res = run_one_case_with_retries(
                row=row,
                out_root=out_root,
                question=question,
                system_preamble=system_preamble,
                temperature=temperature,
                export_tif=export_tif,
                export_case_shapefile=export_case_shapefile,
                attempts_path=attempts_path,
                log_path=log_path,
            )
            final_rows_by_order[int(row["input_order"])] = res
            write_index_csv(index_path, ordered_final_rows())
            print_cost_summary(prefix="RUNNING TOTAL")
    else:
        # Parallel mode. Each worker processes one full case, including its retries.
        futures = {}
        with ThreadPoolExecutor(max_workers=workers) as executor:
            for i, (_, row) in enumerate(valid_points.iterrows(), start=1):
                print(f"Submitting {i}/{n} | point_id={row['point_id']} | case_id={row['case_id']}")
                fut = executor.submit(
                    run_one_case_with_retries,
                    row=row,
                    out_root=out_root,
                    question=question,
                    system_preamble=system_preamble,
                    temperature=temperature,
                    export_tif=export_tif,
                    export_case_shapefile=export_case_shapefile,
                    attempts_path=attempts_path,
                    log_path=log_path,
                )
                futures[fut] = (i, int(row["input_order"]), row["point_id"], row["case_id"])

            completed = 0
            for fut in as_completed(futures):
                i, input_order, point_id, case_id = futures[fut]
                completed += 1
                try:
                    res = fut.result()
                except Exception as e:
                    tb = traceback.format_exc()
                    err = f"{type(e).__name__}: {e}"
                    with LOG_FILE_LOCK:
                        with Path(log_path).open("a", encoding="utf-8") as f_log:
                            f_log.write(f"\n[{datetime.now().isoformat(timespec='seconds')}] case_id={case_id} future_failed\n{err}\n{tb}\n")
                    # Pull lon/lat/zoom from the input row for a useful final failure row.
                    row_match = valid_points[valid_points["input_order"].astype(int) == int(input_order)].iloc[0]
                    res = make_error_result(
                        point_id=point_id,
                        lon=row_match.get("lon", ""),
                        lat=row_match.get("lat", ""),
                        zoom_m=row_match.get("zoom_m", ""),
                        case_id=case_id,
                        case_dir=str(Path(out_root) / CASE_DIR_NAME / case_id),
                        attempt_num=MAX_ATTEMPTS_PER_CASE,
                        error=err,
                    )
                    append_attempt_csv(attempts_path, res)

                final_rows_by_order[input_order] = res
                write_index_csv(index_path, ordered_final_rows())

                ok_txt = "OK" if parse_boolish(res.get("final_complete")) else "FAILED"
                print(f"\nCompleted {completed}/{n} | original_submit={i}/{n} | {ok_txt} | point_id={point_id} | case_id={case_id}")
                print_cost_summary(prefix="RUNNING TOTAL")

    write_index_csv(index_path, ordered_final_rows())

    if export_batch_shapefile:
        try:
            batch_shp = build_batch_pond_points_layers(index_path, out_root)
            print("Wrote batch pond shapefile:", batch_shp)
        except Exception as e:
            print(f"Warning: failed to build batch shapefile: {e}")

    write_text(
        out_root / "README_STUDENTS.txt",
        "Student workflow:\n"
        "1) Open RESULTS/README_RESULTS.md first.\n"
        "2) Open RESULTS/tables/final_index_one_row_per_input_point.csv for chip-level results.\n"
        "3) Open RESULTS/gis/candidate_pond_centers.gpkg for mapped candidate pond centers.\n"
        "4) Open report.png files in CASES folders or RESULTS/review_pack/candidate_reports for visual review.\n"
        "5) Treat all mapped points as candidate drafting locations only. Field/local verification is required.\n",
    )

    build_results_folder(
        out_root=out_root,
        points_csv=points_csv,
        prompt_text=question,
        system_preamble=system_preamble or "",
    )

    return str(index_path)


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    MODE = "batch"

    out_root = DEFAULT_OUT_ROOT
    points_csv = DEFAULT_POINTS_CSV
    temperature = 0

    scene_width_m = int(DEFAULT_ZOOM_M * 2)
    question = build_pond_prompt(scene_width_m=scene_width_m)
    system_preamble = build_system_preamble()

    if MODE == "single":
        point_id = "pt001test23"
        lon, lat = -105.3227622, 40.55704649

        res = naip_qa_case(
            point_id=point_id,
            lon=lon,
            lat=lat,
            out_root=out_root,
            question=question,
            zoom_m=DEFAULT_ZOOM_M,
            system_preamble=system_preamble,
            temperature=temperature,
            export_tif=True,
            export_case_shapefile=True,
            start=DEFAULT_START,
            end=DEFAULT_END,
            attempt_num=1,
        )
        print(json.dumps(res, indent=2))
        build_results_folder(
            out_root=out_root,
            points_csv=points_csv,
            prompt_text=question,
            system_preamble=system_preamble,
        )
        print_cost_summary(prefix="SESSION TOTAL")

    else:
        idx = run_batch_points_csv(
            points_csv=points_csv,
            out_root=out_root,
            question=question,
            zoom_m_default=DEFAULT_ZOOM_M,
            system_preamble=system_preamble,
            temperature=temperature,
            export_tif=True,
            export_case_shapefile=True,
            export_batch_shapefile=True,
            start=DEFAULT_START,
            end=DEFAULT_END,
            max_parallel_workers=MAX_PARALLEL_WORKERS,
        )
        print("Wrote index:", idx)
        print_cost_summary(prefix="SESSION TOTAL")


In [ ]:
# THIS script converts the main index.csv rows into map point - CLAUDE
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# =============================================================================
# INPUTS
# =============================================================================

index_csv = r"C:\Users\index.csv"
out_dir = r"C:\Users\"
os.makedirs(out_dir, exist_ok=True)

out_shp_all = os.path.join(out_dir, "ai_pond_points_all_500m_CLAUDE.shp")
out_shp_yes = os.path.join(out_dir, "ai_pond_points_yes_500m_CLAUDE.shp")
out_shp_no = os.path.join(out_dir, "ai_pond_points_no_500m_CLAUDE.shp")
out_plot = os.path.join(out_dir, "ai_is_there_a_pond_map_500m_CLAUDE.png")
out_csv_clean = os.path.join(out_dir, "ai_pond_points_clean_500m_CLAUDE.csv")

# =============================================================================
# READ CSV
# =============================================================================

df = pd.read_csv(index_csv)

# Keep only rows with valid lat/lon
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
df = df.dropna(subset=["lat", "lon"]).copy()

# Normalize the pond field
df["ai_is_there_a_pond"] = (
    df["ai_is_there_a_pond"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Optional: keep only yes/no rows for plotting/classification
valid_labels = ["yes", "no"]
df["pond_class"] = df["ai_is_there_a_pond"].where(df["ai_is_there_a_pond"].isin(valid_labels), "other")

# Save cleaned CSV
df.to_csv(out_csv_clean, index=False)

# =============================================================================
# CREATE GEODATAFRAME
# =============================================================================

geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Shapefile field names should be short to avoid truncation issues
gdf["pond_ai"] = gdf["pond_class"]

# Save all points
gdf.to_file(out_shp_all)

# Save yes/no subsets
gdf_yes = gdf[gdf["pond_ai"] == "yes"].copy()
gdf_no = gdf[gdf["pond_ai"] == "no"].copy()

if len(gdf_yes) > 0:
    gdf_yes.to_file(out_shp_yes)

if len(gdf_no) > 0:
    gdf_no.to_file(out_shp_no)

# =============================================================================
# PLOT
# =============================================================================

fig, ax = plt.subplots(figsize=(10, 10))

# Plot "no" first so "yes" draws on top
if len(gdf_no) > 0:
    gdf_no.plot(ax=ax, markersize=12, label="No pond", alpha=0.7)

if len(gdf_yes) > 0:
    gdf_yes.plot(ax=ax, markersize=18, label="Yes pond", alpha=0.9)

gdf_other = gdf[gdf["pond_ai"] == "other"].copy()
if len(gdf_other) > 0:
    gdf_other.plot(ax=ax, markersize=10, label="Other/blank", alpha=0.5)

ax.set_title("AI Pond Classification from index.csv")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(out_plot, dpi=300)
plt.show()

# =============================================================================
# SUMMARY
# =============================================================================

print("\nDone.")
print(f"Input CSV: {index_csv}")
print(f"Clean CSV: {out_csv_clean}")
print(f"All-points shapefile: {out_shp_all}")
print(f"Yes-points shapefile: {out_shp_yes if len(gdf_yes) > 0 else 'No YES records to save'}")
print(f"No-points shapefile: {out_shp_no if len(gdf_no) > 0 else 'No NO records to save'}")
print(f"Plot: {out_plot}")
print("\nCounts:")
print(gdf["pond_ai"].value_counts(dropna=False))

In [ ]:
# THIS script exports 2 - all pond centers and only likely draftable pond centers - CLAUDE
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path

# =============================================================================
# INPUTS
# =============================================================================

INDEX_CSV = r"C:\Users\index.csv"

OUT_DIR = r"C:\Users\"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_DRAFTABLE_SHP = os.path.join(OUT_DIR, "ai_draftable_pond_centers_500m_CLAUDE.shp")
OUT_DRAFTABLE_GEOJSON = os.path.join(OUT_DIR, "ai_draftable_pond_centers_500m_CLAUDE.geojson")
OUT_DRAFTABLE_CSV = os.path.join(OUT_DIR, "ai_draftable_pond_centers_500m_CLAUDE.csv")

OUT_ALL_PONDS_WITH_STATUS_SHP = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_500m_CLAUDE.shp")
OUT_ALL_PONDS_WITH_STATUS_GEOJSON = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_500m_CLAUDE.geojson")
OUT_ALL_PONDS_WITH_STATUS_CSV = os.path.join(OUT_DIR, "ai_all_pond_centers_with_draftable_status_500m_CLAUDE.csv")

# =============================================================================
# HELPERS
# =============================================================================

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def short_shp_columns(gdf):
    """
    Rename columns to shapefile-safe names <= 10 characters.
    """
    rename = {
        "case_id": "case_id",
        "point_id": "point_id",
        "src_lon": "src_lon",
        "src_lat": "src_lat",
        "naip_id": "naip_id",
        "pond_lbl": "pond_lbl",
        "pond_label": "pond_lbl",
        "conf": "conf",
        "confidence": "conf",
        "norm_x": "norm_x",
        "norm_y": "norm_y",
        "pix_x": "pix_x",
        "pixel_x": "pix_x",
        "pix_y": "pix_y",
        "pixel_y": "pix_y",
        "ai_is_there_surface_water": "ai_sw",
        "ai_is_there_a_pond": "ai_pond",
        "ai_road_or_access_nearby": "ai_access",
        "ai_likely_draftable": "ai_draft",
        "ai_pond_count": "ai_pcnt",
        "report_png": "report_png",
        "image_png": "image_png",
        "ai_json": "ai_json",
        "pond_points_shp": "src_shp",
        "pond_points_geojson": "src_geojs",
        "pond_points_csv": "src_csv",
    }

    out = gdf.copy()
    out = out.rename(columns={c: rename[c] for c in out.columns if c in rename})

    # Shapefile is picky about object fields and long strings
    for c in out.columns:
        if c == "geometry":
            continue
        if out[c].dtype == object:
            out[c] = out[c].fillna("").astype(str)

    return out

# =============================================================================
# READ INDEX
# =============================================================================

if not os.path.exists(INDEX_CSV):
    raise FileNotFoundError(f"Index CSV not found:\n{INDEX_CSV}")

idx = pd.read_csv(INDEX_CSV)

required_cols = [
    "pond_points_shp",
    "ai_likely_draftable",
    "ai_is_there_a_pond",
]

missing = [c for c in required_cols if c not in idx.columns]
if missing:
    raise ValueError(
        "The index.csv is missing required columns:\n"
        + "\n".join(missing)
        + "\n\nAvailable columns are:\n"
        + "\n".join(idx.columns)
    )

idx["ai_likely_draftable_norm"] = idx["ai_likely_draftable"].apply(norm_text)
idx["ai_is_there_a_pond_norm"] = idx["ai_is_there_a_pond"].apply(norm_text)

# Keep only rows where AI said likely_draftable == yes
draftable_idx = idx[idx["ai_likely_draftable_norm"] == "yes"].copy()

print("Index rows:", len(idx))
print("Rows where ai_likely_draftable == yes:", len(draftable_idx))

# =============================================================================
# MERGE ALL POND POINTS WITH STATUS
# =============================================================================

all_pond_gdfs = []
draftable_gdfs = []

for _, row in idx.iterrows():
    shp_path = str(row.get("pond_points_shp", "")).strip()

    if not shp_path or not os.path.exists(shp_path):
        continue

    try:
        ponds = gpd.read_file(shp_path)
    except Exception as e:
        print(f"Could not read: {shp_path}")
        print(f"  {type(e).__name__}: {e}")
        continue

    if ponds.empty:
        continue

    # Add classification fields from index.csv to each pond center point
    ponds["ai_is_there_surface_water"] = row.get("ai_is_there_surface_water", "")
    ponds["ai_is_there_a_pond"] = row.get("ai_is_there_a_pond", "")
    ponds["ai_road_or_access_nearby"] = row.get("ai_road_or_access_nearby", "")
    ponds["ai_likely_draftable"] = row.get("ai_likely_draftable", "")
    ponds["ai_pond_count"] = row.get("ai_pond_count", "")
    ponds["report_png"] = row.get("report_png", "")
    ponds["image_png"] = row.get("image_png", "")
    ponds["ai_json"] = row.get("ai_json", "")
    ponds["pond_points_shp"] = shp_path
    ponds["pond_points_geojson"] = row.get("pond_points_geojson", "")
    ponds["pond_points_csv"] = row.get("pond_points_csv", "")

    all_pond_gdfs.append(ponds)

    if norm_text(row.get("ai_likely_draftable", "")) == "yes":
        draftable_gdfs.append(ponds.copy())

# =============================================================================
# EXPORT ALL POND CENTERS WITH DRAFTABLE STATUS
# =============================================================================

if len(all_pond_gdfs) > 0:
    all_ponds = gpd.GeoDataFrame(
        pd.concat(all_pond_gdfs, ignore_index=True),
        geometry="geometry",
        crs=all_pond_gdfs[0].crs
    )

    if all_ponds.crs is None:
        all_ponds = all_ponds.set_crs("EPSG:4326")

    all_ponds_wgs84 = all_ponds.to_crs("EPSG:4326")

    all_ponds_wgs84["lon"] = all_ponds_wgs84.geometry.x
    all_ponds_wgs84["lat"] = all_ponds_wgs84.geometry.y

    all_ponds_wgs84.drop(columns="geometry").to_csv(OUT_ALL_PONDS_WITH_STATUS_CSV, index=False)
    all_ponds_wgs84.to_file(OUT_ALL_PONDS_WITH_STATUS_GEOJSON, driver="GeoJSON")

    all_ponds_shp = short_shp_columns(all_ponds_wgs84)
    all_ponds_shp.to_file(OUT_ALL_PONDS_WITH_STATUS_SHP)

    print("\nWrote all pond centers with draftable status:")
    print(OUT_ALL_PONDS_WITH_STATUS_SHP)
    print(OUT_ALL_PONDS_WITH_STATUS_GEOJSON)
    print(OUT_ALL_PONDS_WITH_STATUS_CSV)

else:
    print("\nNo pond center shapefiles found to merge.")

# =============================================================================
# EXPORT ONLY DRAFTABLE POND CENTERS
# =============================================================================

if len(draftable_gdfs) > 0:
    draftable = gpd.GeoDataFrame(
        pd.concat(draftable_gdfs, ignore_index=True),
        geometry="geometry",
        crs=draftable_gdfs[0].crs
    )

    if draftable.crs is None:
        draftable = draftable.set_crs("EPSG:4326")

    draftable_wgs84 = draftable.to_crs("EPSG:4326")

    draftable_wgs84["lon"] = draftable_wgs84.geometry.x
    draftable_wgs84["lat"] = draftable_wgs84.geometry.y

    draftable_wgs84.drop(columns="geometry").to_csv(OUT_DRAFTABLE_CSV, index=False)
    draftable_wgs84.to_file(OUT_DRAFTABLE_GEOJSON, driver="GeoJSON")

    draftable_shp = short_shp_columns(draftable_wgs84)
    draftable_shp.to_file(OUT_DRAFTABLE_SHP)

    print("\nWrote draftable pond centers only:")
    print(OUT_DRAFTABLE_SHP)
    print(OUT_DRAFTABLE_GEOJSON)
    print(OUT_DRAFTABLE_CSV)

    print("\nDraftable pond count:")
    print(len(draftable_wgs84))

else:
    print("\nNo records where ai_likely_draftable == yes were found.")
    print("No draftable pond shapefile was created.")

# =============================================================================
# SUMMARY
# =============================================================================

print("\nSummary from index.csv:")
print(idx["ai_likely_draftable_norm"].value_counts(dropna=False))

print("\nDone.")